<a href="https://colab.research.google.com/github/DmitriyKolesnikM8O/MOEX-Scripts/blob/main/Adviser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
MOEX Screener v2 — технический скринер акций Московской биржи для часовой торговли
====================================================================================
Что изменилось по сравнению с v1 (правки по реальным багам и фидбеку):
  - Убран расчёт размера позиции в рублях/штуках — он был не нужен и вводил в
    заблуждение. Оставлены только уровни: вход, стоп-лосс, тейк-профит, R:R.
  - ИСПРАВЛЕНО округление цен — раньше дешёвые тикеры (доли рубля, напр. TGKA)
    схлопывались в 0.01/0.00/0.01 из-за фиксированных 2 знаков после запятой.
    Теперь округление адаптивное под порядок цены.
  - ИСПРАВЛЕН источник ложных свечных паттернов — раньше использовалась
    самодельная наивная логика без порога на размер тела свечи и без проверки,
    что последняя свеча вообще ЗАКРЫЛАСЬ (последний бар в свежих данных может
    быть ещё в процессе формирования — это давало мусорную геометрию и ложные
    сигналы). Обе проблемы исправлены; свечные паттерны теперь считает
    библиотека pandas-ta-classic (62 паттерна, протестированы на соответствие
    TA-Lib) вместо ручного кода.
  - ДОБАВЛЕНО много новых индикаторов через pandas-ta-classic: Stochastic,
    CCI, Williams %R, OBV, SuperTrend, Aroon — в дополнение к своим ADX/VWAP.
  - ДОБАВЛЕНО подтверждение по старшему таймфрейму (дневной тренд): часовой
    сигнал против дневного тренда получает штраф к score, по тренду — бонус.
    Это отдельно посчитано из уже скачанных часовых данных (ресемпл в дневки),
    без дополнительных запросов к API.
  - ДОБАВЛЕНА относительная сила к индексу IMOEX — тикер, который растёт
    быстрее рынка, интереснее для лонга, чем тот, что просто ползёт вместе
    с общим ростом рынка (и наоборот для шорта).
  - ML-компонент остался, но его вес в score по-прежнему автоматически
    приглушается, если AUC модели близко к случайности (см. предупреждение).

ВАЖНО, ПРОЧТИТЕ ПЕРЕД ИСПОЛЬЗОВАНИЕМ:
  - Это НЕ финансовая рекомендация и не Грааль. Инструмент сужает список
    кандидатов и структурирует анализ. Решение и его последствия — на вас.
  - ROC-AUC модели в районе 0.5-0.58 — это ожидаемая реальность рынка, а не
    брак: устойчивый легко находимый паттерн по цене/объёму давно был бы
    вычищен крупными фондами. Не судите весь инструмент по одному числу ML —
    техническая часть (тренд, паттерны, дивергенции, мультитаймфрейм,
    относительная сила) работает независимо от ML и весит в score больше.
  - Автоматические паттерны (голова-плечи, треугольники и т.д.) — это
    эвристика по локальным экстремумам, а не ручная разметка трейдера.
    На часовиках такие паттерны шумнее, чем на дневках/неделях.
  - Фундаментал сознательно не включён — инструмент под внутридневную/
    часовую спекуляцию, а не под долгосрочные инвестиции.

Как запустить в Google Colab:
  1. Создайте новый notebook на colab.research.google.com
  2. Runtime → Change runtime type → Hardware accelerator → None (GPU тут
     не используется ничем в этом скрипте и не ускоряет ничего)
  3. Вставьте содержимое этого файла в одну ячейку и запустите
  4. Настройте параметры в блоке CONFIG ниже под себя
"""

import os
import time
import pickle
import warnings
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.signal import argrelextrema

warnings.filterwarnings("ignore")


def _ensure(pkg_import_name, pip_name=None):
    try:
        return __import__(pkg_import_name)
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", pip_name or pkg_import_name])
        return __import__(pkg_import_name)


_ensure("tqdm")
_ensure("sklearn", "scikit-learn")
try:
    import pandas_ta_classic as pta
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "pandas-ta-classic"])
    import pandas_ta_classic as pta

from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# =============================================================================
# CONFIG — настройте под себя
# =============================================================================
CONFIG = {
    "history_days": 240,          # сколько дней истории тянуть для часовых свечей
    "candle_interval": 60,        # 60 = часовой таймфрейм (MOEX ISS interval)
    "max_tickers": None,          # None = использовать liquidity_top_n, либо число для теста
    "liquidity_top_n": 120,       # сколько самых ликвидных тикеров брать в работу
    "min_atr_pct": 0.15,          # мин. волатильность (ATR% от цены) — отсекает фонды
                                   # ликвидности (LQDT, AKMM, SBMM и т.п.)
    "atr_stop_mult": 1.5,         # множитель ATR для стоп-лосса
    "risk_reward_target": 2.0,    # целевое соотношение прибыль/риск для тейк-профита
    "ml_horizon_bars": 6,         # на сколько часовых баров вперёд прогнозируем движение
    "ml_up_threshold_atr": 0.5,   # рост считается "успешным", если цена ушла > 0.5*ATR
    "rel_strength_window": 20,    # окно (в часовых барах) для относительной силы к IMOEX
    "top_n_report": 15,           # сколько лучших идей показать в итоге
    "max_workers": 10,            # сколько тикеров качать параллельно
    "cache_dir": "/content/moex_cache",  # кэш на время сессии
    "cache_ttl_hours": 6,         # сколько часов кэш считается свежим
    "swing_order": 3,             # чувствительность поиска локальных экстремумов
    "pattern_lookback_bars": 90,  # окно поиска графических паттернов
    "backtest_enabled": True,     # прогонять ли бэктест технического score
    "backtest_max_tickers": 20,   # на скольких тикерах бэктестить (~5-7 минут при дефолтах;
                                   # больше тикеров = точнее оценка, но дольше)
    "backtest_stride": 6,         # шаг по барам (6 = проверять сигнал раз в 6 часов, не на каждом)
    "backtest_max_holding_bars": 40,  # макс. срок удержания позиции в барах, если ни стоп ни тейк не пробиты
    "backtest_min_samples_per_component": 40,  # мин. число случаев компонента, чтобы доверять его лифту
    "persist_calibration_across_runs": True,  # копить статистику калибровки между повторными запусками
                                               # в этой сессии (сбрасывается при полном перезапуске Colab)
    "walkforward_n_dates": 4,          # сколько тестовых дат равномерно раскидать по истории
    "walkforward_top_n": 10,           # сколько топ-рекомендаций оценивать на каждую дату
    "walkforward_eval_max_bars": 60,   # горизонт проверки исхода вперёд, в часовых барах (~1.5-2 недели)
    "walkforward_calibration_tickers": 15,  # тикеров для калибровки весов НА КАЖДУЮ тестовую дату
                                             # (меньше, чем в обычном прогоне — дат несколько, считать дольше)
    "stop_sensitivity_enabled": True,
    "stop_sensitivity_multipliers": [1.0, 1.5, 2.0, 2.5, 3.0],  # какие множители ATR для стопа сравнить
    "stop_sensitivity_tickers": 10,     # на скольких тикерах прогонять сравнение (доп. проход, держим малым)
    "stop_sensitivity_stride": 10,      # шаг по барам для этого анализа (можно реже, чем основной бэктест)
}

ISS_BASE = "https://iss.moex.com/iss"

import re

# Русские названия для самых значимых свечных паттернов из pandas-ta-classic.
# Если паттерна нет в словаре — показываем его техническое имя как есть.
import re

# ВАЖНО: это НЕ полный список из 62 паттернов pandas-ta-classic, а осознанно
# урезанный до тех, что реально различимы и что-то значат. Причина: добрая
# половина из 62 — это по сути одно и то же явление (маленькое тело свечи)
# под разными именами (Доджи всех видов, Волчок, Высокая волна, Long/Short
# Line, Rickshaw Man) — на одной и той же свече срабатывают сразу 5-6 из
# них одновременно, добавляя не новую информацию, а шум и раздутый список.
# Паттерн, которого нет в этом словаре, НЕ показывается вообще (см.
# detect_candle_patterns) — это фильтр, а не забытый перевод.
CDL_NAMES_RU = {
    "CDL_ENGULFING": "Поглощение",
    "CDL_HAMMER": "Молот",
    "CDL_HANGINGMAN": "Повешенный",
    "CDL_SHOOTINGSTAR": "Падающая звезда",
    "CDL_INVERTEDHAMMER": "Перевёрнутый молот",
    "CDL_MORNINGSTAR": "Утренняя звезда",
    "CDL_MORNINGDOJISTAR": "Утренняя звезда-доджи",
    "CDL_EVENINGSTAR": "Вечерняя звезда",
    "CDL_EVENINGDOJISTAR": "Вечерняя звезда-доджи",
    "CDL_3WHITESOLDIERS": "Три белых солдата",
    "CDL_3BLACKCROWS": "Три чёрные вороны",
    "CDL_HARAMI": "Харами",
    "CDL_HARAMICROSS": "Харами-крест",
    "CDL_DARKCLOUDCOVER": "Завеса из тёмных облаков",
    "CDL_PIERCING": "Просвет в облаках",
    "CDL_MARUBOZU": "Марубозу",
    "CDL_CLOSINGMARUBOZU": "Марубозу закрытия",
    "CDL_ABANDONEDBABY": "Брошенный младенец",
    "CDL_TRISTAR": "Три звезды",
    "CDL_KICKING": "Пинок",
}


# =============================================================================
# ФОРМАТИРОВАНИЕ ЦЕН
# =============================================================================
def smart_round(x, sig_figs=4):
    """
    Адаптивное округление: у дорогих бумаг (SBER ~280) — 2 знака,
    у дешёвых (TGKA ~0.0075) — достаточно значащих цифр, чтобы стоп и
    тейк не схлопывались в одно и то же число.
    """
    if x is None or not np.isfinite(x):
        return x
    if x == 0:
        return 0.0
    if abs(x) >= 1:
        return round(x, 2)
    magnitude = int(np.floor(np.log10(abs(x))))
    decimals = min(max(-magnitude + (sig_figs - 1), 2), 8)
    return round(x, decimals)


# =============================================================================
# 1. ЗАГРУЗКА ДАННЫХ С MOEX ISS API
# =============================================================================
def get_market_snapshot():
    """
    Один запрос отдаёт live-данные сразу по всем акциям TQBR: сегодняшний
    оборот (для префильтра по ликвидности) и текущие BID/OFFER (для оценки
    реалистичной цены входа и спреда). Переиспользуем один и тот же запрос
    для обеих задач, чтобы не дёргать API дважды.
    """
    url = f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {
        "iss.meta": "off",
        "iss.only": "securities,marketdata",
        "securities.columns": "SECID,SHORTNAME",
        "marketdata.columns": "SECID,VALTODAY,BID,OFFER,LAST",
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    js = r.json()

    sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
    md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])
    for col in ["VALTODAY", "BID", "OFFER", "LAST"]:
        md[col] = pd.to_numeric(md[col], errors="coerce")
    md["VALTODAY"] = md["VALTODAY"].fillna(0)

    return sec.merge(md, on="SECID", how="left")


def get_liquid_tickers(snapshot, top_n):
    merged = snapshot.copy()
    if merged["VALTODAY"].sum() > 0:
        merged = merged.sort_values("VALTODAY", ascending=False)
    return merged["SECID"].head(top_n).tolist()



def _cache_path(cache_dir, key):
    return os.path.join(cache_dir, f"{key}.pkl")


def _load_from_cache(cache_dir, key, ttl_hours):
    path = _cache_path(cache_dir, key)
    if not os.path.exists(path):
        return None
    age_hours = (time.time() - os.path.getmtime(path)) / 3600
    if age_hours > ttl_hours:
        return None
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception:
        return None


def _save_to_cache(cache_dir, key, df):
    os.makedirs(cache_dir, exist_ok=True)
    with open(_cache_path(cache_dir, key), "wb") as f:
        pickle.dump(df, f)


def _fetch_candles_raw(url, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    all_rows, start, columns = [], 0, None
    while True:
        params = {
            "from": since.strftime("%Y-%m-%d"),
            "till": till.strftime("%Y-%m-%d"),
            "interval": interval,
            "start": start,
        }
        r = requests.get(url, params=params, timeout=20)
        if r.status_code != 200:
            break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None:
            columns = js.get("columns", [])
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 500:
            break
        start += len(rows)

    if not all_rows or columns is None:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.rename(columns={
        "open": "Open", "close": "Close", "high": "High",
        "low": "Low", "volume": "Volume", "begin": "Date", "end": "DateEnd"
    })
    df = df[["Date", "DateEnd", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date")
    # MOEX ISS иногда отдаёт повторяющийся бар на стыке страниц пагинации —
    # без дедупликации это ломает любое дальнейшее выравнивание по датам
    df = df.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)

    # ВАЖНО: если последняя свеча ещё не закрылась (её конец в будущем
    # относительно момента запроса), выкидываем её — иначе индикаторы и
    # свечные паттерны считаются по недостроенному бару и дают мусор.
    if len(df) > 0 and df["DateEnd"].iloc[-1] > pd.Timestamp.now():
        df = df.iloc[:-1]

    return df.drop(columns=["DateEnd"]).reset_index(drop=True)


def get_hourly_candles(ticker, days_back, interval=60):
    url = (f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/"
           f"securities/{ticker}/candles.json")
    return _fetch_candles_raw(url, days_back, interval)


def get_index_candles(secid, days_back, interval=60):
    """Свечи по индексу (по умолчанию IMOEX) — для расчёта относительной силы."""
    url = f"{ISS_BASE}/engines/stock/markets/index/boards/SNDX/securities/{secid}/candles.json"
    return _fetch_candles_raw(url, days_back, interval)


def download_all_candles(tickers, days_back, interval, cache_dir, ttl_hours, max_workers):
    results = {}
    to_download = []
    for t in tickers:
        cached = _load_from_cache(cache_dir, t, ttl_hours)
        if cached is not None and not cached.empty:
            results[t] = cached
        else:
            to_download.append(t)

    if results:
        print(f"   Из кэша сессии загружено сразу: {len(results)} тикеров")

    if to_download:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(get_hourly_candles, t, days_back, interval): t
                       for t in to_download}
            for fut in tqdm(as_completed(futures), total=len(futures),
                             desc="   Качаю свечи параллельно"):
                t = futures[fut]
                try:
                    df = fut.result()
                    if not df.empty:
                        results[t] = df
                        _save_to_cache(cache_dir, t, df)
                except Exception:
                    pass

    return results


# =============================================================================
# 2. ИНДИКАТОРЫ
# =============================================================================
def add_indicators(df):
    df = df.copy()
    df["EMA20"] = df["Close"].ewm(span=20, adjust=False).mean()
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()

    delta = df["Close"].diff()
    gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["RSI14"] = (100 - (100 / (1 + rs))).fillna(50)

    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_hist"] = df["MACD"] - df["MACD_signal"]

    # ВАЖНО: раньше компонент "macd" в score смотрел на статичный знак
    # гистограммы (>0 / <0) — а он почти никогда не равен нулю, то есть
    # компонент "срабатывал" почти на КАЖДОМ баре. Из-за этого бэктест не
    # мог его откалибровать (не с чем сравнивать — группа "не сработал"
    # почти пустая). Кроссовер (смена знака) — гораздо более редкое и
    # содержательное событие: momentum только что развернулся.
    macd_sign = np.sign(df["MACD_hist"])
    prev_sign = macd_sign.shift(1)
    df["MACD_cross_up"] = ((macd_sign > 0) & (prev_sign <= 0)).astype(int)
    df["MACD_cross_down"] = ((macd_sign < 0) & (prev_sign >= 0)).astype(int)

    prev_close = df["Close"].shift(1)
    tr = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs()
    ], axis=1).max(axis=1)
    df["ATR14"] = tr.ewm(alpha=1/14, adjust=False).mean()
    df["ATR_pct"] = df["ATR14"] / df["Close"] * 100

    sma20 = df["Close"].rolling(20).mean()
    std20 = df["Close"].rolling(20).std()
    df["BB_upper"] = sma20 + 2 * std20
    df["BB_lower"] = sma20 - 2 * std20
    df["BB_pctB"] = (df["Close"] - df["BB_lower"]) / (df["BB_upper"] - df["BB_lower"])

    df["Vol_SMA20"] = df["Volume"].rolling(20).mean()
    df["Vol_ratio"] = df["Volume"] / df["Vol_SMA20"].replace(0, np.nan)

    df["trend_up"] = ((df["EMA20"] > df["EMA50"]) & (df["EMA50"] > df["EMA200"])).astype(int)
    df["trend_down"] = ((df["EMA20"] < df["EMA50"]) & (df["EMA50"] < df["EMA200"])).astype(int)

    up_move, down_move = df["High"].diff(), -df["Low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    atr_for_adx = df["ATR14"].replace(0, np.nan)
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/14, adjust=False).mean() / atr_for_adx
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/14, adjust=False).mean() / atr_for_adx
    dx = ((plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)) * 100
    df["ADX14"] = dx.ewm(alpha=1/14, adjust=False).mean().fillna(0)

    session = df["Date"].dt.date
    typical_price = (df["High"] + df["Low"] + df["Close"]) / 3
    pv = typical_price * df["Volume"]
    df["VWAP"] = pv.groupby(session).cumsum() / df["Volume"].groupby(session).cumsum().replace(0, np.nan)
    df["price_vs_vwap_pct"] = (df["Close"] - df["VWAP"]) / df["VWAP"] * 100

    # --- Доп. индикаторы из pandas-ta-classic (Stochastic, CCI, Williams %R,
    # OBV, SuperTrend, Aroon) — обёрнуто в try, чтобы падение одного
    # индикатора не роняло весь пайплайн по всем тикерам.
    ta_input = df.rename(columns={
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Volume": "volume"
    }).set_index("Date")

    def _reindexed(result):
        """pandas-ta иногда сам выкидывает NaN-строки из результата — выравниваем
        обратно по полному индексу дат, чтобы длина всегда совпадала с df."""
        if result is None:
            return None
        return result.reindex(ta_input.index)

    try:
        stoch = _reindexed(ta_input.ta.stoch(k=14, d=3, smooth_k=3))
        if stoch is not None and not stoch.empty:
            df["STOCH_k"] = stoch.iloc[:, 0].values
            df["STOCH_d"] = stoch.iloc[:, 1].values
    except Exception:
        df["STOCH_k"], df["STOCH_d"] = np.nan, np.nan

    try:
        cci = _reindexed(ta_input.ta.cci(length=14))
        df["CCI14"] = cci.values if cci is not None else np.nan
    except Exception:
        df["CCI14"] = np.nan

    try:
        willr = _reindexed(ta_input.ta.willr(length=14))
        df["WILLR14"] = willr.values if willr is not None else np.nan
    except Exception:
        df["WILLR14"] = np.nan

    try:
        obv = _reindexed(ta_input.ta.obv())
        df["OBV"] = obv.values if obv is not None else np.nan
        df["OBV_SMA20"] = pd.Series(df["OBV"]).rolling(20).mean().values
    except Exception:
        df["OBV"], df["OBV_SMA20"] = np.nan, np.nan

    try:
        aroon = _reindexed(ta_input.ta.aroon(length=14))
        if aroon is not None and not aroon.empty:
            df["AROON_up"] = aroon.iloc[:, 0].values
            df["AROON_down"] = aroon.iloc[:, 1].values
    except Exception:
        df["AROON_up"], df["AROON_down"] = np.nan, np.nan

    try:
        st = _reindexed(ta_input.ta.supertrend(length=7, multiplier=3.0))
        if st is not None and not st.empty:
            dir_col = [c for c in st.columns if c.startswith("SUPERTd")]
            df["SuperTrend_dir"] = st[dir_col[0]].values if dir_col else np.nan
    except Exception:
        df["SuperTrend_dir"] = np.nan

    # Защитный барьер: если какой-то из шагов выше случайно продублировал
    # колонку (встречалось на реальных данных с нетипичной формой ответа
    # pandas-ta для отдельных проблемных тикеров), не даём этому дальше
    # ломать сравнения DataFrame при обучении ML.
    df = df.loc[:, ~df.columns.duplicated()]
    # ВАЖНО: НЕ храним ta_input в df.attrs. pandas при pd.concat() пытается
    # сравнить .attrs всех объединяемых таблиц между собой (obj.attrs == attrs),
    # а сравнение словарей, где значение — целый DataFrame, ломается с
    # "ambiguous truth value" / "can only compare identically-labeled
    # DataFrame objects". Это и было причиной падения на этапе обучения ML —
    # нашёл через прямой traceback, не наугад.
    return df


def add_daily_context(df, pattern_lookback_days=60, swing_order=2):
    """
    Ресемплит уже скачанные часовые данные в дневные бары — общая картина
    "с высоты птичьего полёта" без дополнительных запросов к API: дневной
    тренд, дневной RSI/ADX, и паттерны/дивергенции, найденные уже НА САМИХ
    ДНЕВКАХ той же логикой, что и на часовиках (переиспользуем те же функции
    поиска паттернов — они не привязаны к таймфрейму).

    ВАЖНО: компонент "daily_pattern" в score НЕ откалиброван по бэктесту —
    честно говоря, полноценный дневной анализ на каждом историческом шаге
    бэктеста был бы слишком медленным (это отдельный ресемпл + поиск паттернов
    на каждый из тысяч шагов), так что для него в бэктесте всегда будет мало
    данных и он останется на весе по умолчанию. Это осознанный компромисс
    между глубиной анализа и скоростью, а не недосмотр.
    """
    daily = (df.set_index("Date")
               .resample("1D")
               .agg({"Open": "first", "High": "max", "Low": "min",
                     "Close": "last", "Volume": "sum"})
               .dropna())
    if len(daily) < 55:
        return None

    ema20 = daily["Close"].ewm(span=20, adjust=False).mean()
    ema50 = daily["Close"].ewm(span=50, adjust=False).mean()
    trend_up = bool(ema20.iloc[-1] > ema50.iloc[-1]) if ema20.iloc[-1] != ema50.iloc[-1] else None

    daily = daily.reset_index()
    result = {"trend_up": trend_up, "rsi": None, "adx": None, "patterns": []}
    try:
        daily_ind = add_indicators(daily)
        last_daily = daily_ind.iloc[-1]
        if pd.notna(last_daily.get("RSI14")):
            result["rsi"] = round(last_daily["RSI14"], 1)
        if pd.notna(last_daily.get("ADX14")):
            result["adx"] = round(last_daily["ADX14"], 1)

        lb = min(pattern_lookback_days, len(daily_ind) - 1)
        if lb >= 30:
            cp = detect_candle_patterns(daily_ind)
            chp = detect_chart_patterns(daily_ind, lb, swing_order)
            dp = detect_divergence(daily_ind, lb, swing_order)
            result["patterns"] = ([f"Дневной паттерн: {p}" for p in (cp + chp)] +
                                   [f"Дневная дивергенция: {p}" for p in dp])
    except Exception:
        pass  # дневной контекст — бонус, не критичная часть; при сбое просто без него

    return result


def compute_relative_strength(df, index_df, window):
    """
    Относительная сила к IMOEX: разница накопленной доходности тикера и
    индекса за последние `window` часовых баров, в процентных пунктах.
    Положительное значение = тикер обгоняет рынок.

    ВАЖНО: используем reindex/merge_asof по отсортированному индексу вместо
    обычного merge на Date. Обычный merge на дублирующихся датах (а MOEX
    ISS иногда отдаёт повторяющиеся бары на стыке страниц пагинации) даёт
    строк БОЛЬШЕ, чем в исходном df, и тогда результат перестаёт совпадать
    по длине с df — именно это рушило пайплайн на реальных данных.
    """
    if index_df is None or index_df.empty:
        df["rel_strength_pct"] = np.nan
        return df

    idx = index_df[["Date", "Close"]].rename(columns={"Close": "Index_Close"})
    idx = idx.drop_duplicates(subset="Date").sort_values("Date")
    base = df[["Date"]].sort_values("Date")

    aligned = pd.merge_asof(base, idx, on="Date", direction="backward")
    aligned = aligned.set_index(base.index).reindex(df.index)

    ticker_ret = df["Close"].pct_change(window)
    index_ret = aligned["Index_Close"].pct_change(window)
    df["rel_strength_pct"] = ((ticker_ret.values - index_ret.values) * 100)
    return df


def compute_index_trend(df, index_df, fast=20, slow=50):
    """
    Направление тренда самого IMOEX на каждый момент времени (EMA20 vs
    EMA50 по индексу), выровненное по датам тикера. ВАЖНО: это НЕ то же
    самое, что относительная сила — акция может "обгонять" падающий индекс,
    оставаясь при этом падающей, просто чуть меньше остальных рынок. До
    этой версии в системе вообще не было явного учёта направления самого
    рынка — только относительное сравнение. Добавлено по прямому запросу:
    похоже, реальный источник части неудачных сделок — рекомендации против
    общего направления рынка, которые ничем не фильтровались.
    Значения: +1 индекс растёт, -1 индекс падает, 0/NaN — нет данных.
    """
    if index_df is None or index_df.empty:
        df["index_trend_up"] = np.nan
        return df

    idx = index_df[["Date", "Close"]].copy().drop_duplicates(subset="Date").sort_values("Date")
    idx["idx_ema_fast"] = idx["Close"].ewm(span=fast, adjust=False).mean()
    idx["idx_ema_slow"] = idx["Close"].ewm(span=slow, adjust=False).mean()
    idx["index_trend_up"] = np.sign(idx["idx_ema_fast"] - idx["idx_ema_slow"])

    base = df[["Date"]].sort_values("Date")
    aligned = pd.merge_asof(base, idx[["Date", "index_trend_up"]], on="Date", direction="backward")
    aligned = aligned.set_index(base.index).reindex(df.index)
    df["index_trend_up"] = aligned["index_trend_up"].values
    return df


# =============================================================================
# 3. СВЕЧНЫЕ ПАТТЕРНЫ (через pandas-ta-classic — надёжнее самодельных правил)
# =============================================================================
def detect_candle_patterns(df, context_bars=60):
    """
    Возвращает список паттернов, сработавших на последней ЗАКРЫТОЙ свече.

    ВАЖНО (фикс производительности): нам нужен только ПОСЛЕДНИЙ бар, а не вся
    история. Раньше здесь считалось на всём переданном df — в живом прогоне
    это одна лишняя, но терпимая трата; в бэктесте, где на каждом шаге
    передаётся растущее окно (к концу истории — почти вся история тикера),
    это давало квадратичный рост времени и превращало 20 тикеров в 36 минут
    вместо ожидаемых пары минут. Свечным паттернам в принципе не нужно
    больше 10-15 баров контекста — берём с запасом 60 и не теряем в точности.
    """
    if len(df) < 5:
        return []
    window = df.tail(context_bars) if len(df) > context_bars else df
    ta_input = window.rename(columns={
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Volume": "volume"
    }).set_index("Date")
    try:
        cdl = ta_input.ta.cdl_pattern(name="all")
        if cdl is not None:
            cdl = cdl.reindex(ta_input.index)  # см. комментарий в add_indicators про reindex
    except Exception:
        return []
    if cdl is None or cdl.empty:
        return []

    last = cdl.iloc[-1]
    found = []
    for col, val in last.items():
        if val == 0 or pd.isna(val):
            continue
        base_name = re.sub(r"_\d+_[\d.]+$", "", col).upper()
        if base_name not in CDL_NAMES_RU:
            continue  # осознанно отфильтровано — см. комментарий у CDL_NAMES_RU
        name_ru = CDL_NAMES_RU[base_name]
        direction = "бычий" if val > 0 else "медвежий"
        # префикс "Свеча:" — чтобы отличать от графических (структурных)
        # паттернов при разборе в compute_signal_components: одна свеча —
        # намного более слабый и краткосрочный сигнал, чем подтверждённая
        # фигура на свингах с измеренной целью, их нельзя весить одинаково
        found.append(f"Свеча: {name_ru} ({direction})")
    return found


# =============================================================================
# 4. ГРАФИЧЕСКИЕ ПАТТЕРНЫ (свинги) И ДИВЕРГЕНЦИИ
# =============================================================================
def find_swings(series, order):
    values = series.values
    highs_idx = argrelextrema(values, np.greater_equal, order=order)[0]
    lows_idx = argrelextrema(values, np.less_equal, order=order)[0]
    highs_idx = np.array(sorted(set(highs_idx.tolist())))
    lows_idx = np.array(sorted(set(lows_idx.tolist())))
    return highs_idx, lows_idx


FIB_RATIOS = [0.236, 0.382, 0.5, 0.618, 0.786]
FIB_EXT_RATIOS = [1.272, 1.618]


def compute_fibonacci_signal(df, lookback, swing_order, tolerance_atr_mult=0.35):
    """
    Берёт последний значимый свинг (от последнего экстремума high к последнему
    экстремуму low или наоборот — какой из них более свежий) и считает
    стандартные уровни коррекции Фибоначчи (23.6/38.2/50/61.8/78.6%) плюс
    уровни расширения (127.2/161.8%) от него. Если текущая цена находится
    рядом (в пределах ~0.35 ATR) с одним из уровней коррекции — это
    потенциальная зона отскока/продолжения тренда, в направлении самого
    свинга. Уровни расширения используются как ориентир для тейк-профита,
    а не как отдельный сигнал.

    ВАЖНО: сами по себе уровни Фибо — это, по сути, психологические зоны
    (многие трейдеры на них смотрят, отсюда самосбывающийся эффект), а не
    что-то, доказанное статистически само по себе. Компонент "fib_level"
    в score калибруется по бэктесту наравне со всеми остальными — если
    эффекта нет, вес станет 0, как и с дивергенцией/незавершёнными паттернами.
    """
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return None, []

    high_idx, low_idx = find_swings(window["High"], swing_order)
    high_idx2, low_idx2 = find_swings(window["Low"], swing_order)
    # берём последний экстремум high и последний экстремум low в окне
    last_high_i = high_idx[-1] if len(high_idx) else None
    last_low_i = low_idx2[-1] if len(low_idx2) else None
    if last_high_i is None or last_low_i is None:
        return None, []

    close_last = window["Close"].iloc[-1]
    atr_last = df["ATR14"].iloc[-1] if "ATR14" in df.columns else close_last * 0.01
    tol = atr_last * tolerance_atr_mult

    # Свинг вверх (low раньше, high позже) → коррекция вниз от high, уровни
    # коррекции ниже high; отскок ВВЕРХ от уровня коррекции = продолжение
    # восходящего движения (long). Свинг вниз — зеркально.
    if last_low_i < last_high_i:
        swing_low, swing_high = window["Low"].iloc[last_low_i], window["High"].iloc[last_high_i]
        direction_bias = "long"
    else:
        swing_low, swing_high = window["Low"].iloc[last_low_i], window["High"].iloc[last_high_i]
        direction_bias = "short"

    span = swing_high - swing_low
    if span <= 0:
        return None, []

    levels = {}
    for r in FIB_RATIOS:
        levels[f"{r*100:.1f}%"] = (swing_high - span * r) if direction_bias == "long" else (swing_low + span * r)
    for r in FIB_EXT_RATIOS:
        levels[f"ext {r*100:.1f}%"] = (swing_high + span * (r - 1)) if direction_bias == "long" else (swing_low - span * (r - 1))

    near_level = None
    for label, price in levels.items():
        if abs(close_last - price) <= tol:
            near_level = (label, price)
            break

    found = []
    if near_level:
        label, price = near_level
        tag = "бычий" if direction_bias == "long" else "медвежий"
        found.append(f"Цена у уровня Фибоначчи {label} ({tag})")

    return {"levels": levels, "direction_bias": direction_bias, "near": near_level}, found


def detect_wolfe_wave(df, lookback, swing_order):
    """
    Упрощённая эвристика волны Вульфа (Wolfe Wave) — 5-точечная разворотная
    формация: точки 1-3-5 (одна сторона) против 2-4 (другая), где точка 5
    пробивает продолжение линии 1-3 ("ложный пробой" уровня), а точка 4
    остаётся внутри канала. Цель разворота — линия 1-4, продлённая вперёд
    (EPA, estimated price at arrival).

    ЧЕСТНО: это одна из самых субъективных техник в техническом анализе —
    даже опытные трейдеры часто расходятся, есть ли на графике волна Вульфа.
    Здесь это грубая геометрическая эвристика по свингам, а не строгая
    валидация. Вес компонента "wolfe_wave" в score по умолчанию 0 — его
    включает только калибровка по бэктесту, и только если на исторических
    данных он покажет реальный, а не воображаемый эффект.
    """
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 40:
        return []

    high_idx, _ = find_swings(window["High"], swing_order)
    _, low_idx = find_swings(window["Low"], swing_order)
    # argrelextrema ненадёжно определяет экстремумы у самой границы окна —
    # отсекаем свинги в первых/последних swing_order барах как потенциально
    # ложные краевые артефакты
    edge = swing_order
    high_idx = high_idx[(high_idx >= edge) & (high_idx < len(window) - edge)]
    low_idx = low_idx[(low_idx >= edge) & (low_idx < len(window) - edge)]

    # объединяем свинги в один чередующийся список точек (время, цена, тип)
    points = [(i, window["High"].iloc[i], "H") for i in high_idx] + \
             [(i, window["Low"].iloc[i], "L") for i in low_idx]
    points.sort(key=lambda p: p[0])
    if len(points) < 5:
        return []

    # Фильтр значимости (упрощённый zigzag): мелкие колебания на хвосте окна
    # иначе забивают "последние 5 точек" шумом вместо структуры самой волны —
    # оставляем только развороты не меньше ~2 ATR.
    atr_last = df["ATR14"].iloc[-1] if "ATR14" in df.columns and pd.notna(df["ATR14"].iloc[-1]) else None
    min_move = (atr_last * 2.0) if atr_last else (window["Close"].std() * 0.5)

    filtered = [points[0]]
    for p in points[1:]:
        prev = filtered[-1]
        if p[2] == prev[2]:
            # два свинга одного типа подряд (бывает, т.к. high/low ищутся раздельно) —
            # оставляем более экстремальный
            if (p[2] == "H" and p[1] > prev[1]) or (p[2] == "L" and p[1] < prev[1]):
                filtered[-1] = p
            continue
        if abs(p[1] - prev[1]) < min_move:
            continue
        filtered.append(p)
    points = filtered
    if len(points) < 4:
        return []

    found = []
    # Точка 5 — это ТЕКУЩАЯ цена (ещё формируется), а не уже подтверждённый
    # свинг. Если ждать, пока точка 5 сама станет подтверждённым экстремумом,
    # сигнал по определению придёт на несколько баров позже, чем нужно —
    # смысл волны Вульфа именно в входе В МОМЕНТ прокола уровня 1-3, а не
    # после того, как разворот уже случился и стал виден постфактум.
    last4 = points[-4:]
    types4 = [p[2] for p in last4]
    p5_idx = len(window) - 1
    p5_price_low, p5_price_high = window["Low"].iloc[-1], window["High"].iloc[-1]

    # Бычья волна Вульфа: 1-4 = L,H,L,H — точки 1,3,5 образуют нисходящую линию
    # поддержки, точки 2,4 — вторую (примерно параллельную) линию сверху.
    # Условия: точка 3 ниже точки 1 (нисходящая линия опоры), точка 4 ниже
    # точки 2 (канал сужается/снижается), точка 5 (текущая цена) пробивает
    # линию 1-3 вниз — это и есть вход в лонг на "ложном" пробое поддержки.
    if types4 == ["L", "H", "L", "H"]:
        p1, p2, p3, p4 = last4
        if p3[0] != p1[0] and p4[0] != p1[0] and p3[1] < p1[1] and p4[1] < p2[1]:
            slope13 = (p3[1] - p1[1]) / (p3[0] - p1[0])
            line13_at_5 = p1[1] + slope13 * (p5_idx - p1[0])
            if p5_price_low < line13_at_5:
                slope14 = (p4[1] - p1[1]) / (p4[0] - p1[0])
                epa = p1[1] + slope14 * (p5_idx - p1[0])
                if epa > p5_price_low:
                    found.append(f"Волна Вульфа (бычья, цель ~{smart_round(epa)})")

    # Медвежья волна Вульфа: 1-4 = H,L,H,L — зеркально, восходящий канал,
    # точка 5 пробивает линию 1-3 вверх ("ложный" пробой сопротивления)
    if types4 == ["H", "L", "H", "L"]:
        p1, p2, p3, p4 = last4
        if p3[0] != p1[0] and p4[0] != p1[0] and p3[1] > p1[1] and p4[1] > p2[1]:
            slope13 = (p3[1] - p1[1]) / (p3[0] - p1[0])
            line13_at_5 = p1[1] + slope13 * (p5_idx - p1[0])
            if p5_price_high > line13_at_5:
                slope14 = (p4[1] - p1[1]) / (p4[0] - p1[0])
                epa = p1[1] + slope14 * (p5_idx - p1[0])
                if epa < p5_price_high:
                    found.append(f"Волна Вульфа (медвежья, цель ~{smart_round(epa)})")

    return found


def pattern_stage_label(breakout_price, target_price, current_price):
    """
    Насколько далеко цена уже прошла от точки пробоя к измеренной цели
    (классическое правило "высота фигуры = размер хода" — для Г-и-П,
    двойной вершины/дна, пробоя диапазона). Прогресс < 15% — пробой только
    что случился; 15-85% — паттерн УЖЕ ОТРАБАТЫВАЕТ прямо сейчас (это
    и есть то самое "уже начинает реализовываться", а не только прогноз);
    > 85% — цель почти достигнута, заходить по нему уже поздновато, риск,
    что движение исчерпано.
    """
    total = target_price - breakout_price
    if total == 0:
        return None
    progress = (current_price - breakout_price) / total
    if progress < 0:
        return None
    if progress < 0.15:
        return "свежий пробой"
    elif progress < 0.85:
        return "уже отрабатывает"
    else:
        return "близко к цели, вероятно исчерпан"


def detect_chart_patterns(df, lookback, swing_order):
    found = []
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return found

    highs, lows = window["High"], window["Low"]
    close_last = window["Close"].iloc[-1]
    high_idx, _ = find_swings(highs, swing_order)
    _, low_idx = find_swings(lows, swing_order)
    tol = window["Close"].std() * 0.6 if window["Close"].std() > 0 else close_last * 0.01

    if len(high_idx) >= 3:
        last3 = high_idx[-3:]
        h_vals = highs.iloc[last3].values
        left_sh, head, right_sh = h_vals[0], h_vals[1], h_vals[2]
        if head > left_sh + tol and head > right_sh + tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = lows.iloc[last3[0]:last3[2]].min()
            if close_last < neckline:
                target = neckline - (head - neckline)
                stage = pattern_stage_label(neckline, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Голова и плечи (подтверждено пробоем шеи вниз{tag})")
            else:
                found.append("Голова и плечи (формируется, шея не пробита)")

    if len(low_idx) >= 3:
        last3 = low_idx[-3:]
        l_vals = lows.iloc[last3].values
        left_sh, head, right_sh = l_vals[0], l_vals[1], l_vals[2]
        if head < left_sh - tol and head < right_sh - tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = highs.iloc[last3[0]:last3[2]].max()
            if close_last > neckline:
                target = neckline + (neckline - head)
                stage = pattern_stage_label(neckline, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Перевёрнутая голова и плечи (подтверждено пробоем вверх{tag})")
            else:
                found.append("Перевёрнутая голова и плечи (формируется)")

    if len(high_idx) >= 2:
        h1, h2 = highs.iloc[high_idx[-2]], highs.iloc[high_idx[-1]]
        if abs(h1 - h2) < tol and (high_idx[-1] - high_idx[-2]) > swing_order * 2:
            trough = lows.iloc[high_idx[-2]:high_idx[-1]].min()
            if close_last < trough:
                top_level = (h1 + h2) / 2
                target = trough - (top_level - trough)
                stage = pattern_stage_label(trough, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Двойная вершина (подтверждена пробоем вниз{tag})")
            else:
                found.append("Двойная вершина (формируется)")

    if len(low_idx) >= 2:
        l1, l2 = lows.iloc[low_idx[-2]], lows.iloc[low_idx[-1]]
        if abs(l1 - l2) < tol and (low_idx[-1] - low_idx[-2]) > swing_order * 2:
            peak = highs.iloc[low_idx[-2]:low_idx[-1]].max()
            if close_last > peak:
                bottom_level = (l1 + l2) / 2
                target = peak + (peak - bottom_level)
                stage = pattern_stage_label(peak, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Двойное дно (подтверждено пробоем вверх{tag})")
            else:
                found.append("Двойное дно (формируется)")

    if len(high_idx) >= 3 and len(low_idx) >= 3:
        hx, hy = high_idx[-3:], highs.iloc[high_idx[-3:]].values
        lx, ly = low_idx[-3:], lows.iloc[low_idx[-3:]].values
        slope_high = np.polyfit(hx, hy, 1)[0]
        slope_low = np.polyfit(lx, ly, 1)[0]
        flat = close_last * 0.0006
        if abs(slope_high) < flat and slope_low > flat:
            found.append("Восходящий треугольник")
        elif slope_high < -flat and abs(slope_low) < flat:
            found.append("Нисходящий треугольник")
        elif slope_high < -flat and slope_low > flat:
            found.append("Симметричный треугольник (сужение, ждём пробоя)")

    range_high = window["High"].iloc[-lookback:-1].max()
    range_low = window["Low"].iloc[-lookback:-1].min()
    last_vol, avg_vol = window["Volume"].iloc[-1], window["Volume"].iloc[:-1].mean()
    if last_vol > avg_vol * 1.5:
        range_height = range_high - range_low
        if close_last > range_high:
            target = range_high + range_height
            stage = pattern_stage_label(range_high, target, close_last)
            tag = f", {stage}" if stage else ""
            found.append(f"Пробой диапазона вверх на объёме{tag}")
        elif close_last < range_low:
            target = range_low - range_height
            stage = pattern_stage_label(range_low, target, close_last)
            tag = f", {stage}" if stage else ""
            found.append(f"Пробой диапазона вниз на объёме{tag}")

    return found


def detect_divergence(df, lookback, swing_order):
    found = []
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return found

    _, low_idx = find_swings(window["Low"], swing_order)
    high_idx, _ = find_swings(window["High"], swing_order)

    if len(low_idx) >= 2:
        i1, i2 = low_idx[-2], low_idx[-1]
        if (window["Low"].iloc[i2] < window["Low"].iloc[i1] and
                (window["RSI14"].iloc[i2] > window["RSI14"].iloc[i1] or
                 window["MACD_hist"].iloc[i2] > window["MACD_hist"].iloc[i1])):
            found.append("Бычья дивергенция (цена ниже, осциллятор выше)")

    if len(high_idx) >= 2:
        i1, i2 = high_idx[-2], high_idx[-1]
        if (window["High"].iloc[i2] > window["High"].iloc[i1] and
                (window["RSI14"].iloc[i2] < window["RSI14"].iloc[i1] or
                 window["MACD_hist"].iloc[i2] < window["MACD_hist"].iloc[i1])):
            found.append("Медвежья дивергенция (цена выше, осциллятор ниже)")

    return found


# =============================================================================
# 5. ML-МОДЕЛЬ
# =============================================================================
FEATURES = ["RSI14", "MACD_hist", "ATR_pct", "BB_pctB", "Vol_ratio",
            "trend_up", "trend_down", "ADX14", "price_vs_vwap_pct",
            "STOCH_k", "CCI14", "WILLR14", "rel_strength_pct"]


def build_ml_dataset(df, horizon, up_thresh_atr):
    # Защитный барьер от дублирующихся колонок (см. комментарий в add_indicators) —
    # без него операции ниже могут внезапно начать сравнивать DataFrame с DataFrame
    # вместо Series с Series и падать с "identically-labeled" ошибкой.
    d = df.loc[:, ~df.columns.duplicated()].copy()
    future_close = d["Close"].shift(-horizon)
    move = future_close - d["Close"]
    d["target"] = (move > up_thresh_atr * d["ATR14"]).astype(int)
    d = d.dropna(subset=FEATURES + ["target"])
    return d


def train_and_predict(all_data_by_ticker, horizon, up_thresh_atr):
    frames = []
    failed_tickers = []
    for ticker, df in all_data_by_ticker.items():
        # Один "проблемный" тикер (нетипичная форма данных, редкий edge-case
        # от pandas-ta и т.п.) не должен ронять обучение модели по всем
        # остальным — ловим и пропускаем, а не падаем всем пайплайном.
        try:
            ds = build_ml_dataset(df, horizon, up_thresh_atr)
            if len(ds) > 50:
                ds = ds.copy()
                ds["ticker"] = ticker
                frames.append(ds)
        except Exception as e:
            failed_tickers.append((ticker, str(e)))

    if failed_tickers:
        print(f"   Пропущено при построении ML-датасета: {len(failed_tickers)} тикеров "
              f"(напр. {failed_tickers[0][0]}: {failed_tickers[0][1][:80]})")

    if not frames:
        return None, None

    full = pd.concat(frames, ignore_index=True).sort_values("Date")
    split_idx = int(len(full) * 0.8)
    train, valid = full.iloc[:split_idx], full.iloc[split_idx:]

    model = RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=50,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
    model.fit(train[FEATURES], train["target"])

    auc = None
    if valid["target"].nunique() > 1:
        preds = model.predict_proba(valid[FEATURES])[:, 1]
        auc = roc_auc_score(valid["target"], preds)
    return model, auc


# =============================================================================
# 6. РИСК-МЕНЕДЖМЕНТ (только уровни — без расчёта размера позиции)
# =============================================================================
def compute_risk_levels(last_row, direction, atr_mult, rr_target, bid=None, offer=None):
    """
    bid/offer — текущий стакан (необязательные). Если переданы, вход считается
    по реалистичной цене исполнения (long заходит по офферу/аску, short — по
    биду), а не по last close, который на споте недостижим по определению —
    вы либо покупаете дороже close (по офферу), либо продаёте дешевле (по биду).
    Также считает спред и его долю от риска на сделку — узкий стоп на бумаге
    с широким спредом может не окупать сам спред при входе и выходе.
    """
    close_price, atr = last_row["Close"], last_row["ATR14"]

    entry_price = close_price
    if direction == "long" and offer and offer > 0:
        entry_price = offer
    elif direction == "short" and bid and bid > 0:
        entry_price = bid

    min_risk = entry_price * 0.001

    if direction == "long":
        stop = entry_price - atr_mult * atr
        risk_per_unit = max(entry_price - stop, min_risk)
        target = entry_price + risk_per_unit * rr_target
    else:
        stop = entry_price + atr_mult * atr
        risk_per_unit = max(stop - entry_price, min_risk)
        target = entry_price - risk_per_unit * rr_target

    spread_abs, spread_pct, risk_per_spread = None, None, None
    if bid and offer and bid > 0 and offer > bid:
        spread_abs = offer - bid
        mid = (offer + bid) / 2
        spread_pct = spread_abs / mid * 100
        risk_per_spread = risk_per_unit / spread_abs if spread_abs > 0 else None

    spread_warning = None
    if risk_per_spread is not None:
        if risk_per_spread < 3:
            spread_warning = "Спред широкий относительно стопа — исполнение съест заметную часть риска"
        elif risk_per_spread < 6:
            spread_warning = "Спред заметный, учитывайте при входе/выходе"

    return {
        "entry": smart_round(entry_price),
        "stop_loss": smart_round(stop),
        "take_profit": smart_round(target),
        "risk_reward": rr_target,
        "spread_pct": round(spread_pct, 3) if spread_pct is not None else None,
        "risk_per_spread": round(risk_per_spread, 1) if risk_per_spread is not None else None,
        "spread_warning": spread_warning,
    }


# =============================================================================
# 7. СБОРКА ИТОГОВОГО SCORE
# =============================================================================
# Веса по умолчанию (мои изначальные, "на глаз") — используются, если веса
# ещё не откалиброваны по бэктесту, либо для конкретного компонента не
# набралось достаточно исторических случаев, чтобы доверять его лифту.
DEFAULT_WEIGHTS = {
    "trend": 1.5,
    "index_trend": 1.2,   # НОВОЕ: тренд самого IMOEX — раньше в системе не было
                           # вообще никакого учёта общего направления рынка,
                           # только относительная сила (обгоняет ли акция
                           # индекс), а это не то же самое, что направление
                           # самого индекса. Вес выше среднего осознанно —
                           # торговать против всего рынка обычно плохая идея,
                           # но конечный вес всё равно проверит калибровка
    "rsi_extreme": 1.0,
    "macd": 0.5,
    "stoch_extreme": 0.4,
    "willr_extreme": 0.3,
    "vwap_dev": 0.3,
    "pattern_confirmed": 1.3,   # ГРАФИЧЕСКИЙ паттерн (Г-и-П, двойное дно и т.д.) подтверждён пробоем
    "pattern_forming": 0.6,     # графический паттерн ещё формируется, пробоя не было
    "pattern_playing_out": 1.0,   # паттерн подтверждён и цена уже частично прошла путь к цели
    "candle_pattern": 0.4,      # свечной паттерн (1 бар) — намного слабее графического, вес ниже
    "divergence": 1.5,
    "rel_strength": 0.5,
    "fib_level": 0.0,   # новый, более спекулятивный компонент — вес по умолчанию 0,
    "wolfe_wave": 0.0,  # его включает только калибровка по бэктесту, если найдёт эффект
    "daily_pattern": 0.0,   # паттерн/дивергенция подтвердились ЕЩЁ И на дневном таймфрейме
}
COMPONENT_NAMES = list(DEFAULT_WEIGHTS.keys())


PATTERN_BULLISH_KW = ["бычий", "вверх", "дно", "Перевёрнутая"]
PATTERN_BEARISH_KW = ["медвежий", "вниз", "вершина"]


def format_patterns_for_display(patterns, direction):
    """
    Раньше все найденные паттерны просто перечислялись подряд в одну строку —
    бычьи и медвежьи вперемешку, свечные наравне с графическими, без
    указания, что из этого вообще поддерживает итоговую рекомендацию.
    Теперь делим явно на "ЗА" (согласны с направлением) и "ПРОТИВ"
    (противоречат ему) — так сразу видно совокупную картину, а не приходится
    самому парсить слова "бычий"/"медвежий" в каждой записи.
    Паттерны без явного направления (треугольники, "цена у уровня Фибоначчи"
    без явного бычий/медвежий и т.п.) попадают в "ЗА" только если явно
    совпадают, иначе не показываются как конфликтующие.
    """
    if not patterns:
        return "-"
    is_long = direction == "long"
    supporting, conflicting = [], []
    for p in patterns:
        is_bull = any(k in p for k in PATTERN_BULLISH_KW)
        is_bear = any(k in p for k in PATTERN_BEARISH_KW)
        if not (is_bull or is_bear):
            continue  # паттерны без направления (напр. просто "Восходящий треугольник") пропускаем в этом резюме
        aligned = (is_bull and is_long) or (is_bear and not is_long)
        (supporting if aligned else conflicting).append(p)

    parts = []
    if supporting:
        parts.append("ЗА: " + "; ".join(supporting))
    if conflicting:
        parts.append("ПРОТИВ: " + "; ".join(conflicting))
    return " | ".join(parts) if parts else "-"


def compute_signal_components(last, patterns):
    """
    Каждый компонент теперь голосует ЗНАКОМ: +1 = тянет в лонг, -1 = тянет
    в шорт, 0 = не высказался. Раньше все сработавшие компоненты просто
    СКЛАДЫВАЛИСЬ в score независимо от направления — то есть бумага с явно
    противоречивыми сигналами (часть тянет вверх, часть вниз) могла набрать
    высокий score просто потому что сработало МНОГО всего, а не потому что
    сигналы реально совпадали. Это был системный баг, искажавший доверие
    к высокому score. Теперь направление и score считаются ОДНОВременно и
    согласованно в score_from_components — противоречащие друг другу сигналы
    взаимно гасятся, а не суммируются.
    """
    c = {name: 0 for name in COMPONENT_NAMES}

    if last["trend_up"]:
        c["trend"] = 1
    elif last["trend_down"]:
        c["trend"] = -1

    if last["RSI14"] < 30:
        c["rsi_extreme"] = 1
    elif last["RSI14"] > 70:
        c["rsi_extreme"] = -1

    if last.get("MACD_cross_up", 0) == 1:
        c["macd"] = 1
    elif last.get("MACD_cross_down", 0) == 1:
        c["macd"] = -1

    stoch_k = last.get("STOCH_k", np.nan)
    if pd.notna(stoch_k):
        if stoch_k < 20:
            c["stoch_extreme"] = 1
        elif stoch_k > 80:
            c["stoch_extreme"] = -1

    willr = last.get("WILLR14", np.nan)
    if pd.notna(willr):
        if willr < -80:
            c["willr_extreme"] = 1
        elif willr > -20:
            c["willr_extreme"] = -1

    vwap_dev = last.get("price_vs_vwap_pct", 0)
    if pd.notna(vwap_dev):
        if vwap_dev > 0.3:
            c["vwap_dev"] = 1
        elif vwap_dev < -0.3:
            c["vwap_dev"] = -1

    index_trend = last.get("index_trend_up")
    if index_trend is not None and pd.notna(index_trend):
        c["index_trend"] = 1 if index_trend > 0 else (-1 if index_trend < 0 else 0)

    bullish_kw, bearish_kw = PATTERN_BULLISH_KW, PATTERN_BEARISH_KW
    for p in patterns:
        if "Вульфа" in p:
            c["wolfe_wave"] = 1 if "бычья" in p else -1
            continue
        if "Фибоначчи" in p:
            c["fib_level"] = 1 if "бычий" in p else -1
            continue

        is_bull = any(k in p for k in bullish_kw)
        is_bear = any(k in p for k in bearish_kw)
        if not (is_bull or is_bear):
            continue
        sign = 1 if is_bull else -1

        if "Свеча:" in p:
            # свечной паттерн (1 бар) — отдельный, более слабый компонент.
            # Не смешиваем с графическими паттернами, у которых есть
            # измеренная цель и подтверждение пробоем — это разного калибра
            # сигналы, и раньше они считались одним компонентом, размывая
            # калибровку (доджи и подтверждённый пробой весили одинаково)
            c["candle_pattern"] = sign
            continue

        if "уже отрабатывает" in p:
            c["pattern_playing_out"] = sign
        confirmed = "подтвержд" in p or "Пробой" in p
        if confirmed:
            c["pattern_confirmed"] = sign
        else:
            c["pattern_forming"] = sign

    if any("Бычья дивергенция" in p for p in patterns):
        c["divergence"] = 1
    if any("Медвежья дивергенция" in p for p in patterns):
        c["divergence"] = -1

    if any("Дневной паттерн" in p or "Дневная дивергенция" in p for p in patterns):
        # у дневного контекста нет единого простого знака (там может быть и
        # бычье, и медвежье одновременно) — этот компонент отмечает сам факт
        # подтверждения на дневках, знак берём от самих строк паттернов
        daily_bull = any(("Дневной" in p or "Дневная" in p) and any(k in p for k in bullish_kw) for p in patterns)
        daily_bear = any(("Дневной" in p or "Дневная" in p) and any(k in p for k in bearish_kw) for p in patterns)
        if daily_bull and not daily_bear:
            c["daily_pattern"] = 1
        elif daily_bear and not daily_bull:
            c["daily_pattern"] = -1

    rel_strength = last.get("rel_strength_pct", np.nan)
    if pd.notna(rel_strength):
        if rel_strength > 1.0:
            c["rel_strength"] = 1
        elif rel_strength < -1.0:
            c["rel_strength"] = -1

    return c


def score_from_components(last, components, weights, ml_prob, ml_auc, daily_trend_up):
    """
    net > 0 — суммарно перевешивает лонг, net < 0 — шорт. Величина |net| —
    это и есть score: насколько СОГЛАСОВАННО (а не просто "насколько много
    всего сработало") сигналы указывают в одну сторону. Противоречащие друг
    другу компоненты по конструкции взаимно вычитаются.
    """
    net = sum(weights.get(name, 0.0) * val for name, val in components.items())
    direction = "long" if net >= 0 else "short"

    adx = last.get("ADX14", 0)
    trend_strength_mult = 1.3 if adx > 25 else (0.6 if adx < 20 else 1.0)
    score = abs(net) * trend_strength_mult

    if daily_trend_up is not None:
        aligned = (daily_trend_up and direction == "long") or (not daily_trend_up and direction == "short")
        score *= 1.25 if aligned else 0.7

    if ml_prob is not None:
        edge = (1 - ml_prob) - 0.5 if direction == "short" else ml_prob - 0.5
        auc_confidence = max(min((ml_auc - 0.5) * 10, 1.0), 0.0) if ml_auc else 0.3
        score += max(edge, 0) * 4 * auc_confidence

    return round(score, 2), direction


def score_ticker(last, patterns, ml_prob, ml_auc=None, daily_trend_up=None, weights=None):
    weights = weights or DEFAULT_WEIGHTS
    components = compute_signal_components(last, patterns)
    score, direction = score_from_components(last, components, weights, ml_prob, ml_auc, daily_trend_up)
    return score, direction


# =============================================================================
# 8. БЭКТЕСТ + КАЛИБРОВКА ВЕСОВ ПО ФАКТИЧЕСКИМ ДАННЫМ
# =============================================================================
def simulate_trade_outcome(future_df, direction, stop, target, max_holding_bars):
    """
    Идёт вперёд по факту истории и смотрит, что было пробито раньше — стоп
    или тейк. Если в один и тот же бар пробито и то, и другое (случается на
    часовиках при резких свечах) — консервативно считаем, что сработал стоп
    (так безопаснее для оценки: не завышаем результат бэктеста).
    """
    for _, row in future_df.head(max_holding_bars).iterrows():
        high, low = row["High"], row["Low"]
        if direction == "long":
            hit_stop, hit_target = low <= stop, high >= target
        else:
            hit_stop, hit_target = high >= stop, low <= target
        if hit_stop:
            return "stop", stop
        if hit_target:
            return "target", target
    tail = future_df.head(max_holding_bars)
    if len(tail) > 0:
        return "timeout", tail["Close"].iloc[-1]
    return "no_data", None


def simulate_trade_path(future_df, direction, entry, stop, target, max_holding_bars):
    """
    Более подробная версия simulate_trade_outcome: помимо исхода, считает
    MAE (Maximum Adverse Excursion — насколько глубоко цена уходила ПРОТИВ
    позиции до выхода) и MFE (Maximum Favorable Excursion — насколько
    далеко в ПОЛЬЗУ позиции), обе в единицах риска (R), и классифицирует
    "форму" движения. Это прямой ответ на вопрос "цена сразу пошла против,
    колебалась, или шла по сценарию, но не дошла" — вместо того чтобы просто
    знать финальный результат, видно ЧТО происходило по пути.
    """
    risk = abs(entry - stop)
    if risk == 0:
        return {"outcome": "no_data", "exit_price": None, "bars_held": 0,
                "mae_r": None, "mfe_r": None, "path_label": "нет данных"}

    max_adverse_r, max_favorable_r = 0.0, 0.0
    bars_held = 0
    outcome, exit_price = "timeout", None

    for _, row in future_df.head(max_holding_bars).iterrows():
        bars_held += 1
        high, low = row["High"], row["Low"]
        if direction == "long":
            adverse_r = (entry - low) / risk
            favorable_r = (high - entry) / risk
            hit_stop, hit_target = low <= stop, high >= target
        else:
            adverse_r = (high - entry) / risk
            favorable_r = (entry - low) / risk
            hit_stop, hit_target = high >= stop, low <= target

        max_adverse_r = max(max_adverse_r, adverse_r)
        max_favorable_r = max(max_favorable_r, favorable_r)

        if hit_stop:
            outcome, exit_price = "stop", stop
            break
        if hit_target:
            outcome, exit_price = "target", target
            break
    else:
        tail = future_df.head(max_holding_bars)
        if len(tail) > 0:
            exit_price = tail["Close"].iloc[-1]
        outcome = "timeout"

    final_r = r_multiple(direction, entry, stop, exit_price) if exit_price is not None else None

    # Классификация "формы" движения — см. докстринг
    if outcome == "stop":
        path_label = "Сразу против" if max_favorable_r < 0.3 else "Сходил в плюс, потом развернулся и выбил"
    elif outcome == "target":
        path_label = "Дошёл до цели с сильной просадкой по пути" if max_adverse_r > 0.5 else "Дошёл до цели уверенно"
    elif outcome == "timeout":
        if final_r is not None and final_r > 0.5:
            path_label = "Шёл по сценарию, не успел дойти за отведённое время"
        elif final_r is not None and final_r < -0.3:
            path_label = "Шёл против, но не успел выбить стоп"
        else:
            path_label = "Болтался без выраженного направления"
    else:
        path_label = "нет данных"

    return {"outcome": outcome, "exit_price": exit_price, "bars_held": bars_held,
            "mae_r": round(max_adverse_r, 3), "mfe_r": round(max_favorable_r, 3),
            "r_multiple": round(final_r, 3) if final_r is not None else None,
            "path_label": path_label}


def r_multiple(direction, entry, stop, exit_price):
    risk = abs(entry - stop)
    if risk == 0 or exit_price is None:
        return None
    pnl = (exit_price - entry) if direction == "long" else (entry - exit_price)
    return pnl / risk


def backtest_ticker(df, config):
    """
    Идёт по истории тикера с шагом backtest_stride. На каждом шаге считает
    СЫРЫЕ компоненты сигнала (тренд/паттерны/дивергенция и т.д.) через ту же
    самую логику, что и в живом прогоне, направление и уровни входа/стопа/
    тейка — и смотрит, что случилось дальше по факту истории.

    Записывает именно компоненты, а не готовый score с весами — чтобы потом
    можно было посчитать, какие компоненты статистически связаны с прибылью,
    БЕЗ повторного прогона бэктеста (веса пересчитываются задним числом по
    уже собранным данным, это быстро).

    ML не участвует — модель обучена на всём периоде разом, подмешивать её
    сюда значит частично тестировать на данных, которые она уже видела при
    обучении, это нечестно.
    """
    trades = []
    n = len(df)
    start = 210
    for i in range(start, n - 5, config["backtest_stride"]):
        window = df.iloc[:i + 1]
        last = window.iloc[-1]
        if pd.isna(last["ATR14"]) or last["ATR14"] <= 0:
            continue

        candle_p = detect_candle_patterns(window)
        chart_p = detect_chart_patterns(window, config["pattern_lookback_bars"], config["swing_order"])
        div_p = detect_divergence(window, config["pattern_lookback_bars"], config["swing_order"])
        _, fib_p = compute_fibonacci_signal(window, config["pattern_lookback_bars"], config["swing_order"])
        wolfe_p = detect_wolfe_wave(window, config["pattern_lookback_bars"], config["swing_order"])
        patterns = candle_p + chart_p + div_p + fib_p + wolfe_p

        components = compute_signal_components(last, patterns)
        # Направление в бэктесте всегда решается ПО ДЕФОЛТНЫМ весам (не по уже
        # подобранным ранее) — иначе получается циклическая зависимость:
        # калибровка решала бы направление сама для себя. Это соответствует
        # тому, что произошло бы при самом первом запуске без истории калибровки.
        _, direction = score_from_components(last, components, DEFAULT_WEIGHTS, None, None, None)
        risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"])

        future = df.iloc[i + 1:]
        if future.empty:
            continue
        path = simulate_trade_path(
            future, direction, risk["entry"], risk["stop_loss"], risk["take_profit"],
            config["backtest_max_holding_bars"]
        )
        if path["r_multiple"] is not None:
            row = dict(components)
            row.update({"direction": direction, "outcome": path["outcome"],
                        "r_multiple": path["r_multiple"], "mae_r": path["mae_r"],
                        "mfe_r": path["mfe_r"], "path_label": path["path_label"],
                        "bars_held": path["bars_held"], "ADX14": last.get("ADX14", 0)})
            trades.append(row)

    return pd.DataFrame(trades)


def stop_multiplier_sensitivity(df, config, multipliers):
    """
    Проверка гипотезы "может, 1.5×ATR — просто плохо подобранное число":
    для тех же самых точек входа/направления (по дефолтным весам) пересчитывает
    исход при РАЗНЫХ множителях стопа, используя тот же самый кусок истории.
    Если винрейт/средний R растут по мере расширения стопа (до какого-то
    предела) — это прямое доказательство, что текущий множитель слишком тесен
    и система режет будущих победителей раньше времени. Если нет —
    проблема не в ширине стопа, а где-то ещё (например, в самом направлении).
    """
    results = {m: [] for m in multipliers}
    n = len(df)
    start = 210
    for i in range(start, n - 5, config["stop_sensitivity_stride"]):
        window = df.iloc[:i + 1]
        last = window.iloc[-1]
        if pd.isna(last["ATR14"]) or last["ATR14"] <= 0:
            continue

        candle_p = detect_candle_patterns(window)
        chart_p = detect_chart_patterns(window, config["pattern_lookback_bars"], config["swing_order"])
        div_p = detect_divergence(window, config["pattern_lookback_bars"], config["swing_order"])
        _, fib_p = compute_fibonacci_signal(window, config["pattern_lookback_bars"], config["swing_order"])
        wolfe_p = detect_wolfe_wave(window, config["pattern_lookback_bars"], config["swing_order"])
        patterns = candle_p + chart_p + div_p + fib_p + wolfe_p

        components = compute_signal_components(last, patterns)
        _, direction = score_from_components(last, components, DEFAULT_WEIGHTS, None, None, None)

        future = df.iloc[i + 1:]
        if future.empty:
            continue

        for m in multipliers:
            risk = compute_risk_levels(last, direction, m, config["risk_reward_target"])
            path = simulate_trade_path(future, direction, risk["entry"], risk["stop_loss"],
                                        risk["take_profit"], config["backtest_max_holding_bars"])
            if path["r_multiple"] is not None:
                results[m].append({"outcome": path["outcome"], "r_multiple": path["r_multiple"]})

    return results


def run_stop_sensitivity_analysis(data_by_ticker, config):
    tickers = list(data_by_ticker.keys())[:config["stop_sensitivity_tickers"]]
    multipliers = config["stop_sensitivity_multipliers"]
    combined = {m: [] for m in multipliers}

    for t in tqdm(tickers, desc="   Проверка разных множителей стопа по тикерам"):
        try:
            res = stop_multiplier_sensitivity(data_by_ticker[t], config, multipliers)
            for m in multipliers:
                combined[m].extend(res[m])
        except Exception:
            continue

    rows = []
    for m in multipliers:
        trades = combined[m]
        if not trades:
            continue
        n = len(trades)
        win_rate = sum(1 for x in trades if x["outcome"] == "target") / n * 100
        avg_r = sum(x["r_multiple"] for x in trades) / n
        sum_r = sum(x["r_multiple"] for x in trades)
        rows.append({"Множитель ATR (стоп)": m, "Сделок": n, "Винрейт %": round(win_rate, 1),
                     "Средний_R": round(avg_r, 3), "Сумма_R": round(sum_r, 1)})

    return pd.DataFrame(rows)


def compute_component_aggregates(trades):
    """Сырые суммы/счётчики по каждому компоненту — то, что можно копить между
    запусками (в отличие от готового 'лифта', суммы корректно складываются).

    "Сработал" теперь означает "знак компонента совпал с итоговым направлением
    сделки" (компоненты знаковые: +1/0/-1), а не просто "компонент не ноль" —
    иначе противоречащие направлению сигналы засчитывались бы как поддержка."""
    aggs = {}
    direction_sign = trades["direction"].map({"long": 1, "short": -1})
    for name in COMPONENT_NAMES:
        if name not in trades.columns:
            continue
        mask = trades[name] == direction_sign
        aggs[name] = {
            "n_true": int(mask.sum()),
            "sum_true": float(trades.loc[mask, "r_multiple"].sum()),
            "n_false": int((~mask).sum()),
            "sum_false": float(trades.loc[~mask, "r_multiple"].sum()),
        }
    return aggs


def merge_aggregates(a, b):
    empty = {"n_true": 0, "sum_true": 0.0, "n_false": 0, "sum_false": 0.0}
    merged = {}
    for name in set(a.keys()) | set(b.keys()):
        av, bv = a.get(name, empty), b.get(name, empty)
        merged[name] = {
            "n_true": av["n_true"] + bv["n_true"],
            "sum_true": av["sum_true"] + bv["sum_true"],
            "n_false": av["n_false"] + bv["n_false"],
            "sum_false": av["sum_false"] + bv["sum_false"],
        }
    return merged


CALIBRATION_CACHE_KEY = "calibration_aggregates_v1"


def load_accumulated_calibration(cache_dir):
    try:
        cached = _load_from_cache(cache_dir, CALIBRATION_CACHE_KEY, ttl_hours=24 * 30)
        return cached.get("value") if isinstance(cached, dict) and "value" in cached else cached
    except Exception:
        return None


def save_accumulated_calibration(cache_dir, aggs):
    try:
        _save_to_cache(cache_dir, CALIBRATION_CACHE_KEY, aggs)
    except Exception:
        pass


def fit_weights_from_aggregates(aggs, min_samples=40):
    """То же самое, что fit_weights_from_backtest, но по накопленным суммам/
    счётчикам (возможно, из нескольких прогонов) вместо разовой таблицы
    сделок — чем больше накоплено запусков, тем устойчивее веса."""
    diag_rows, new_weights = [], {}

    for name in COMPONENT_NAMES:
        a = aggs.get(name)
        if not a or a["n_true"] < min_samples or a["n_false"] < min_samples:
            new_weights[name] = DEFAULT_WEIGHTS[name]
            diag_rows.append({"Компонент": name, "Случаев": a["n_true"] if a else 0,
                               "Средний_R(есть)": None, "Средний_R(нет)": None, "Лифт": None,
                               "Вес": f"{DEFAULT_WEIGHTS[name]} (данных мало, оставлен дефолт)"})
            continue

        mean_true = a["sum_true"] / a["n_true"]
        mean_false = a["sum_false"] / a["n_false"]
        lift = mean_true - mean_false
        diag_rows.append({"Компонент": name, "Случаев": a["n_true"],
                           "Средний_R(есть)": round(mean_true, 3),
                           "Средний_R(нет)": round(mean_false, 3),
                           "Лифт": round(lift, 3), "Вес": None})
        new_weights[name] = lift

    raw_positive = [w for name, w in new_weights.items()
                    if isinstance(w, float) and w > 0 and "оставлен дефолт" not in
                    str(next((r["Вес"] for r in diag_rows if r["Компонент"] == name), ""))]
    scale = (1.5 / max(raw_positive)) if raw_positive else 1.0

    final_weights = {}
    for row in diag_rows:
        name = row["Компонент"]
        if row["Вес"] is not None:
            final_weights[name] = DEFAULT_WEIGHTS[name]
        else:
            w = max(row["Лифт"], 0) * scale
            final_weights[name] = round(w, 2)
            row["Вес"] = round(w, 2)

    return final_weights, pd.DataFrame(diag_rows)


def fit_weights_from_backtest(trades, min_samples=40):
    """
    Для каждого компонента считает "лифт": средний R на сделках, где этот
    компонент сработал, минус средний R на сделках, где не сработал. Это
    простое, прозрачное, легко объяснимое сравнение (не многофакторная
    регрессия — с 10 частично пересекающимися бинарными компонентами и
    неизвестным числом сделок регрессия рискует переобучиться и дать
    красивые, но случайные коэффициенты; разница средних честнее и её
    можно руками проверить).

    Компоненты с недостаточным числом случаев (< min_samples) не трогаем —
    оставляем дефолтный вес, потому что "нет данных" не то же самое, что
    "доказанно не работает".

    Возвращает (новые_веса, диагностическую_таблицу).
    """
    if trades is None or trades.empty:
        return dict(DEFAULT_WEIGHTS), None
    return fit_weights_from_aggregates(compute_component_aggregates(trades), min_samples)


def score_for_fixed_direction(last, components, direction, weights):
    """
    Для сравнения ДО/ПОСЛЕ в бэктесте: направление сделки уже зафиксировано
    на момент симуляции (см. backtest_ticker), пересчитывать его заново под
    новые веса нельзя — тогда бы разошлись направление и уже случившийся
    исход. Тут просто считаем, насколько компоненты (с данными весами)
    поддерживают ИМЕННО ЭТО, уже случившееся направление — то же самое
    взвешенное голосование, но без права передумать по направлению.
    """
    net = sum(weights.get(name, 0.0) * val for name, val in components.items())
    aligned = net if direction == "long" else -net
    adx = last.get("ADX14", 0)
    mult = 1.3 if adx > 25 else (0.6 if adx < 20 else 1.0)
    return round(aligned * mult, 2)


def bucket_summary_for(trades, score_col):
    bins = [-100, 3, 5, 7, 100]
    labels = ["<3 (слабый)", "3-5", "5-7", "7+ (сильный)"]
    t = trades.copy()
    t["bucket"] = pd.cut(t[score_col], bins=bins, labels=labels)
    return t.groupby("bucket", observed=True).agg(
        Сделок=("r_multiple", "count"),
        Винрейт=("outcome", lambda x: round((x == "target").mean() * 100, 1)),
        Средний_R=("r_multiple", lambda x: round(x.mean(), 3)),
        Сумма_R=("r_multiple", lambda x: round(x.sum(), 1)),
    ).reset_index()


def run_backtest(data_by_ticker, config):
    tickers = list(data_by_ticker.keys())[:config["backtest_max_tickers"]]
    all_trades = []
    for t in tqdm(tickers, desc="   Бэктест по тикерам"):
        try:
            tr = backtest_ticker(data_by_ticker[t], config)
            if not tr.empty:
                tr["ticker"] = t
                all_trades.append(tr)
        except Exception:
            continue

    if not all_trades:
        return None, None, None, None

    trades = pd.concat(all_trades, ignore_index=True)

    this_run_aggs = compute_component_aggregates(trades)
    accumulated = None
    if config.get("persist_calibration_across_runs", True):
        cached = load_accumulated_calibration(config["cache_dir"])
        accumulated = merge_aggregates(cached, this_run_aggs) if cached else this_run_aggs
        save_accumulated_calibration(config["cache_dir"], accumulated)
    else:
        accumulated = this_run_aggs

    fitted_weights, diagnostics = fit_weights_from_aggregates(
        accumulated, min_samples=config["backtest_min_samples_per_component"]
    )
    if config.get("persist_calibration_across_runs", True) and diagnostics is not None:
        total_now = sum(a["n_true"] + a["n_false"] for a in this_run_aggs.values()) // max(len(this_run_aggs), 1)
        total_acc = sum(a["n_true"] + a["n_false"] for a in accumulated.values()) // max(len(accumulated), 1)
        print(f"   Накоплено данных калибровки: ~{total_acc} сделок за все прогоны в этой сессии "
              f"(из них ~{total_now} за этот прогон) — веса стабильнее с каждым повторным запуском.")

    def bucket_summary(score_col):
        return bucket_summary_for(trades, score_col)

    trades["score_default"] = trades.apply(
        lambda row: score_for_fixed_direction(row, {c: row[c] for c in COMPONENT_NAMES},
                                               row["direction"], DEFAULT_WEIGHTS),
        axis=1
    )
    trades["score_fitted"] = trades.apply(
        lambda row: score_for_fixed_direction(row, {c: row[c] for c in COMPONENT_NAMES},
                                               row["direction"], fitted_weights),
        axis=1
    )

    summary_default = bucket_summary("score_default")
    summary_fitted = bucket_summary("score_fitted")

    return trades, summary_default, summary_fitted, {"weights": fitted_weights, "diagnostics": diagnostics}


# =============================================================================
# ГЛАВНЫЙ PIPELINE
# =============================================================================
def run_screener(config=CONFIG):
    print("1/6: Быстрый префильтр по ликвидности + текущий стакан (bid/offer)...")
    snapshot = get_market_snapshot()
    top_n = config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(snapshot, top_n)
    quotes = snapshot.set_index("SECID")[["BID", "OFFER"]].to_dict("index")
    print(f"   Беру в работу {len(tickers)} тикеров")

    print("2/6: Скачиваю часовые свечи параллельно (без ещё не закрытого последнего бара)...")
    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )

    print("2b/6: Скачиваю индекс IMOEX для расчёта относительной силы...")
    try:
        index_df = get_index_candles("IMOEX", config["history_days"], config["candle_interval"])
    except Exception:
        index_df = None
        print("   Не удалось получить IMOEX — относительная сила будет пропущена")

    data_by_ticker, daily_context_by_ticker = {}, {}
    skipped_low_vol, failed_indicator_tickers = 0, []
    for t, raw in raw_by_ticker.items():
        if len(raw) < 210:
            continue
        try:
            df_ind = add_indicators(raw)
            median_atr_pct = df_ind["ATR_pct"].tail(200).median()
            if median_atr_pct < config["min_atr_pct"]:
                skipped_low_vol += 1
                continue
            df_ind = compute_relative_strength(df_ind, index_df, config["rel_strength_window"])
            df_ind = compute_index_trend(df_ind, index_df)
            daily_context_by_ticker[t] = add_daily_context(raw)
            data_by_ticker[t] = df_ind
        except Exception as e:
            failed_indicator_tickers.append((t, str(e)))

    if failed_indicator_tickers:
        print(f"   Не удалось посчитать индикаторы для {len(failed_indicator_tickers)} тикеров "
              f"(напр. {failed_indicator_tickers[0][0]}: {failed_indicator_tickers[0][1][:80]}) — пропущены")

    print(f"   Хватает истории для анализа: {len(data_by_ticker)} тикеров "
          f"(отсеяно как низковолатильные фонды: {skipped_low_vol})")

    print("3/6: Обучаю ML-модель на объединённой истории всех тикеров...")
    model, auc = train_and_predict(
        data_by_ticker, config["ml_horizon_bars"], config["ml_up_threshold_atr"]
    )
    if auc is not None:
        print(f"   ROC-AUC на валидации (out-of-sample): {auc:.3f}")
        print("   (0.5 = случайность; веса ML в score уже приглушены пропорционально этому числу)")
    else:
        print("   Недостаточно данных для ML — работаем на техническом score.")

    print("4/6: Бэктест на истории + калибровка весов score по факту (не на глаз)...")
    fitted_weights, weight_diagnostics = dict(DEFAULT_WEIGHTS), None
    backtest_trades, backtest_summary_default, backtest_summary_fitted = None, None, None
    if config["backtest_enabled"]:
        print(f"   ~{config['backtest_max_tickers']} тикеров, обычно 1-3 минуты...")
        backtest_trades, backtest_summary_default, backtest_summary_fitted, fit_result = run_backtest(data_by_ticker, config)
        if fit_result is not None:
            fitted_weights = fit_result["weights"]
            weight_diagnostics = fit_result["diagnostics"]
            print("   Веса пересчитаны по факту истории — используются в отчёте ниже.")

            # Лог: сколько сделок какого направления в бэктесте, и краткая
            # сводка по path_label — как именно вели себя цены в проигранных
            # и выигранных сделках. Это прямой ответ на вопрос "сразу пошло
            # против, колебалось, или шло по сценарию, но не дошло".
            if backtest_trades is not None and not backtest_trades.empty and "path_label" in backtest_trades.columns:
                print("\n   Как вели себя цены в сделках бэктеста (path_label):")
                path_counts = backtest_trades["path_label"].value_counts()
                path_pct = (path_counts / len(backtest_trades) * 100).round(1)
                for label, cnt in path_counts.items():
                    print(f"     {label}: {cnt} ({path_pct[label]}%)")
                mae_mean = backtest_trades["mae_r"].mean()
                mfe_mean = backtest_trades["mfe_r"].mean()
                print(f"   Средняя максимальная просадка против позиции (MAE): {mae_mean:.2f}R")
                print(f"   Средний максимальный ход в пользу позиции (MFE): {mfe_mean:.2f}R")
                print(f"   (Стоп стоит на 1.0R по построению — если MAE у выигрышных сделок")
                print(f"   часто подбирается близко к 1.0R, стоп в среднем расположен разумно;")
                print(f"   смотрите таблицу чувствительности к множителю стопа ниже для точного ответа.)")
        else:
            print("   Недостаточно сделок для калибровки — использую веса по умолчанию.")

    stop_sensitivity_df = None
    if config["stop_sensitivity_enabled"] and len(data_by_ticker) > 0:
        print(f"\n4b/6: Проверяю, не слишком ли тесен стоп 1.5×ATR — сравниваю разные множители "
              f"на {config['stop_sensitivity_tickers']} тикерах (та же история, тот же вход)...")
        stop_sensitivity_df = run_stop_sensitivity_analysis(data_by_ticker, config)
        if stop_sensitivity_df is not None and not stop_sensitivity_df.empty:
            print("\n" + "=" * 100)
            print("ЧУВСТВИТЕЛЬНОСТЬ К МНОЖИТЕЛЮ СТОПА (тот же вход/направление, разная ширина стопа)")
            print("=" * 100)
            print(stop_sensitivity_df.to_string(index=False))
            print("\nЕсли винрейт/средний R растут по мере расширения стопа — текущий 1.5x слишком тесен")
            print("и режет будущих победителей раньше времени (см. предыдущее сообщение про MAE/MFE).")
            print("Если после какой-то точки рост останавливается или разворачивается — там и есть")
            print("разумный ориентир для atr_stop_mult, а не 1.5 наугад.")

    print("5/6: Считаю свечные паттерны, графику, Фибо, волны, дневной контекст, риск-уровни и score...")
    results = []
    failed_result_tickers = []
    for t, df in data_by_ticker.items():
        try:
            candle_patterns = detect_candle_patterns(df)
            chart_patterns = detect_chart_patterns(df, config["pattern_lookback_bars"], config["swing_order"])
            divergences = detect_divergence(df, config["pattern_lookback_bars"], config["swing_order"])
            _, fib_patterns = compute_fibonacci_signal(df, config["pattern_lookback_bars"], config["swing_order"])
            wolfe_patterns = detect_wolfe_wave(df, config["pattern_lookback_bars"], config["swing_order"])

            daily_ctx = daily_context_by_ticker.get(t) or {}
            daily_patterns = daily_ctx.get("patterns", [])

            patterns = (candle_patterns + chart_patterns + divergences +
                        fib_patterns + wolfe_patterns + daily_patterns)
            last = df.iloc[-1]

            ml_prob = None
            if model is not None and not last[FEATURES].isna().any():
                ml_prob = float(model.predict_proba(last[FEATURES].values.reshape(1, -1))[0, 1])

            daily_trend_up = daily_ctx.get("trend_up")
            score, direction = score_ticker(last, patterns, ml_prob, ml_auc=auc,
                                             daily_trend_up=daily_trend_up, weights=fitted_weights)

            q = quotes.get(t, {})
            bid, offer = q.get("BID"), q.get("OFFER")
            risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"],
                                        bid=bid, offer=offer)
        except Exception as e:
            failed_result_tickers.append((t, str(e)))
            continue

        daily_trend_label = ("Совпадает" if daily_trend_up is not None and
                              ((daily_trend_up and direction == "long") or (not daily_trend_up and direction == "short"))
                              else ("Против" if daily_trend_up is not None else "Н/Д"))

        idx_trend_val = last.get("index_trend_up")
        if pd.isna(idx_trend_val) if idx_trend_val is not None else True:
            index_trend_label = "Н/Д"
        else:
            idx_up = idx_trend_val > 0
            index_trend_label = ("Совпадает" if (idx_up and direction == "long") or (not idx_up and direction == "short")
                                  else "ПРОТИВ РЫНКА")

        results.append({
            "Тикер": t,
            "Цена": smart_round(last["Close"]),
            "Направление": "ЛОНГ" if direction == "long" else "ШОРТ",
            "Score": score,
            "ML_проб_роста": round(ml_prob, 3) if ml_prob is not None else None,
            "RSI14": round(last["RSI14"], 1),
            "ADX14": round(last["ADX14"], 1),
            "Тренд IMOEX": index_trend_label,
            "Дневной тренд": daily_trend_label,
            "Дневной RSI": daily_ctx.get("rsi"),
            "Дневной ADX": daily_ctx.get("adx"),
            "Отн.сила к IMOEX %": round(last["rel_strength_pct"], 2) if pd.notna(last.get("rel_strength_pct")) else None,
            "Паттерны": format_patterns_for_display(patterns, direction),
            "Вход": risk["entry"],
            "Стоп-лосс": risk["stop_loss"],
            "Тейк-профит": risk["take_profit"],
            "Risk/Reward": risk["risk_reward"],
            "Спред %": risk["spread_pct"],
            "Риск/спред x": risk["risk_per_spread"],
            "⚠ Спред": risk["spread_warning"] or "-",
        })

    report = pd.DataFrame(results).sort_values("Score", ascending=False).reset_index(drop=True)

    if failed_result_tickers:
        print(f"   Пропущено на этапе паттернов/score: {len(failed_result_tickers)} тикеров "
              f"(напр. {failed_result_tickers[0][0]}: {failed_result_tickers[0][1][:80]})")

    if not report.empty:
        n_long = (report["Направление"] == "ЛОНГ").sum()
        n_short = (report["Направление"] == "ШОРТ").sum()
        total = n_long + n_short
        long_pct = n_long / total * 100 if total else 0
        print(f"\n   Баланс направлений по всем {total} тикерам: ЛОНГ {n_long} ({long_pct:.0f}%) / ШОРТ {n_short} ({100-long_pct:.0f}%)")
        if long_pct > 80 or long_pct < 20:
            print("   ⚠ Сильный перекос в одну сторону — это может быть реальная фаза рынка (весь")
            print("   рынок растёт/падает), а может быть структурный перекос в логике скринера.")
            print("   Проверьте IMOEX за этот период: если он тоже сильно трендовый — это ожидаемо.")

        if "Тренд IMOEX" in report.columns:
            against_market = (report["Тренд IMOEX"] == "ПРОТИВ РЫНКА").sum()
            known = report["Тренд IMOEX"].isin(["Совпадает", "ПРОТИВ РЫНКА"]).sum()
            if known > 0:
                print(f"   Рекомендаций ПРОТИВ тренда самого IMOEX: {against_market} из {known} "
                      f"({against_market/known*100:.0f}%) — раньше эта проверка отсутствовала в системе.")

    if weight_diagnostics is not None:
        print("\n" + "=" * 100)
        print("КАКИЕ КОМПОНЕНТЫ SCORE РЕАЛЬНО ПОКАЗАЛИ ЭДЖ НА ИСТОРИИ (а не просто мои изначальные веса)")
        print("=" * 100)
        print(weight_diagnostics.to_string(index=False))
        print("\n'Лифт' = средний R на сделках с этим сигналом минус средний R без него. Положительный")
        print("и заметный лифт = компонент статистически что-то ловит. Около нуля/отрицательный = веса")
        print("занижены/обнулены — компонент не показал доказанного эффекта на этой выборке.")
        print("\n'fib_level' и 'wolfe_wave' — более спекулятивные техники, добавлены в этой версии.")
        print("'daily_pattern' в бэктесте калиброваться не может (см. комментарий в add_daily_context")
        print("в коде) и всегда остаётся на весе по умолчанию — это осознанное ограничение, не баг.")

    if backtest_summary_default is not None and backtest_summary_fitted is not None:
        print("\n" + "=" * 100)
        print("ДО/ПОСЛЕ: score по старым весам (на глаз) vs по откалиброванным весам")
        print("=" * 100)
        print("-- ДО (веса по умолчанию) --")
        print(backtest_summary_default.to_string(index=False))
        print("-- ПОСЛЕ (веса по факту бэктеста) --")
        print(backtest_summary_fitted.to_string(index=False))
        print("\nЕсли справа рост Среднего_R от 'слабого' к '7+' стал более выраженным, чем слева —")
        print("калибровка реально помогла отличать хорошие сигналы от плохих. Если нет — значит на")
        print("этом наборе тикеров и периоде технический score в принципе слабо различает исходы,")
        print("и это стоит знать, а не закрывать на это глаза.")

    print("\n6/6: Готово.\n")
    print("=" * 100)
    print(f"ТОП-{config['top_n_report']} ИДЕЙ ПО СКРИНЕРУ (не финансовая рекомендация!)")
    print("=" * 100)
    with pd.option_context("display.max_columns", None, "display.width", 220):
        print(report.head(config["top_n_report"]).to_string(index=False))

    return report


# =============================================================================
# 9. WALK-FORWARD ВАЛИДАЦИЯ: "что бы он сказал тогда — и что случилось по факту"
# =============================================================================
def _generate_asof_report(visible_by_ticker, index_visible, weights, model, auc, config):
    """
    Строит отчёт-рекомендацию ТОЧНО той же логикой, что и живой прогон, но
    используя только данные "видимые" на дату среза (visible_by_ticker) —
    без единого бита информации из будущего относительно этой даты.
    Возвращает DataFrame того же вида, что топ-таблица run_screener.
    """
    results = []
    for t, raw in visible_by_ticker.items():
        if len(raw) < 210:
            continue
        try:
            df_ind = add_indicators(raw)
            if df_ind["ATR_pct"].tail(200).median() < config["min_atr_pct"]:
                continue
            df_ind = compute_relative_strength(df_ind, index_visible, config["rel_strength_window"])
            df_ind = compute_index_trend(df_ind, index_visible)
            candle_patterns = detect_candle_patterns(df_ind)
            chart_patterns = detect_chart_patterns(df_ind, config["pattern_lookback_bars"], config["swing_order"])
            divergences = detect_divergence(df_ind, config["pattern_lookback_bars"], config["swing_order"])
            _, fib_patterns = compute_fibonacci_signal(df_ind, config["pattern_lookback_bars"], config["swing_order"])
            wolfe_patterns = detect_wolfe_wave(df_ind, config["pattern_lookback_bars"], config["swing_order"])
            daily_ctx = add_daily_context(raw) or {}  # raw уже обрезан по дату среза — дневной контекст тоже честный
            daily_patterns = daily_ctx.get("patterns", [])

            patterns = candle_patterns + chart_patterns + divergences + fib_patterns + wolfe_patterns + daily_patterns
            last = df_ind.iloc[-1]

            ml_prob = None
            if model is not None and not last[FEATURES].isna().any():
                ml_prob = float(model.predict_proba(last[FEATURES].values.reshape(1, -1))[0, 1])

            score, direction = score_ticker(last, patterns, ml_prob, ml_auc=auc,
                                             daily_trend_up=daily_ctx.get("trend_up"), weights=weights)
            # Исторического стакана (bid/offer) для прошлых дат не существует в бесплатном
            # MOEX ISS — используем close как вход, это честное ограничение теста,
            # а не то же самое, что реалистичный вход в живом прогоне.
            risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"])

            results.append({
                "Тикер": t, "direction": direction, "Score": score,
                "Вход": risk["entry"], "Стоп-лосс": risk["stop_loss"], "Тейк-профит": risk["take_profit"],
            })
        except Exception:
            continue

    return pd.DataFrame(results).sort_values("Score", ascending=False).reset_index(drop=True)


def run_walkforward_test(config=CONFIG):
    """
    Главная идея: берём тикеры, качаем ПОЛНУЮ историю (она у нас всё равно
    есть — MOEX отдаёт её всю), и на нескольких равномерно распределённых
    исторических датах ПРИТВОРЯЕМСЯ, что "сегодня" — эта дата: обрезаем все
    данные по ней, прогоняем через ровно тот же пайплайн (индикаторы, ML,
    калибровку весов, паттерны, score, риск-уровни), получаем рекомендации —
    а дальше смотрим в уже скачанные данные ПОСЛЕ этой даты и честно
    проверяем, сработал стоп или тейк.

    Несколько дат, а не одна — потому что один исторический день это один
    рыночный режим (могло просто повезти/не повезти с общим движением рынка
    в тот момент), и делать вывод об инструменте по одной дате неверно.
    """
    print("Walk-forward валидация: 'что бы скринер сказал тогда — и что было по факту'\n")
    print("1/4: Тикеры + полная история (включая период ПОСЛЕ тестовых дат — для честной проверки)...")
    snapshot = get_market_snapshot()
    top_n = config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(snapshot, top_n)
    print(f"   Внимание: список тикеров основан на ТЕКУЩЕЙ ликвидности, не на ликвидности на")
    print(f"   тестовую дату — на практике почти всегда одно и то же, но это стоит знать.")

    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )
    try:
        index_raw = get_index_candles("IMOEX", config["history_days"], config["candle_interval"])
    except Exception:
        index_raw = None

    all_dates = pd.concat([df["Date"] for df in raw_by_ticker.values()], ignore_index=True)
    min_date, max_date = all_dates.min(), all_dates.max()

    # тестовая дата должна иметь ~210+ баров ДО себя (на индикаторы/калибровку)
    # и walkforward_eval_max_bars часов ПОСЛЕ себя (чтобы было что проверять)
    history_buffer = pd.Timedelta(days=45)
    eval_buffer = pd.Timedelta(hours=config["walkforward_eval_max_bars"] * 2)  # с запасом на выходные/праздники
    window_start = min_date + history_buffer
    window_end = max_date - eval_buffer

    if window_start >= window_end:
        print("   Недостаточно истории для walk-forward теста с текущими настройками "
              "(увеличьте history_days или уменьшите walkforward_eval_max_bars).")
        return None

    n = config["walkforward_n_dates"]
    test_dates = pd.date_range(window_start, window_end, periods=n) if n > 1 else pd.DatetimeIndex([window_end])
    print(f"   Диапазон истории: {min_date.date()} — {max_date.date()}")
    print(f"   Тестовые даты ({n}): {', '.join(d.strftime('%Y-%m-%d %H:%M') for d in test_dates)}\n")

    all_eval_rows = []
    for di, as_of in enumerate(test_dates, 1):
        print(f"2/4: [{di}/{n}] Дата среза: {as_of.strftime('%Y-%m-%d %H:%M')}...")
        visible_by_ticker = {t: df[df["Date"] <= as_of].reset_index(drop=True) for t, df in raw_by_ticker.items()}
        future_by_ticker = {t: df[df["Date"] > as_of].reset_index(drop=True) for t, df in raw_by_ticker.items()}
        index_visible = index_raw[index_raw["Date"] <= as_of].reset_index(drop=True) if index_raw is not None else None

        data_by_ticker = {}
        for t, raw in visible_by_ticker.items():
            if len(raw) < 210:
                continue
            try:
                df_ind = add_indicators(raw)
                if df_ind["ATR_pct"].tail(200).median() < config["min_atr_pct"]:
                    continue
                df_ind = compute_relative_strength(df_ind, index_visible, config["rel_strength_window"])
                df_ind = compute_index_trend(df_ind, index_visible)
                data_by_ticker[t] = df_ind
            except Exception:
                continue

        if len(data_by_ticker) < 10:
            print(f"   Мало тикеров с достаточной историей на эту дату ({len(data_by_ticker)}), пропускаю.")
            continue

        model, auc = train_and_predict(data_by_ticker, config["ml_horizon_bars"], config["ml_up_threshold_atr"])

        wf_backtest_config = dict(config)
        wf_backtest_config["backtest_max_tickers"] = config["walkforward_calibration_tickers"]
        wf_backtest_config["persist_calibration_across_runs"] = False  # каждая дата калибруется независимо
        _, _, _, fit_result = run_backtest(data_by_ticker, wf_backtest_config)
        weights = fit_result["weights"] if fit_result else dict(DEFAULT_WEIGHTS)

        report = _generate_asof_report(visible_by_ticker, index_visible, weights, model, auc, config)
        if report.empty:
            print("   Не удалось построить рекомендации на эту дату, пропускаю.")
            continue

        top = report.head(config["walkforward_top_n"])
        print(f"   Рекомендаций на эту дату: {len(top)}. Проверяю по факту истории...")

        for _, row in top.iterrows():
            t = row["Тикер"]
            future = future_by_ticker.get(t)
            if future is None or future.empty:
                continue
            outcome, exit_price = simulate_trade_outcome(
                future, row["direction"], row["Стоп-лосс"], row["Тейк-профит"], config["walkforward_eval_max_bars"]
            )
            r = r_multiple(row["direction"], row["Вход"], row["Стоп-лосс"], exit_price)
            if r is None:
                continue
            all_eval_rows.append({
                "Дата среза": as_of.strftime("%Y-%m-%d %H:%M"),
                "Тикер": t,
                "Направление": "ЛОНГ" if row["direction"] == "long" else "ШОРТ",
                "Score": row["Score"],
                "Вход": row["Вход"], "Стоп": row["Стоп-лосс"], "Тейк": row["Тейк-профит"],
                "Исход": {"target": "✅ Тейк", "stop": "❌ Стоп", "timeout": "⏱ Не дошёл ни туда ни сюда"}.get(outcome, outcome),
                "R": round(r, 2),
            })

    print("\n3/4: Готово.\n")
    if not all_eval_rows:
        print("Не набралось ни одной проверяемой рекомендации — попробуйте увеличить history_days")
        print("или уменьшить walkforward_eval_max_bars.")
        return None

    eval_df = pd.DataFrame(all_eval_rows)

    print("=" * 100)
    print(f"WALK-FORWARD РЕЗУЛЬТАТ: {len(eval_df)} рекомендаций на {len(test_dates)} исторических датах")
    print("=" * 100)
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(eval_df.to_string(index=False))

    print("\n4/4: Сводка")
    print("=" * 100)
    win_rate = (eval_df["Исход"] == "✅ Тейк").mean() * 100
    avg_r = eval_df["R"].mean()
    sum_r = eval_df["R"].sum()
    print(f"Всего рекомендаций проверено: {len(eval_df)}")
    print(f"Винрейт (дошло до тейка): {win_rate:.1f}%")
    print(f"Средний R: {avg_r:.3f}   Суммарный R: {sum_r:.1f}")

    by_date = eval_df.groupby("Дата среза").agg(
        Рекомендаций=("R", "count"),
        Винрейт=("Исход", lambda x: round((x == "✅ Тейк").mean() * 100, 1)),
        Средний_R=("R", lambda x: round(x.mean(), 3)),
    ).reset_index()
    print("\nПо каждой тестовой дате отдельно (чтобы видеть, не тянет ли результат одна удачная дата):")
    print(by_date.to_string(index=False))

    print(f"\n⚠ Это {len(test_dates)} исторических даты, не {len(test_dates)*100} независимых экспериментов —")
    print("рекомендации внутри одной даты сильно коррелируют между собой (один и тот же день на рынке).")
    print("Реальное число независимых 'проверок режима' — это число дат, а не число тикеров.")
    print(f"С {len(test_dates)} датами доверительный интервал для винрейта/R огромный. Это ориентир,")
    print("а не статистическое доказательство. Чтобы получить осмысленную уверенность, нужны")
    print("десятки тестовых дат, разнесённых на месяцы/годы, в идеале через разные рыночные режимы")
    print("(рост, падение, боковик) — увеличьте walkforward_n_dates и history_days, если готовы ждать дольше.")

    return eval_df


if __name__ == "__main__":
    # full_report = run_screener(CONFIG)
    # full_report.to_csv("moex_screener_report.csv", index=False, encoding="utf-8-sig")

    # Чтобы проверить, насколько вообще можно доверять рекомендациям скринера,
    # раскомментируйте строку ниже — прогонит walk-forward тест на нескольких
    # исторических датах и честно сверит прогнозы с тем, что случилось по факту:
    walkforward_results = run_walkforward_test(CONFIG)

"""
ЧТО МОЖНО ДОБАВИТЬ ДАЛЬШЕ:
  - Ichimoku Cloud (есть в pandas-ta-classic: ta_input.ta.ichimoku()) —
    не включил по умолчанию, у него сложный многослойный вывод, стоит
    добавлять отдельно и тестировать на ваших тикерах.
  - Уведомления в Telegram, чтобы не заходить в Colab руками.
  - Журнал сделок в реальном времени: то же, что walk-forward, но для
    будущих сигналов, а не исторических — копить рекомендации по мере
    появления и сверять с фактом позже, без "подглядывания", т.к. будущего
    ещё не существует на момент рекомендации.
"""

Walk-forward валидация: 'что бы скринер сказал тогда — и что было по факту'

1/4: Тикеры + полная история (включая период ПОСЛЕ тестовых дат — для честной проверки)...
   Внимание: список тикеров основан на ТЕКУЩЕЙ ликвидности, не на ликвидности на
   тестовую дату — на практике почти всегда одно и то же, но это стоит знать.


   Качаю свечи параллельно:   0%|          | 0/120 [00:00<?, ?it/s]

   Диапазон истории: 2025-12-14 — 2026-08-11
   Тестовые даты (4): 2026-01-28 09:00, 2026-04-01 16:20, 2026-06-03 23:40, 2026-08-06 07:00

2/4: [1/4] Дата среза: 2026-01-28 09:00...


   Бэктест по тикерам:   0%|          | 0/15 [00:00<?, ?it/s]

   Рекомендаций на эту дату: 10. Проверяю по факту истории...
2/4: [2/4] Дата среза: 2026-04-01 16:20...


...

   Бэктест по тикерам:   0%|          | 0/15 [00:00<?, ?it/s]

   Рекомендаций на эту дату: 10. Проверяю по факту истории...
2/4: [4/4] Дата среза: 2026-08-06 07:00...


   Бэктест по тикерам:   0%|          | 0/15 [00:00<?, ?it/s]

   Рекомендаций на эту дату: 10. Проверяю по факту истории...

3/4: Готово.

WALK-FORWARD РЕЗУЛЬТАТ: 40 рекомендаций на 4 исторических датах
      Дата среза Тикер Направление  Score         Вход         Стоп         Тейк  Исход     R
2026-01-28 09:00  RAGR        ЛОНГ   5.09   124.680000   123.460000   127.110000 ❌ Стоп -1.00
2026-01-28 09:00  MTLR        ЛОНГ   5.06    75.010000    74.430000    76.170000 ❌ Стоп -1.00
2026-01-28 09:00  BRZL        ЛОНГ   5.06  1876.000000  1829.140000  1969.720000 ✅ Тейк  2.00
2026-01-28 09:00  TGKA        ЛОНГ   4.95     0.007146     0.007058     0.007321 ❌ Стоп -1.00
2026-01-28 09:00  TGKN        ЛОНГ   4.95     0.007360     0.007035     0.008010 ✅ Тейк  2.00
2026-01-28 09:00  SVAV        ЛОНГ   4.90   588.000000   581.320000   601.370000 ❌ Стоп -1.00
2026-01-28 09:00  EUTR        ЛОНГ   4.88   148.350000   147.280000   150.500000 ❌ Стоп -1.00
2026-01-28 09:00  OZPH        ЛОНГ   4.52    53.810000    53.340000    54.750000 ❌ Стоп -1.00
2026-01-28 09

'\nЧТО МОЖНО ДОБАВИТЬ ДАЛЬШЕ:\n  - Ichimoku Cloud (есть в pandas-ta-classic: ta_input.ta.ichimoku()) —\n    не включил по умолчанию, у него сложный многослойный вывод, стоит\n    добавлять отдельно и тестировать на ваших тикерах.\n  - Уведомления в Telegram, чтобы не заходить в Colab руками.\n  - Журнал сделок в реальном времени: то же, что walk-forward, но для\n    будущих сигналов, а не исторических — копить рекомендации по мере\n    появления и сверять с фактом позже, без "подглядывания", т.к. будущего\n    ещё не существует на момент рекомендации.\n'

In [ ]:
# -*- coding: utf-8 -*-
"""
MOEX Screener — ЧИСТЫЙ ТЕСТ (только "статистические уровни + волны + симметрии")
====================================================================================
Это ОТДЕЛЬНЫЙ эксперимент, не замена основному скринеру. Идея: в интернете
встретилось категоричное утверждение, что в техническом анализе реально
работают только 3 вещи — статистические уровни, статистические волны (волны
Вульфа как их часть) и симметрии (Г-и-П, двойные вершины/дна) — а всё
остальное (Фибоначчи, треугольники, скользящие средние, пробои хаёв/лоёв)
"математически опровергнуто". Само утверждение дано БЕЗ единой цифры или
методологии — то есть имеет ровно тот же статус, что и любое другое
голословное заявление. Проверяем его тем же способом, каким проверяли все
свои собственные идеи: считаем и смотрим на бэктест, а не верим на слово.

Что внутри (и чем это отличается от основного скринера):
  1. "Статистические уровни" реализованы как ОБЪЁМНЫЙ ПРОФИЛЬ (Volume
     Profile) — зоны цены, где исторически прошло больше всего объёма
     (Point of Control и зоны высокого объёма). Это честно статистическая
     вещь: распределение реального объёма по цене, а не визуально
     проведённая линия.
  2. "Статистические волны" = волна Вульфа (5-точечная разворотная
     формация) — тот же код, что и в основном скринере, уже проверенный.
  3. "Симметрии" = голова-плечи (обычная и перевёрнутая) + двойная
     вершина/двойное дно. "Паттерн Батман" не имеет единого общепринятого
     строгого определения — не стал выдумывать своё и выдавать за
     авторитетное, оставил то, что однозначно описано.
  4. НЕТ: Фибоначчи, треугольников, скользящих средних/тренда, пробоев
     диапазона на объёме, RSI/MACD/Stochastic/Williams %R/VWAP, свечных
     паттернов, дивергенций, относительной силы, дневного контекста, ML —
     всё это либо прямо названо в претензии как "бред", либо не относится
     ни к одной из трёх заявленных категорий. Цель — чистый тест ИМЕННО
     этих трёх вещей, без постороннего шума, который мог бы смазать результат.

Структура пайплайна и методология проверки — ТА ЖЕ, что в основном
скринере: калибровка весов по бэктесту (лифт средний R "есть сигнал" минус
"нет сигнала"), MAE/MFE и классификация путей сделок, свип по множителю
стопа, walk-forward валидация на нескольких исторических датах.

ВАЖНО: то, что что-то встречается в статье/посте, не делает это правдой —
и то, что мы реализуем и тестируем эти 3 вещи, не означает, что они
покажут эдж. Единственный источник истины ниже — цифры из бэктеста.

Как запустить в Google Colab:
  1. Вставьте содержимое этого файла в отдельную ячейку и запустите
  2. Runtime → Change runtime type → Hardware accelerator → None
  3. В конце файла — вызовы run_screener(CONFIG) и (закомментированный)
     run_walkforward_test(CONFIG)
"""

import os
import re
import time
import pickle
import warnings
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.signal import argrelextrema

warnings.filterwarnings("ignore")


def _ensure(pkg_import_name, pip_name=None):
    try:
        return __import__(pkg_import_name)
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", pip_name or pkg_import_name])
        return __import__(pkg_import_name)


_ensure("tqdm")
from tqdm.auto import tqdm


# =============================================================================
# CONFIG
# =============================================================================
CONFIG = {
    "history_days": 240,
    "candle_interval": 60,
    "max_tickers": None,
    "liquidity_top_n": 120,
    "min_atr_pct": 0.15,
    "atr_stop_mult": 1.5,
    "risk_reward_target": 2.0,
    "top_n_report": 15,
    "max_workers": 10,
    "cache_dir": "/content/moex_cache_purist",
    "cache_ttl_hours": 6,
    "swing_order": 3,
    "pattern_lookback_bars": 90,
    "volume_profile_bins": 40,          # на сколько ценовых корзин делить диапазон для объёмного профиля
    "volume_profile_tolerance_atr": 0.35,  # допуск близости цены к уровню, в ATR
    "volume_profile_lookback_bars": 400,   # окно ИМЕННО для объёмного профиля — заметно длиннее, чем
                                            # у остальных паттернов (90 часов мало для "статистической значимости")
    "volume_profile_min_touches": 3,       # зона должна быть сформирована минимум N отдельными барами
    "backtest_enabled": True,
    "backtest_max_tickers": 20,
    "backtest_stride": 6,
    "backtest_max_holding_bars": 40,
    "backtest_min_samples_per_component": 40,
    "persist_calibration_across_runs": True,
    "walkforward_n_dates": 5,
    "walkforward_top_n": 10,
    "walkforward_eval_max_bars": 60,
    "walkforward_calibration_tickers": 15,
    "stop_sensitivity_enabled": True,
    "stop_sensitivity_multipliers": [1.0, 1.5, 2.0, 2.5, 3.0],
    "stop_sensitivity_tickers": 10,
    "stop_sensitivity_stride": 10,
}

ISS_BASE = "https://iss.moex.com/iss"


# =============================================================================
# ФОРМАТИРОВАНИЕ ЦЕН
# =============================================================================
def smart_round(x, sig_figs=4):
    if x is None or not np.isfinite(x):
        return x
    if x == 0:
        return 0.0
    if abs(x) >= 1:
        return round(x, 2)
    magnitude = int(np.floor(np.log10(abs(x))))
    decimals = min(max(-magnitude + (sig_figs - 1), 2), 8)
    return round(x, decimals)


# =============================================================================
# 1. ЗАГРУЗКА ДАННЫХ С MOEX ISS API (тот же код, что в основном скринере)
# =============================================================================
def get_market_snapshot():
    url = f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {
        "iss.meta": "off",
        "iss.only": "securities,marketdata",
        "securities.columns": "SECID,SHORTNAME",
        "marketdata.columns": "SECID,VALTODAY,BID,OFFER,LAST",
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    js = r.json()
    sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
    md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])
    for col in ["VALTODAY", "BID", "OFFER", "LAST"]:
        md[col] = pd.to_numeric(md[col], errors="coerce")
    md["VALTODAY"] = md["VALTODAY"].fillna(0)
    return sec.merge(md, on="SECID", how="left")


def get_liquid_tickers(snapshot, top_n):
    merged = snapshot.copy()
    if merged["VALTODAY"].sum() > 0:
        merged = merged.sort_values("VALTODAY", ascending=False)
    return merged["SECID"].head(top_n).tolist()


def _cache_path(cache_dir, key):
    return os.path.join(cache_dir, f"{key}.pkl")


def _load_from_cache(cache_dir, key, ttl_hours):
    path = _cache_path(cache_dir, key)
    if not os.path.exists(path):
        return None
    age_hours = (time.time() - os.path.getmtime(path)) / 3600
    if age_hours > ttl_hours:
        return None
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception:
        return None


def _save_to_cache(cache_dir, key, df):
    os.makedirs(cache_dir, exist_ok=True)
    with open(_cache_path(cache_dir, key), "wb") as f:
        pickle.dump(df, f)


def _fetch_candles_raw(url, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    all_rows, start, columns = [], 0, None
    while True:
        params = {
            "from": since.strftime("%Y-%m-%d"),
            "till": till.strftime("%Y-%m-%d"),
            "interval": interval,
            "start": start,
        }
        r = requests.get(url, params=params, timeout=20)
        if r.status_code != 200:
            break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None:
            columns = js.get("columns", [])
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 500:
            break
        start += len(rows)

    if not all_rows or columns is None:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.rename(columns={
        "open": "Open", "close": "Close", "high": "High",
        "low": "Low", "volume": "Volume", "begin": "Date", "end": "DateEnd"
    })
    df = df[["Date", "DateEnd", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date")
    df = df.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)

    if len(df) > 0 and df["DateEnd"].iloc[-1] > pd.Timestamp.now():
        df = df.iloc[:-1]

    return df.drop(columns=["DateEnd"]).reset_index(drop=True)


def get_hourly_candles(ticker, days_back, interval=60):
    url = (f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/"
           f"securities/{ticker}/candles.json")
    return _fetch_candles_raw(url, days_back, interval)


def download_all_candles(tickers, days_back, interval, cache_dir, ttl_hours, max_workers):
    results = {}
    to_download = []
    for t in tickers:
        cached = _load_from_cache(cache_dir, t, ttl_hours)
        if cached is not None and not cached.empty:
            results[t] = cached
        else:
            to_download.append(t)

    if results:
        print(f"   Из кэша сессии загружено сразу: {len(results)} тикеров")

    if to_download:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(get_hourly_candles, t, days_back, interval): t
                       for t in to_download}
            for fut in tqdm(as_completed(futures), total=len(futures),
                             desc="   Качаю свечи параллельно"):
                t = futures[fut]
                try:
                    df = fut.result()
                    if not df.empty:
                        results[t] = df
                        _save_to_cache(cache_dir, t, df)
                except Exception:
                    pass

    return results


# =============================================================================
# 2. МИНИМАЛЬНЫЕ ИНДИКАТОРЫ — только ATR (нужен для риск-менеджмента, это не
#    "скользящая средняя как сигнал", а просто мера волатильности для стопа)
# =============================================================================
def add_atr(df):
    df = df.copy()
    prev_close = df["Close"].shift(1)
    tr = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs()
    ], axis=1).max(axis=1)
    df["ATR14"] = tr.ewm(alpha=1/14, adjust=False).mean()
    df["ATR_pct"] = df["ATR14"] / df["Close"] * 100
    return df


# =============================================================================
# 3. СВИНГИ (нужны и для волны Вульфа, и для симметрий)
# =============================================================================
def find_swings(series, order):
    values = series.values
    highs_idx = argrelextrema(values, np.greater_equal, order=order)[0]
    lows_idx = argrelextrema(values, np.less_equal, order=order)[0]
    highs_idx = np.array(sorted(set(highs_idx.tolist())))
    lows_idx = np.array(sorted(set(lows_idx.tolist())))
    return highs_idx, lows_idx


# =============================================================================
# 4. "СТАТИСТИЧЕСКИЕ УРОВНИ" — объёмный профиль (Volume Profile), v2
# =============================================================================
def compute_volume_profile_signal(df, lookback, n_bins, tolerance_atr_mult, min_touches=3):
    """
    v2 — переработано по трём слабым местам, найденным после первого прогона:
      1. Окно теперь заметно длиннее (`lookback` для профиля — отдельный,
         больший параметр, чем окно паттернов) — 90 часов маловато для
         "статистически значимого" уровня.
      2. POC (самая объёмная зона) и второстепенные зоны высокого объёма
         (HVN) теперь РАЗНЫЕ сигналы — раньше считались одним и тем же,
         хотя теоретически POC надёжнее.
      3. Добавлен фильтр качества: зона считается значимой только если её
         сформировало НЕСКОЛЬКО отдельных баров (min_touches), а не один
         бар с случайно большим объёмом — это ближе к "тут цена подолгу
         билась" (настоящая борьба спроса/предложения), а не случайный
         объёмный выброс на одной свече.
    """
    window = df.tail(lookback)
    if len(window) < 30:
        return None, []

    price_min, price_max = window["Low"].min(), window["High"].max()
    if price_max <= price_min:
        return None, []

    bin_edges = np.linspace(price_min, price_max, n_bins + 1)
    bin_volumes = np.zeros(n_bins)
    bin_touches = np.zeros(n_bins)

    lows = window["Low"].values
    highs = window["High"].values
    vols = window["Volume"].values

    for lo, hi, vol in zip(lows, highs, vols):
        if vol <= 0:
            continue
        if hi <= lo:
            idx = int(np.clip((lo - price_min) / (price_max - price_min) * n_bins, 0, n_bins - 1))
            bin_volumes[idx] += vol
            bin_touches[idx] += 1
            continue
        lo_idx = int(np.clip(np.searchsorted(bin_edges, lo, side="right") - 1, 0, n_bins - 1))
        hi_idx = int(np.clip(np.searchsorted(bin_edges, hi, side="right") - 1, 0, n_bins - 1))
        if lo_idx == hi_idx:
            bin_volumes[lo_idx] += vol
            bin_touches[lo_idx] += 1
            continue
        total_range = hi - lo
        for b in range(lo_idx, hi_idx + 1):
            b_lo, b_hi = max(bin_edges[b], lo), min(bin_edges[b + 1], hi)
            frac = max(b_hi - b_lo, 0) / total_range
            bin_volumes[b] += vol * frac
            bin_touches[b] += frac  # частичное касание тоже частично считается

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # POC — зона с максимальным объёмом СРЕДИ достаточно "натоптанных" зон
    qualified = bin_touches >= min_touches
    if not qualified.any():
        return None, []
    qualified_volumes = np.where(qualified, bin_volumes, -1)
    poc_bin = int(np.argmax(qualified_volumes))
    poc_price = bin_centers[poc_bin]

    hvn_idx, _ = find_swings(pd.Series(np.where(qualified, bin_volumes, 0)), order=2)
    hvn_idx = [i for i in hvn_idx if i != poc_bin and qualified[i]]
    hvn_levels = list(bin_centers[hvn_idx])

    close_last = window["Close"].iloc[-1]
    atr_last = df["ATR14"].iloc[-1] if "ATR14" in df.columns and pd.notna(df["ATR14"].iloc[-1]) else close_last * 0.01
    tol = atr_last * tolerance_atr_mult
    recent = window["Close"].tail(6)
    approaching_from_below = recent.iloc[0] < close_last

    found = []

    if abs(close_last - poc_price) <= tol:
        tag = "медвежий" if approaching_from_below else "бычий"
        direction_word = "снизу" if approaching_from_below else "сверху"
        found.append(f"У POC (объём) ~{smart_round(poc_price)} (подход {direction_word}, {tag})")

    nearest_hvn, nearest_dist = None, None
    for lvl in hvn_levels:
        dist = abs(close_last - lvl)
        if nearest_dist is None or dist < nearest_dist:
            nearest_dist, nearest_hvn = dist, lvl
    if nearest_hvn is not None and nearest_dist <= tol:
        appr_below = recent.iloc[0] < nearest_hvn
        tag = "медвежий" if appr_below else "бычий"
        direction_word = "снизу" if appr_below else "сверху"
        found.append(f"У зоны объёма (HVN) ~{smart_round(nearest_hvn)} (подход {direction_word}, {tag})")

    return {"poc": poc_price, "hvn_levels": hvn_levels}, found


# =============================================================================
# 5. "СТАТИСТИЧЕСКИЕ ВОЛНЫ" — волна Вульфа (тот же код, что в основном скринере)
# =============================================================================
def detect_wolfe_wave(df, lookback, swing_order):
    """
    Упрощённая эвристика волны Вульфа (Wolfe Wave) — 5-точечная разворотная
    формация: точки 1-3-5 (одна сторона) против 2-4 (другая), где точка 5
    пробивает продолжение линии 1-3 ("ложный пробой" уровня), а точка 4
    остаётся внутри канала. Цель разворота — линия 1-4, продлённая вперёд
    (EPA, estimated price at arrival). Точка 5 — ТЕКУЩАЯ цена (ещё
    формируется), а не уже подтверждённый свинг — иначе сигнал всегда
    приходит на бар позже, чем нужно.
    """
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 40:
        return []

    high_idx, _ = find_swings(window["High"], swing_order)
    _, low_idx = find_swings(window["Low"], swing_order)
    edge = swing_order
    high_idx = high_idx[(high_idx >= edge) & (high_idx < len(window) - edge)]
    low_idx = low_idx[(low_idx >= edge) & (low_idx < len(window) - edge)]

    points = [(i, window["High"].iloc[i], "H") for i in high_idx] + \
             [(i, window["Low"].iloc[i], "L") for i in low_idx]
    points.sort(key=lambda p: p[0])
    if len(points) < 4:
        return []

    atr_last = df["ATR14"].iloc[-1] if "ATR14" in df.columns and pd.notna(df["ATR14"].iloc[-1]) else None
    min_move = (atr_last * 2.0) if atr_last else (window["Close"].std() * 0.5)

    filtered = [points[0]]
    for p in points[1:]:
        prev = filtered[-1]
        if p[2] == prev[2]:
            if (p[2] == "H" and p[1] > prev[1]) or (p[2] == "L" and p[1] < prev[1]):
                filtered[-1] = p
            continue
        if abs(p[1] - prev[1]) < min_move:
            continue
        filtered.append(p)
    points = filtered
    if len(points) < 4:
        return []

    found = []
    last4 = points[-4:]
    types4 = [p[2] for p in last4]
    p5_idx = len(window) - 1
    p5_price_low, p5_price_high = window["Low"].iloc[-1], window["High"].iloc[-1]

    if types4 == ["L", "H", "L", "H"]:
        p1, p2, p3, p4 = last4
        if p3[0] != p1[0] and p4[0] != p1[0] and p3[1] < p1[1] and p4[1] < p2[1]:
            slope13 = (p3[1] - p1[1]) / (p3[0] - p1[0])
            line13_at_5 = p1[1] + slope13 * (p5_idx - p1[0])
            if p5_price_low < line13_at_5:
                slope14 = (p4[1] - p1[1]) / (p4[0] - p1[0])
                epa = p1[1] + slope14 * (p5_idx - p1[0])
                if epa > p5_price_low:
                    found.append(f"Волна Вульфа (бычья, цель ~{smart_round(epa)})")

    if types4 == ["H", "L", "H", "L"]:
        p1, p2, p3, p4 = last4
        if p3[0] != p1[0] and p4[0] != p1[0] and p3[1] > p1[1] and p4[1] > p2[1]:
            slope13 = (p3[1] - p1[1]) / (p3[0] - p1[0])
            line13_at_5 = p1[1] + slope13 * (p5_idx - p1[0])
            if p5_price_high > line13_at_5:
                slope14 = (p4[1] - p1[1]) / (p4[0] - p1[0])
                epa = p1[1] + slope14 * (p5_idx - p1[0])
                if epa < p5_price_high:
                    found.append(f"Волна Вульфа (медвежья, цель ~{smart_round(epa)})")

    return found


# =============================================================================
# 5b. ЛИНИИ ТРЕНДА И ИХ ПРОБОЙ (добавлено по прямому запросу — важная
#     оговорка: исходный пост называет "пробои хаёв/лоёв" бредом, а линия
#     тренда — близкий родственник этой идеи, просто по диагонали, а не
#     горизонтали. Если этот компонент покажет эдж — это будет ещё одним
#     ударом по достоверности поста, а не подтверждением его правоты.
# =============================================================================
def compute_trendline_signal(df, lookback, swing_order):
    """
    Строит линию сопротивления через последние 3 свинг-хая и линию
    поддержки через последние 3 свинг-лоя (обычная линейная регрессия,
    как в основном скринере для треугольников). Сигнал — только СВЕЖИЙ
    пробой (цена только что перешла с одной стороны линии на другую),
    не "цена давно выше линии".
    """
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return []

    high_idx, _ = find_swings(window["High"], swing_order)
    _, low_idx = find_swings(window["Low"], swing_order)
    found = []
    close_last = window["Close"].iloc[-1]
    close_prev = window["Close"].iloc[-2] if len(window) >= 2 else close_last

    if len(high_idx) >= 3:
        hx, hy = high_idx[-3:], window["High"].iloc[high_idx[-3:]].values
        slope, intercept = np.polyfit(hx, hy, 1)
        line_now = slope * (len(window) - 1) + intercept
        line_prev = slope * (len(window) - 2) + intercept
        if close_prev <= line_prev and close_last > line_now:
            found.append("Пробой линии сопротивления вверх (бычий)")

    if len(low_idx) >= 3:
        lx, ly = low_idx[-3:], window["Low"].iloc[low_idx[-3:]].values
        slope, intercept = np.polyfit(lx, ly, 1)
        line_now = slope * (len(window) - 1) + intercept
        line_prev = slope * (len(window) - 2) + intercept
        if close_prev >= line_prev and close_last < line_now:
            found.append("Пробой линии поддержки вниз (медвежий)")

    return found


# =============================================================================
# 6. "СИММЕТРИИ" — голова-плечи (+ перевёрнутая) и двойная вершина/дно.
#    ("Батман" не имеет единого строгого определения — не реализован
#    отдельно, см. объяснение в шапке файла.)
# =============================================================================
def detect_symmetry_patterns(df, lookback, swing_order):
    found = []
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return found

    highs, lows = window["High"], window["Low"]
    close_last = window["Close"].iloc[-1]
    high_idx, _ = find_swings(highs, swing_order)
    _, low_idx = find_swings(lows, swing_order)
    tol = window["Close"].std() * 0.6 if window["Close"].std() > 0 else close_last * 0.01

    if len(high_idx) >= 3:
        last3 = high_idx[-3:]
        h_vals = highs.iloc[last3].values
        left_sh, head, right_sh = h_vals[0], h_vals[1], h_vals[2]
        if head > left_sh + tol and head > right_sh + tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = lows.iloc[last3[0]:last3[2]].min()
            found.append("Голова и плечи (подтверждено пробоем шеи вниз, медвежий)" if close_last < neckline
                          else "Голова и плечи (формируется, шея не пробита, медвежий)")

    if len(low_idx) >= 3:
        last3 = low_idx[-3:]
        l_vals = lows.iloc[last3].values
        left_sh, head, right_sh = l_vals[0], l_vals[1], l_vals[2]
        if head < left_sh - tol and head < right_sh - tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = highs.iloc[last3[0]:last3[2]].max()
            found.append("Перевёрнутая голова и плечи (подтверждено пробоем вверх, бычий)" if close_last > neckline
                          else "Перевёрнутая голова и плечи (формируется, бычий)")

    if len(high_idx) >= 2:
        h1, h2 = highs.iloc[high_idx[-2]], highs.iloc[high_idx[-1]]
        if abs(h1 - h2) < tol and (high_idx[-1] - high_idx[-2]) > swing_order * 2:
            trough = lows.iloc[high_idx[-2]:high_idx[-1]].min()
            found.append("Двойная вершина (подтверждена пробоем вниз, медвежий)" if close_last < trough
                          else "Двойная вершина (формируется, медвежий)")

    if len(low_idx) >= 2:
        l1, l2 = lows.iloc[low_idx[-2]], lows.iloc[low_idx[-1]]
        if abs(l1 - l2) < tol and (low_idx[-1] - low_idx[-2]) > swing_order * 2:
            peak = highs.iloc[low_idx[-2]:low_idx[-1]].max()
            found.append("Двойное дно (подтверждено пробоем вверх, бычий)" if close_last > peak
                          else "Двойное дно (формируется, бычий)")

    return found


# =============================================================================
# 7. РИСК-МЕНЕДЖМЕНТ (тот же код, что в основном скринере)
# =============================================================================
def compute_risk_levels(last_row, direction, atr_mult, rr_target, bid=None, offer=None):
    close_price, atr = last_row["Close"], last_row["ATR14"]

    entry_price = close_price
    if direction == "long" and offer and offer > 0:
        entry_price = offer
    elif direction == "short" and bid and bid > 0:
        entry_price = bid

    min_risk = entry_price * 0.001

    if direction == "long":
        stop = entry_price - atr_mult * atr
        risk_per_unit = max(entry_price - stop, min_risk)
        target = entry_price + risk_per_unit * rr_target
    else:
        stop = entry_price + atr_mult * atr
        risk_per_unit = max(stop - entry_price, min_risk)
        target = entry_price - risk_per_unit * rr_target

    spread_abs, spread_pct, risk_per_spread = None, None, None
    if bid and offer and bid > 0 and offer > bid:
        spread_abs = offer - bid
        mid = (offer + bid) / 2
        spread_pct = spread_abs / mid * 100
        risk_per_spread = risk_per_unit / spread_abs if spread_abs > 0 else None

    spread_warning = None
    if risk_per_spread is not None:
        if risk_per_spread < 3:
            spread_warning = "Спред широкий относительно стопа"
        elif risk_per_spread < 6:
            spread_warning = "Спред заметный"

    return {
        "entry": smart_round(entry_price),
        "stop_loss": smart_round(stop),
        "take_profit": smart_round(target),
        "risk_reward": rr_target,
        "spread_pct": round(spread_pct, 3) if spread_pct is not None else None,
        "risk_per_spread": round(risk_per_spread, 1) if risk_per_spread is not None else None,
        "spread_warning": spread_warning,
    }


# =============================================================================
# 8. СИГНАЛЬНЫЕ КОМПОНЕНТЫ — ровно 3, знаковые (+1/0/-1)
# =============================================================================
DEFAULT_WEIGHTS = {
    "poc_level": 1.0,     # Point of Control объёмного профиля (сильнейшая зона)
    "hvn_level": 0.7,     # второстепенная зона высокого объёма — слабее POC по построению
    "wolfe_wave": 1.0,    # волна Вульфа
    "symmetry": 1.0,      # Г-и-П / двойная вершина-дно
    "trendline": 0.8,     # пробой линии тренда (см. оговорку в комментарии функции)
    "confluence_2plus": 0.0,  # 2+ базовых сигнала совпали по направлению — калибруется отдельно,
                               # чтобы узнать, даёт ли СОВПАДЕНИЕ сигналов что-то сверх их суммы
}
COMPONENT_NAMES = list(DEFAULT_WEIGHTS.keys())
BASE_COMPONENT_NAMES = ["poc_level", "hvn_level", "wolfe_wave", "symmetry", "trendline"]


def compute_signal_components(patterns):
    c = {name: 0 for name in COMPONENT_NAMES}
    for p in patterns:
        if "POC" in p:
            c["poc_level"] = 1 if "бычий" in p else -1
        elif "зоны объёма" in p:
            c["hvn_level"] = 1 if "бычий" in p else -1
        elif "Вульфа" in p:
            c["wolfe_wave"] = 1 if "бычья" in p else -1
        elif "линии сопротивления" in p or "линии поддержки" in p:
            c["trendline"] = 1 if "бычий" in p else -1
        elif "плечи" in p or "вершина" in p or "дно" in p:
            c["symmetry"] = 1 if "бычий" in p else -1

    # Confluence: сколько БАЗОВЫХ сигналов (без учёта самого confluence)
    # согласны по направлению — отдельный компонент, калибруется сам по
    # себе, чтобы понять, есть ли эффект именно от СОВПАДЕНИЯ нескольких
    # техник одновременно, а не просто от их арифметической суммы весов.
    base_vals = [c[name] for name in BASE_COMPONENT_NAMES]
    n_bull = sum(1 for v in base_vals if v > 0)
    n_bear = sum(1 for v in base_vals if v < 0)
    if n_bull >= 2 and n_bull > n_bear:
        c["confluence_2plus"] = 1
    elif n_bear >= 2 and n_bear > n_bull:
        c["confluence_2plus"] = -1

    return c


def score_from_components(last, components, weights):
    """
    net > 0 — лонг, net < 0 — шорт. |net| — score: насколько согласованно
    (а не просто "сколько всего сработало") сигналы указывают в одну
    сторону. Противоречащие друг другу компоненты взаимно вычитаются
    (тот же принцип, что в основном скринере — см. его комментарии про
    баг с простым суммированием без учёта направления).
    """
    net = sum(weights.get(name, 0.0) * val for name, val in components.items())
    direction = "long" if net >= 0 else "short"
    return round(abs(net), 2), direction


# =============================================================================
# 9. БЭКТЕСТ + КАЛИБРОВКА (структура идентична основному скринеру)
# =============================================================================
def r_multiple(direction, entry, stop, exit_price):
    risk = abs(entry - stop)
    if risk == 0 or exit_price is None:
        return None
    pnl = (exit_price - entry) if direction == "long" else (entry - exit_price)
    return pnl / risk


def simulate_trade_path(future_df, direction, entry, stop, target, max_holding_bars):
    risk = abs(entry - stop)
    if risk == 0:
        return {"outcome": "no_data", "exit_price": None, "bars_held": 0,
                "mae_r": None, "mfe_r": None, "r_multiple": None, "path_label": "нет данных"}

    max_adverse_r, max_favorable_r = 0.0, 0.0
    bars_held = 0
    outcome, exit_price = "timeout", None

    for _, row in future_df.head(max_holding_bars).iterrows():
        bars_held += 1
        high, low = row["High"], row["Low"]
        if direction == "long":
            adverse_r = (entry - low) / risk
            favorable_r = (high - entry) / risk
            hit_stop, hit_target = low <= stop, high >= target
        else:
            adverse_r = (high - entry) / risk
            favorable_r = (entry - low) / risk
            hit_stop, hit_target = high >= stop, low <= target

        max_adverse_r = max(max_adverse_r, adverse_r)
        max_favorable_r = max(max_favorable_r, favorable_r)

        if hit_stop:
            outcome, exit_price = "stop", stop
            break
        if hit_target:
            outcome, exit_price = "target", target
            break
    else:
        tail = future_df.head(max_holding_bars)
        if len(tail) > 0:
            exit_price = tail["Close"].iloc[-1]
        outcome = "timeout"

    final_r = r_multiple(direction, entry, stop, exit_price) if exit_price is not None else None

    if outcome == "stop":
        path_label = "Сразу против" if max_favorable_r < 0.3 else "Сходил в плюс, потом развернулся и выбил"
    elif outcome == "target":
        path_label = "Дошёл до цели с сильной просадкой по пути" if max_adverse_r > 0.5 else "Дошёл до цели уверенно"
    elif outcome == "timeout":
        if final_r is not None and final_r > 0.5:
            path_label = "Шёл по сценарию, не успел дойти за отведённое время"
        elif final_r is not None and final_r < -0.3:
            path_label = "Шёл против, но не успел выбить стоп"
        else:
            path_label = "Болтался без выраженного направления"
    else:
        path_label = "нет данных"

    return {"outcome": outcome, "exit_price": exit_price, "bars_held": bars_held,
            "mae_r": round(max_adverse_r, 3), "mfe_r": round(max_favorable_r, 3),
            "r_multiple": round(final_r, 3) if final_r is not None else None,
            "path_label": path_label}


def backtest_ticker(df, config):
    trades = []
    n = len(df)
    start = 60  # меньше 210 основного скринера — здесь нет тяжёлых индикаторов с долгим прогревом
    for i in range(start, n - 5, config["backtest_stride"]):
        window = df.iloc[:i + 1]
        last = window.iloc[-1]
        if pd.isna(last["ATR14"]) or last["ATR14"] <= 0:
            continue

        _, sr_patterns = compute_volume_profile_signal(
            window, config["volume_profile_lookback_bars"], config["volume_profile_bins"],
            config["volume_profile_tolerance_atr"], config["volume_profile_min_touches"]
        )
        wolfe_p = detect_wolfe_wave(window, config["pattern_lookback_bars"], config["swing_order"])
        symmetry_p = detect_symmetry_patterns(window, config["pattern_lookback_bars"], config["swing_order"])
        trendline_p = compute_trendline_signal(window, config["pattern_lookback_bars"], config["swing_order"])
        patterns = sr_patterns + wolfe_p + symmetry_p + trendline_p

        components = compute_signal_components(patterns)
        _, direction = score_from_components(last, components, DEFAULT_WEIGHTS)
        risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"])

        future = df.iloc[i + 1:]
        if future.empty:
            continue
        path = simulate_trade_path(future, direction, risk["entry"], risk["stop_loss"],
                                    risk["take_profit"], config["backtest_max_holding_bars"])
        if path["r_multiple"] is not None:
            row = dict(components)
            row.update({"direction": direction, "outcome": path["outcome"],
                        "r_multiple": path["r_multiple"], "mae_r": path["mae_r"],
                        "mfe_r": path["mfe_r"], "path_label": path["path_label"],
                        "bars_held": path["bars_held"]})
            trades.append(row)

    return pd.DataFrame(trades)


def stop_multiplier_sensitivity(df, config, multipliers):
    results = {m: [] for m in multipliers}
    n = len(df)
    start = 60
    for i in range(start, n - 5, config["stop_sensitivity_stride"]):
        window = df.iloc[:i + 1]
        last = window.iloc[-1]
        if pd.isna(last["ATR14"]) or last["ATR14"] <= 0:
            continue

        _, sr_patterns = compute_volume_profile_signal(
            window, config["volume_profile_lookback_bars"], config["volume_profile_bins"],
            config["volume_profile_tolerance_atr"], config["volume_profile_min_touches"]
        )
        wolfe_p = detect_wolfe_wave(window, config["pattern_lookback_bars"], config["swing_order"])
        symmetry_p = detect_symmetry_patterns(window, config["pattern_lookback_bars"], config["swing_order"])
        trendline_p = compute_trendline_signal(window, config["pattern_lookback_bars"], config["swing_order"])
        patterns = sr_patterns + wolfe_p + symmetry_p + trendline_p
        components = compute_signal_components(patterns)
        _, direction = score_from_components(last, components, DEFAULT_WEIGHTS)

        future = df.iloc[i + 1:]
        if future.empty:
            continue

        for m in multipliers:
            risk = compute_risk_levels(last, direction, m, config["risk_reward_target"])
            path = simulate_trade_path(future, direction, risk["entry"], risk["stop_loss"],
                                        risk["take_profit"], config["backtest_max_holding_bars"])
            if path["r_multiple"] is not None:
                results[m].append({"outcome": path["outcome"], "r_multiple": path["r_multiple"]})

    return results


def run_stop_sensitivity_analysis(data_by_ticker, config):
    tickers = list(data_by_ticker.keys())[:config["stop_sensitivity_tickers"]]
    multipliers = config["stop_sensitivity_multipliers"]
    combined = {m: [] for m in multipliers}

    for t in tqdm(tickers, desc="   Проверка разных множителей стопа по тикерам"):
        try:
            res = stop_multiplier_sensitivity(data_by_ticker[t], config, multipliers)
            for m in multipliers:
                combined[m].extend(res[m])
        except Exception:
            continue

    rows = []
    for m in multipliers:
        trades = combined[m]
        if not trades:
            continue
        n = len(trades)
        win_rate = sum(1 for x in trades if x["outcome"] == "target") / n * 100
        avg_r = sum(x["r_multiple"] for x in trades) / n
        rows.append({"Множитель ATR (стоп)": m, "Сделок": n, "Винрейт %": round(win_rate, 1),
                     "Средний_R": round(avg_r, 3), "Сумма_R": round(sum(x["r_multiple"] for x in trades), 1)})
    return pd.DataFrame(rows)


def compute_component_aggregates(trades):
    aggs = {}
    direction_sign = trades["direction"].map({"long": 1, "short": -1})
    for name in COMPONENT_NAMES:
        if name not in trades.columns:
            continue
        mask = trades[name] == direction_sign
        aggs[name] = {
            "n_true": int(mask.sum()),
            "sum_true": float(trades.loc[mask, "r_multiple"].sum()),
            "n_false": int((~mask).sum()),
            "sum_false": float(trades.loc[~mask, "r_multiple"].sum()),
        }
    return aggs


def merge_aggregates(a, b):
    empty = {"n_true": 0, "sum_true": 0.0, "n_false": 0, "sum_false": 0.0}
    merged = {}
    for name in set(a.keys()) | set(b.keys()):
        av, bv = a.get(name, empty), b.get(name, empty)
        merged[name] = {
            "n_true": av["n_true"] + bv["n_true"],
            "sum_true": av["sum_true"] + bv["sum_true"],
            "n_false": av["n_false"] + bv["n_false"],
            "sum_false": av["sum_false"] + bv["sum_false"],
        }
    return merged


CALIBRATION_CACHE_KEY = "calibration_aggregates_purist_v1"


def load_accumulated_calibration(cache_dir):
    try:
        return _load_from_cache(cache_dir, CALIBRATION_CACHE_KEY, ttl_hours=24 * 30)
    except Exception:
        return None


def save_accumulated_calibration(cache_dir, aggs):
    try:
        _save_to_cache(cache_dir, CALIBRATION_CACHE_KEY, aggs)
    except Exception:
        pass


def fit_weights_from_aggregates(aggs, min_samples=40):
    diag_rows, new_weights = [], {}
    for name in COMPONENT_NAMES:
        a = aggs.get(name)
        if not a or a["n_true"] < min_samples or a["n_false"] < min_samples:
            new_weights[name] = DEFAULT_WEIGHTS[name]
            diag_rows.append({"Компонент": name, "Случаев": a["n_true"] if a else 0,
                               "Средний_R(есть)": None, "Средний_R(нет)": None, "Лифт": None,
                               "Вес": f"{DEFAULT_WEIGHTS[name]} (данных мало, оставлен дефолт)"})
            continue
        mean_true = a["sum_true"] / a["n_true"]
        mean_false = a["sum_false"] / a["n_false"]
        lift = mean_true - mean_false
        diag_rows.append({"Компонент": name, "Случаев": a["n_true"],
                           "Средний_R(есть)": round(mean_true, 3),
                           "Средний_R(нет)": round(mean_false, 3),
                           "Лифт": round(lift, 3), "Вес": None})
        new_weights[name] = lift

    raw_positive = [w for name, w in new_weights.items()
                    if isinstance(w, float) and w > 0 and "оставлен дефолт" not in
                    str(next((r["Вес"] for r in diag_rows if r["Компонент"] == name), ""))]
    scale = (1.5 / max(raw_positive)) if raw_positive else 1.0

    final_weights = {}
    for row in diag_rows:
        name = row["Компонент"]
        if row["Вес"] is not None:
            final_weights[name] = DEFAULT_WEIGHTS[name]
        else:
            w = max(row["Лифт"], 0) * scale
            final_weights[name] = round(w, 2)
            row["Вес"] = round(w, 2)

    return final_weights, pd.DataFrame(diag_rows)


def score_for_fixed_direction(components, direction, weights):
    net = sum(weights.get(name, 0.0) * val for name, val in components.items())
    return round(net if direction == "long" else -net, 2)


def bucket_summary_for(trades, score_col):
    bins = [-100, 1, 2, 3, 100]
    labels = ["<1 (слабый)", "1-2", "2-3", "3+ (сильный)"]
    t = trades.copy()
    t["bucket"] = pd.cut(t[score_col], bins=bins, labels=labels)
    return t.groupby("bucket", observed=True).agg(
        Сделок=("r_multiple", "count"),
        Винрейт=("outcome", lambda x: round((x == "target").mean() * 100, 1)),
        Средний_R=("r_multiple", lambda x: round(x.mean(), 3)),
        Сумма_R=("r_multiple", lambda x: round(x.sum(), 1)),
    ).reset_index()


def run_backtest(data_by_ticker, config):
    tickers = list(data_by_ticker.keys())[:config["backtest_max_tickers"]]
    all_trades = []
    for t in tqdm(tickers, desc="   Бэктест по тикерам"):
        try:
            tr = backtest_ticker(data_by_ticker[t], config)
            if not tr.empty:
                tr["ticker"] = t
                all_trades.append(tr)
        except Exception:
            continue

    if not all_trades:
        return None, None, None, None

    trades = pd.concat(all_trades, ignore_index=True)

    this_run_aggs = compute_component_aggregates(trades)
    if config.get("persist_calibration_across_runs", True):
        cached = load_accumulated_calibration(config["cache_dir"])
        accumulated = merge_aggregates(cached, this_run_aggs) if cached else this_run_aggs
        save_accumulated_calibration(config["cache_dir"], accumulated)
    else:
        accumulated = this_run_aggs

    fitted_weights, diagnostics = fit_weights_from_aggregates(
        accumulated, min_samples=config["backtest_min_samples_per_component"]
    )
    if config.get("persist_calibration_across_runs", True) and diagnostics is not None:
        total_acc = sum(a["n_true"] + a["n_false"] for a in accumulated.values()) // max(len(accumulated), 1)
        print(f"   Накоплено данных калибровки: ~{total_acc} сделок за все прогоны в этой сессии.")

    trades["score_default"] = trades.apply(
        lambda row: score_for_fixed_direction({c: row[c] for c in COMPONENT_NAMES}, row["direction"], DEFAULT_WEIGHTS),
        axis=1
    )
    trades["score_fitted"] = trades.apply(
        lambda row: score_for_fixed_direction({c: row[c] for c in COMPONENT_NAMES}, row["direction"], fitted_weights),
        axis=1
    )

    summary_default = bucket_summary_for(trades, "score_default")
    summary_fitted = bucket_summary_for(trades, "score_fitted")

    return trades, summary_default, summary_fitted, {"weights": fitted_weights, "diagnostics": diagnostics}


# =============================================================================
# ГЛАВНЫЙ PIPELINE
# =============================================================================
def run_screener(config=CONFIG):
    print("ЧИСТЫЙ ТЕСТ: только статистические уровни + волна Вульфа + симметрии\n")
    print("1/5: Быстрый префильтр по ликвидности + текущий стакан (bid/offer)...")
    snapshot = get_market_snapshot()
    top_n = config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(snapshot, top_n)
    quotes = snapshot.set_index("SECID")[["BID", "OFFER"]].to_dict("index")
    print(f"   Беру в работу {len(tickers)} тикеров")

    print("2/5: Скачиваю часовые свечи параллельно...")
    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )

    data_by_ticker = {}
    skipped_low_vol, failed_indicator_tickers = 0, []
    for t, raw in raw_by_ticker.items():
        if len(raw) < 60:
            continue
        try:
            df_ind = add_atr(raw)
            median_atr_pct = df_ind["ATR_pct"].tail(200).median()
            if median_atr_pct < config["min_atr_pct"]:
                skipped_low_vol += 1
                continue
            data_by_ticker[t] = df_ind
        except Exception as e:
            failed_indicator_tickers.append((t, str(e)))

    print(f"   Хватает истории для анализа: {len(data_by_ticker)} тикеров "
          f"(отсеяно как низковолатильные фонды: {skipped_low_vol})")

    print("3/5: Бэктест на истории + калибровка весов score по факту...")
    fitted_weights, weight_diagnostics = dict(DEFAULT_WEIGHTS), None
    backtest_trades, backtest_summary_default, backtest_summary_fitted = None, None, None
    if config["backtest_enabled"]:
        print(f"   ~{config['backtest_max_tickers']} тикеров...")
        backtest_trades, backtest_summary_default, backtest_summary_fitted, fit_result = run_backtest(data_by_ticker, config)
        if fit_result is not None:
            fitted_weights = fit_result["weights"]
            weight_diagnostics = fit_result["diagnostics"]
            print("   Веса пересчитаны по факту истории — используются в отчёте ниже.")

            if backtest_trades is not None and not backtest_trades.empty:
                print("\n   Как вели себя цены в сделках бэктеста (path_label):")
                path_counts = backtest_trades["path_label"].value_counts()
                path_pct = (path_counts / len(backtest_trades) * 100).round(1)
                for label, cnt in path_counts.items():
                    print(f"     {label}: {cnt} ({path_pct[label]}%)")
                print(f"   Средняя MAE: {backtest_trades['mae_r'].mean():.2f}R, "
                      f"средняя MFE: {backtest_trades['mfe_r'].mean():.2f}R")
        else:
            print("   Недостаточно сделок для калибровки — использую веса по умолчанию.")

    stop_sensitivity_df = None
    if config["stop_sensitivity_enabled"] and len(data_by_ticker) > 0:
        print(f"\n3b/5: Чувствительность к множителю стопа "
              f"({config['stop_sensitivity_tickers']} тикеров)...")
        stop_sensitivity_df = run_stop_sensitivity_analysis(data_by_ticker, config)
        if stop_sensitivity_df is not None and not stop_sensitivity_df.empty:
            print("\n" + "=" * 100)
            print("ЧУВСТВИТЕЛЬНОСТЬ К МНОЖИТЕЛЮ СТОПА")
            print("=" * 100)
            print(stop_sensitivity_df.to_string(index=False))

    print("\n4/5: Считаю сигналы (уровни объёма, волна Вульфа, симметрии), риск-уровни и score...")
    results = []
    failed_result_tickers = []
    for t, df in data_by_ticker.items():
        try:
            _, sr_patterns = compute_volume_profile_signal(
                df, config["volume_profile_lookback_bars"], config["volume_profile_bins"],
                config["volume_profile_tolerance_atr"], config["volume_profile_min_touches"]
            )
            wolfe_patterns = detect_wolfe_wave(df, config["pattern_lookback_bars"], config["swing_order"])
            symmetry_patterns = detect_symmetry_patterns(df, config["pattern_lookback_bars"], config["swing_order"])
            trendline_patterns = compute_trendline_signal(df, config["pattern_lookback_bars"], config["swing_order"])
            patterns = sr_patterns + wolfe_patterns + symmetry_patterns + trendline_patterns
            last = df.iloc[-1]

            components = compute_signal_components(patterns)
            score, direction = score_from_components(last, components, fitted_weights)
            if score == 0:
                continue  # ни один из 3 сигналов не сработал вообще — не показываем "пустышку"

            q = quotes.get(t, {})
            bid, offer = q.get("BID"), q.get("OFFER")
            risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"],
                                        bid=bid, offer=offer)
        except Exception as e:
            failed_result_tickers.append((t, str(e)))
            continue

        results.append({
            "Тикер": t,
            "Цена": smart_round(last["Close"]),
            "Направление": "ЛОНГ" if direction == "long" else "ШОРТ",
            "Score": score,
            "Сигналы": "; ".join(patterns) if patterns else "-",
            "Вход": risk["entry"],
            "Стоп-лосс": risk["stop_loss"],
            "Тейк-профит": risk["take_profit"],
            "Risk/Reward": risk["risk_reward"],
            "Спред %": risk["spread_pct"],
            "Риск/спред x": risk["risk_per_spread"],
            "⚠ Спред": risk["spread_warning"] or "-",
        })

    report = pd.DataFrame(results)
    if not report.empty:
        report = report.sort_values("Score", ascending=False).reset_index(drop=True)

    if failed_result_tickers:
        print(f"   Пропущено: {len(failed_result_tickers)} тикеров "
              f"(напр. {failed_result_tickers[0][0]}: {failed_result_tickers[0][1][:80]})")

    if weight_diagnostics is not None:
        print("\n" + "=" * 100)
        print("КАКИЕ ИЗ 3 КОМПОНЕНТОВ РЕАЛЬНО ПОКАЗАЛИ ЭДЖ НА ИСТОРИИ")
        print("=" * 100)
        print(weight_diagnostics.to_string(index=False))

    if backtest_summary_default is not None and backtest_summary_fitted is not None:
        print("\n" + "=" * 100)
        print("ДО/ПОСЛЕ калибровки")
        print("=" * 100)
        print("-- ДО --")
        print(backtest_summary_default.to_string(index=False))
        print("-- ПОСЛЕ --")
        print(backtest_summary_fitted.to_string(index=False))

    print(f"\n5/5: Готово. Рекомендаций (хотя бы один из 3 сигналов сработал): {len(report)}\n")
    print("=" * 100)
    print(f"ТОП-{config['top_n_report']} ИДЕЙ (не финансовая рекомендация!)")
    print("=" * 100)
    if report.empty:
        print("Ни на одном тикере не сработал ни один из 3 сигналов на момент запуска.")
    else:
        with pd.option_context("display.max_columns", None, "display.width", 220):
            print(report.head(config["top_n_report"]).to_string(index=False))

    return report


# =============================================================================
# WALK-FORWARD ВАЛИДАЦИЯ (структура идентична основному скринеру)
# =============================================================================
def _generate_asof_report(visible_by_ticker, weights, config):
    results = []
    for t, raw in visible_by_ticker.items():
        if len(raw) < 60:
            continue
        try:
            df_ind = add_atr(raw)
            if df_ind["ATR_pct"].tail(200).median() < config["min_atr_pct"]:
                continue

            _, sr_patterns = compute_volume_profile_signal(
                df_ind, config["volume_profile_lookback_bars"], config["volume_profile_bins"],
                config["volume_profile_tolerance_atr"], config["volume_profile_min_touches"]
            )
            wolfe_patterns = detect_wolfe_wave(df_ind, config["pattern_lookback_bars"], config["swing_order"])
            symmetry_patterns = detect_symmetry_patterns(df_ind, config["pattern_lookback_bars"], config["swing_order"])
            trendline_patterns = compute_trendline_signal(df_ind, config["pattern_lookback_bars"], config["swing_order"])
            patterns = sr_patterns + wolfe_patterns + symmetry_patterns + trendline_patterns
            last = df_ind.iloc[-1]

            components = compute_signal_components(patterns)
            score, direction = score_from_components(last, components, weights)
            if score == 0:
                continue
            risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"])

            results.append({
                "Тикер": t, "direction": direction, "Score": score,
                "Вход": risk["entry"], "Стоп-лосс": risk["stop_loss"], "Тейк-профит": risk["take_profit"],
            })
        except Exception:
            continue

    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values("Score", ascending=False).reset_index(drop=True)
    return df


def run_walkforward_test(config=CONFIG):
    print("Walk-forward валидация (ЧИСТЫЙ ТЕСТ): 'что бы сказали 3 сигнала тогда — и что было по факту'\n")
    print("1/4: Тикеры + полная история...")
    snapshot = get_market_snapshot()
    top_n = config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(snapshot, top_n)

    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )

    all_dates = pd.concat([df["Date"] for df in raw_by_ticker.values()], ignore_index=True)
    min_date, max_date = all_dates.min(), all_dates.max()

    history_buffer = pd.Timedelta(days=15)  # меньше основного скринера — здесь нет индикаторов с долгим прогревом
    eval_buffer = pd.Timedelta(hours=config["walkforward_eval_max_bars"] * 2)
    window_start = min_date + history_buffer
    window_end = max_date - eval_buffer

    if window_start >= window_end:
        print("   Недостаточно истории для walk-forward теста с текущими настройками.")
        return None

    n = config["walkforward_n_dates"]
    test_dates = pd.date_range(window_start, window_end, periods=n) if n > 1 else pd.DatetimeIndex([window_end])
    print(f"   Диапазон истории: {min_date.date()} — {max_date.date()}")
    print(f"   Тестовые даты ({n}): {', '.join(d.strftime('%Y-%m-%d %H:%M') for d in test_dates)}\n")

    all_eval_rows = []
    for di, as_of in enumerate(test_dates, 1):
        print(f"2/4: [{di}/{n}] Дата среза: {as_of.strftime('%Y-%m-%d %H:%M')}...")
        visible_by_ticker = {t: df[df["Date"] <= as_of].reset_index(drop=True) for t, df in raw_by_ticker.items()}
        future_by_ticker = {t: df[df["Date"] > as_of].reset_index(drop=True) for t, df in raw_by_ticker.items()}

        data_by_ticker = {}
        for t, raw in visible_by_ticker.items():
            if len(raw) < 60:
                continue
            try:
                df_ind = add_atr(raw)
                if df_ind["ATR_pct"].tail(200).median() < config["min_atr_pct"]:
                    continue
                data_by_ticker[t] = df_ind
            except Exception:
                continue

        if len(data_by_ticker) < 10:
            print(f"   Мало тикеров на эту дату ({len(data_by_ticker)}), пропускаю.")
            continue

        wf_backtest_config = dict(config)
        wf_backtest_config["backtest_max_tickers"] = config["walkforward_calibration_tickers"]
        wf_backtest_config["persist_calibration_across_runs"] = False
        _, _, _, fit_result = run_backtest(data_by_ticker, wf_backtest_config)
        weights = fit_result["weights"] if fit_result else dict(DEFAULT_WEIGHTS)

        report = _generate_asof_report(visible_by_ticker, weights, config)
        if report.empty:
            print("   Ни один сигнал не сработал на эту дату, пропускаю.")
            continue

        top = report.head(config["walkforward_top_n"])
        print(f"   Рекомендаций на эту дату: {len(top)}. Проверяю по факту истории...")

        for _, row in top.iterrows():
            t = row["Тикер"]
            future = future_by_ticker.get(t)
            if future is None or future.empty:
                continue
            path = simulate_trade_path(future, row["direction"], row["Вход"], row["Стоп-лосс"],
                                        row["Тейк-профит"], config["walkforward_eval_max_bars"])
            if path["r_multiple"] is None:
                continue
            all_eval_rows.append({
                "Дата среза": as_of.strftime("%Y-%m-%d %H:%M"),
                "Тикер": t,
                "Направление": "ЛОНГ" if row["direction"] == "long" else "ШОРТ",
                "Score": row["Score"],
                "Вход": row["Вход"], "Стоп": row["Стоп-лосс"], "Тейк": row["Тейк-профит"],
                "Исход": {"target": "✅ Тейк", "stop": "❌ Стоп", "timeout": "⏱ Таймаут"}.get(path["outcome"], path["outcome"]),
                "R": path["r_multiple"],
            })

    print("\n3/4: Готово.\n")
    if not all_eval_rows:
        print("Не набралось ни одной проверяемой рекомендации.")
        return None

    eval_df = pd.DataFrame(all_eval_rows)
    print("=" * 100)
    print(f"WALK-FORWARD РЕЗУЛЬТАТ (ЧИСТЫЙ ТЕСТ): {len(eval_df)} рекомендаций на {len(test_dates)} датах")
    print("=" * 100)
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(eval_df.to_string(index=False))

    print("\n4/4: Сводка")
    print("=" * 100)
    win_rate = (eval_df["Исход"] == "✅ Тейк").mean() * 100
    avg_r, sum_r = eval_df["R"].mean(), eval_df["R"].sum()
    print(f"Всего рекомендаций проверено: {len(eval_df)}")
    print(f"Винрейт: {win_rate:.1f}%   Средний R: {avg_r:.3f}   Суммарный R: {sum_r:.1f}")

    by_date = eval_df.groupby("Дата среза").agg(
        Рекомендаций=("R", "count"),
        Винрейт=("Исход", lambda x: round((x == "✅ Тейк").mean() * 100, 1)),
        Средний_R=("R", lambda x: round(x.mean(), 3)),
    ).reset_index()
    print("\nПо каждой тестовой дате отдельно:")
    print(by_date.to_string(index=False))

    print(f"\n⚠ Это {len(test_dates)} исторических даты — доверительный интервал большой,")
    print("это ориентир, а не статистическое доказательство.")

    return eval_df


# =============================================================================
# ЗАПУСК
# =============================================================================
if __name__ == "__main__":
    full_report = run_screener(CONFIG)
    walkforward_results = run_walkforward_test(CONFIG)

ЧИСТЫЙ ТЕСТ: только статистические уровни + волна Вульфа + симметрии

1/5: Быстрый префильтр по ликвидности + текущий стакан (bid/offer)...
   Беру в работу 120 тикеров
2/5: Скачиваю часовые свечи параллельно...


   Качаю свечи параллельно:   0%|          | 0/120 [00:00<?, ?it/s]

   Хватает истории для анализа: 104 тикеров (отсеяно как низковолатильные фонды: 16)
3/5: Бэктест на истории + калибровка весов score по факту...
   ~20 тикеров...


   Бэктест по тикерам:   0%|          | 0/20 [00:00<?, ?it/s]

   Накоплено данных калибровки: ~11526 сделок за все прогоны в этой сессии.
   Веса пересчитаны по факту истории — используются в отчёте ниже.

   Как вели себя цены в сделках бэктеста (path_label):
     Сходил в плюс, потом развернулся и выбил: 4934 (42.8%)
     Сразу против: 2610 (22.6%)
     Дошёл до цели уверенно: 2106 (18.3%)
     Дошёл до цели с сильной просадкой по пути: 1245 (10.8%)
     Шёл по сценарию, не успел дойти за отведённое время: 291 (2.5%)
     Болтался без выраженного направления: 269 (2.3%)
     Шёл против, но не успел выбить стоп: 71 (0.6%)
   Средняя MAE: 1.12R, средняя MFE: 1.22R

3b/5: Чувствительность к множителю стопа (10 тикеров)...


...

   Рекомендаций на эту дату: 10. Проверяю по факту истории...
2/4: [4/5] Дата среза: 2026-06-13 11:15...


   Бэктест по тикерам:   0%|          | 0/15 [00:00<?, ?it/s]

   Рекомендаций на эту дату: 10. Проверяю по факту истории...
2/4: [5/5] Дата среза: 2026-08-07 13:00...


   Бэктест по тикерам:   0%|          | 0/15 [00:00<?, ?it/s]

   Рекомендаций на эту дату: 10. Проверяю по факту истории...

3/4: Готово.

WALK-FORWARD РЕЗУЛЬТАТ (ЧИСТЫЙ ТЕСТ): 50 рекомендаций на 5 датах
      Дата среза Тикер Направление  Score      Вход      Стоп      Тейк     Исход      R
2025-12-30 06:00 MTLRP        ЛОНГ   1.93   66.5000   65.7500   68.0000    ❌ Стоп -1.000
2025-12-30 06:00  MTSS        ШОРТ   1.93  212.3500  213.5800  209.8800    ❌ Стоп -1.000
2025-12-30 06:00  MVID        ШОРТ   1.93   81.2500   82.1300   79.5000    ❌ Стоп -1.000
2025-12-30 06:00 TRNFP        ЛОНГ   1.50 1377.2000 1368.1000 1395.4100    ❌ Стоп -1.000
2025-12-30 06:00  LKOH        ЛОНГ   1.50 5856.5000 5804.5500 5960.4000 ⏱ Таймаут  0.337
2025-12-30 06:00  MAGN        ЛОНГ   1.50   28.8000   28.5100   29.3900    ❌ Стоп -1.000
2025-12-30 06:00 SNGSP        ЛОНГ   1.50   40.8700   40.5900   41.4300    ✅ Тейк  2.000
2025-12-30 06:00     T        ЛОНГ   1.50  327.4000  324.9500  332.2900    ❌ Стоп -1.000
2025-12-30 06:00  OZON        ЛОНГ   1.50 4520.0000 4474.

In [3]:
# -*- coding: utf-8 -*-
"""
MOEX Screener v3 — ТОЛЬКО ДОКАЗАННО РАБОЧИЕ КОМПОНЕНТЫ
====================================================================================
Третий скрипт — не новый набор гипотез, а синтез того, что реально показало
устойчивый результат в скриптах 1 и 2 за несколько независимых прогонов на
реальных данных MOEX. Отбор был по ОДНОМУ критерию: знак лифта (средний R
"сигнал сработал" минус "не сработал") не менялся между прогонами. Компонент,
который то в плюсе, то в минусе — не входит сюда, сколько бы качественная
идея за ним ни стояла.

ВЗЯТО (со стартовыми весами, пропорциональными числу и стабильности измерений
— это ТОЛЬКО отправная точка, реальные веса всё равно пересчитает калибровка
по бэктесту этого скрипта):
  - trend       (скрипт 1, EMA20>50>200)        — 4/4 измерений в плюс, вес 1.5
  - vwap_dev    (скрипт 1, откл. цены от VWAP)   — 4/4 измерений в плюс, вес 1.4
  - symmetry    (скрипт 2, Г-и-П + дв.вершина/дно) — 3/3 измерений в плюс, вес 1.2
  - pattern_playing_out (скрипт 1, паттерн уже частично отработал) — 3/3, вес 1.0
  - rel_strength (скрипт 1, относительная сила к IMOEX) — 3/4, вес 0.8
  - fib_level   (скрипт 1, уровни Фибоначчи)     — 3/3, но маленькая величина, вес 0.5

НЕ ВЗЯТО (не подтвердили устойчивость — знак прыгал между прогонами):
  волна Вульфа (в ОБОИХ скриптах отдельно — не совпадение), объёмный профиль
  / POC / HVN, confluence (совпадение сигналов), линии тренда (пока 1
  измерение — рано доверять), RSI/MACD/Stochastic/Williams %R, дивергенции,
  свечные паттерны, ML (AUC стабильно ~0.52-0.53, слишком слабо, чтобы того
  стоило).

Убрана зависимость от pandas-ta-classic — она была нужна только для
свечных паттернов/Stochastic/CCI/WillR/OBV/Aroon/SuperTrend, ни один из
которых сюда не попал. Скрипт из-за этого легче и быстрее.

Структура пайплайна (загрузка данных, риск-менеджмент, калибровка весов по
лифту, MAE/MFE и классификация путей сделок, свип по множителю стопа,
walk-forward валидация) — идентична скриптам 1 и 2.

Как запустить в Google Colab:
  1. Вставьте содержимое этого файла в отдельную ячейку и запустите
  2. Runtime → Change runtime type → Hardware accelerator → None
  3. В конце файла — run_screener(CONFIG) и (закомментированный)
     run_walkforward_test(CONFIG)
"""

import os
import time
import pickle
import warnings
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.signal import argrelextrema

warnings.filterwarnings("ignore")


def _ensure(pkg_import_name, pip_name=None):
    try:
        return __import__(pkg_import_name)
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", pip_name or pkg_import_name])
        return __import__(pkg_import_name)


_ensure("tqdm")
from tqdm.auto import tqdm


# =============================================================================
# CONFIG
# =============================================================================
CONFIG = {
    "history_days": 240,
    "candle_interval": 60,
    "max_tickers": None,
    "liquidity_top_n": 120,
    "min_atr_pct": 0.15,
    "atr_stop_mult": 1.5,
    "risk_reward_target": 2.0,
    "top_n_report": 15,
    "max_workers": 10,
    "cache_dir": "/content/moex_cache_v3",
    "cache_ttl_hours": 6,
    "swing_order": 3,
    "pattern_lookback_bars": 90,
    "rel_strength_window": 20,
    "fib_tolerance_atr": 0.35,
    "backtest_enabled": True,
    "backtest_max_tickers": 20,
    "backtest_stride": 6,
    "backtest_max_holding_bars": 40,
    "backtest_min_samples_per_component": 40,
    "persist_calibration_across_runs": True,
    "walkforward_n_dates": 20,
    "walkforward_top_n": 10,          # верхний потолок числа рекомендаций на дату (после фильтра по score)
    "walkforward_min_score": 1.5,    # None = без порога (как раньше). Число (напр. 3.0) = показывать
                                       # только сигналы СИЛЬНЕЕ этого порога; если в какой-то день никто
                                       # не прошёл порог — на эту дату сигналов просто нет, а не "заполним чем есть"
    "walkforward_eval_max_bars": 60,
    "walkforward_calibration_tickers": 15,
    "stop_sensitivity_enabled": True,
    "stop_sensitivity_multipliers": [1.0, 1.5, 2.0, 2.5, 3.0],
    "stop_sensitivity_tickers": 10,
    "stop_sensitivity_stride": 10,
}

ISS_BASE = "https://iss.moex.com/iss"


# =============================================================================
# ФОРМАТИРОВАНИЕ ЦЕН
# =============================================================================
def smart_round(x, sig_figs=4):
    if x is None or not np.isfinite(x):
        return x
    if x == 0:
        return 0.0
    if abs(x) >= 1:
        return round(x, 2)
    magnitude = int(np.floor(np.log10(abs(x))))
    decimals = min(max(-magnitude + (sig_figs - 1), 2), 8)
    return round(x, decimals)


# =============================================================================
# 1. ЗАГРУЗКА ДАННЫХ С MOEX ISS API
# =============================================================================
def get_market_snapshot():
    url = f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {
        "iss.meta": "off",
        "iss.only": "securities,marketdata",
        "securities.columns": "SECID,SHORTNAME",
        "marketdata.columns": "SECID,VALTODAY,BID,OFFER,LAST",
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    js = r.json()
    sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
    md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])
    for col in ["VALTODAY", "BID", "OFFER", "LAST"]:
        md[col] = pd.to_numeric(md[col], errors="coerce")
    md["VALTODAY"] = md["VALTODAY"].fillna(0)
    return sec.merge(md, on="SECID", how="left")


def get_liquid_tickers(snapshot, top_n):
    merged = snapshot.copy()
    if merged["VALTODAY"].sum() > 0:
        merged = merged.sort_values("VALTODAY", ascending=False)
    return merged["SECID"].head(top_n).tolist()


def _cache_path(cache_dir, key):
    return os.path.join(cache_dir, f"{key}.pkl")


def _load_from_cache(cache_dir, key, ttl_hours):
    path = _cache_path(cache_dir, key)
    if not os.path.exists(path):
        return None
    age_hours = (time.time() - os.path.getmtime(path)) / 3600
    if age_hours > ttl_hours:
        return None
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except Exception:
        return None


def _save_to_cache(cache_dir, key, df):
    os.makedirs(cache_dir, exist_ok=True)
    with open(_cache_path(cache_dir, key), "wb") as f:
        pickle.dump(df, f)


def _fetch_candles_raw(url, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    all_rows, start, columns = [], 0, None
    while True:
        params = {
            "from": since.strftime("%Y-%m-%d"),
            "till": till.strftime("%Y-%m-%d"),
            "interval": interval,
            "start": start,
        }
        r = requests.get(url, params=params, timeout=20)
        if r.status_code != 200:
            break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None:
            columns = js.get("columns", [])
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 500:
            break
        start += len(rows)

    if not all_rows or columns is None:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.rename(columns={
        "open": "Open", "close": "Close", "high": "High",
        "low": "Low", "volume": "Volume", "begin": "Date", "end": "DateEnd"
    })
    df = df[["Date", "DateEnd", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date")
    df = df.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)

    if len(df) > 0 and df["DateEnd"].iloc[-1] > pd.Timestamp.now():
        df = df.iloc[:-1]

    return df.drop(columns=["DateEnd"]).reset_index(drop=True)


def get_hourly_candles(ticker, days_back, interval=60):
    url = (f"{ISS_BASE}/engines/stock/markets/shares/boards/TQBR/"
           f"securities/{ticker}/candles.json")
    return _fetch_candles_raw(url, days_back, interval)


def get_index_candles(secid, days_back, interval=60):
    url = f"{ISS_BASE}/engines/stock/markets/index/boards/SNDX/securities/{secid}/candles.json"
    return _fetch_candles_raw(url, days_back, interval)


def download_all_candles(tickers, days_back, interval, cache_dir, ttl_hours, max_workers):
    results = {}
    to_download = []
    for t in tickers:
        cached = _load_from_cache(cache_dir, t, ttl_hours)
        if cached is not None and not cached.empty:
            results[t] = cached
        else:
            to_download.append(t)

    if results:
        print(f"   Из кэша сессии загружено сразу: {len(results)} тикеров")

    if to_download:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(get_hourly_candles, t, days_back, interval): t
                       for t in to_download}
            for fut in tqdm(as_completed(futures), total=len(futures),
                             desc="   Качаю свечи параллельно"):
                t = futures[fut]
                try:
                    df = fut.result()
                    if not df.empty:
                        results[t] = df
                        _save_to_cache(cache_dir, t, df)
                except Exception:
                    pass

    return results


# =============================================================================
# 2. ИНДИКАТОРЫ — только то, что реально нужно 6 отобранным компонентам:
#    EMA20/50/200 (trend), VWAP (vwap_dev), ATR (риск-менеджмент, всем нужен)
# =============================================================================
def add_core_indicators(df):
    df = df.copy()
    df["EMA20"] = df["Close"].ewm(span=20, adjust=False).mean()
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()
    df["trend_up"] = ((df["EMA20"] > df["EMA50"]) & (df["EMA50"] > df["EMA200"])).astype(int)
    df["trend_down"] = ((df["EMA20"] < df["EMA50"]) & (df["EMA50"] < df["EMA200"])).astype(int)

    prev_close = df["Close"].shift(1)
    tr = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs()
    ], axis=1).max(axis=1)
    df["ATR14"] = tr.ewm(alpha=1/14, adjust=False).mean()
    df["ATR_pct"] = df["ATR14"] / df["Close"] * 100

    session = df["Date"].dt.date
    typical_price = (df["High"] + df["Low"] + df["Close"]) / 3
    pv = typical_price * df["Volume"]
    df["VWAP"] = pv.groupby(session).cumsum() / df["Volume"].groupby(session).cumsum().replace(0, np.nan)
    df["price_vs_vwap_pct"] = (df["Close"] - df["VWAP"]) / df["VWAP"] * 100

    return df


def compute_relative_strength(df, index_df, window):
    """Относительная сила к IMOEX — та же реализация, что в скрипте 1
    (merge_asof, устойчиво к дублирующимся барам от пагинации MOEX)."""
    if index_df is None or index_df.empty:
        df["rel_strength_pct"] = np.nan
        return df
    idx = index_df[["Date", "Close"]].rename(columns={"Close": "Index_Close"})
    idx = idx.drop_duplicates(subset="Date").sort_values("Date")
    base = df[["Date"]].sort_values("Date")
    aligned = pd.merge_asof(base, idx, on="Date", direction="backward")
    aligned = aligned.set_index(base.index).reindex(df.index)
    ticker_ret = df["Close"].pct_change(window)
    index_ret = aligned["Index_Close"].pct_change(window)
    df["rel_strength_pct"] = (ticker_ret.values - index_ret.values) * 100
    return df


# =============================================================================
# 3. СВИНГИ
# =============================================================================
def find_swings(series, order):
    values = series.values
    highs_idx = argrelextrema(values, np.greater_equal, order=order)[0]
    lows_idx = argrelextrema(values, np.less_equal, order=order)[0]
    highs_idx = np.array(sorted(set(highs_idx.tolist())))
    lows_idx = np.array(sorted(set(lows_idx.tolist())))
    return highs_idx, lows_idx


# =============================================================================
# 4. ФИБОНАЧЧИ (та же реализация, что в скрипте 1)
# =============================================================================
FIB_RATIOS = [0.236, 0.382, 0.5, 0.618, 0.786]


def compute_fibonacci_signal(df, lookback, swing_order, tolerance_atr_mult):
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return []

    high_idx, _ = find_swings(window["High"], swing_order)
    _, low_idx = find_swings(window["Low"], swing_order)
    last_high_i = high_idx[-1] if len(high_idx) else None
    last_low_i = low_idx[-1] if len(low_idx) else None
    if last_high_i is None or last_low_i is None:
        return []

    close_last = window["Close"].iloc[-1]
    atr_last = df["ATR14"].iloc[-1] if "ATR14" in df.columns and pd.notna(df["ATR14"].iloc[-1]) else close_last * 0.01
    tol = atr_last * tolerance_atr_mult

    swing_low, swing_high = window["Low"].iloc[last_low_i], window["High"].iloc[last_high_i]
    direction_bias = "long" if last_low_i < last_high_i else "short"
    span = swing_high - swing_low
    if span <= 0:
        return []

    levels = {}
    for r in FIB_RATIOS:
        levels[f"{r*100:.1f}%"] = (swing_high - span * r) if direction_bias == "long" else (swing_low + span * r)

    found = []
    for label, price in levels.items():
        if abs(close_last - price) <= tol:
            tag = "бычий" if direction_bias == "long" else "медвежий"
            found.append(f"Цена у уровня Фибоначчи {label} ({tag})")
            break

    return found


# =============================================================================
# 5. СИММЕТРИИ (голова-плечи + двойная вершина/дно) СО СТАДИЕЙ ОТРАБОТКИ
#    — комбинация чистого детектора из скрипта 2 (без треугольников и
#    пробоя диапазона — они не входили в проверенный набор) и логики
#    "стадии" из скрипта 1 (pattern_playing_out тоже подтвердил себя отдельно)
# =============================================================================
def pattern_stage_label(breakout_price, target_price, current_price):
    total = target_price - breakout_price
    if total == 0:
        return None
    progress = (current_price - breakout_price) / total
    if progress < 0:
        return None
    if progress < 0.15:
        return "свежий пробой"
    elif progress < 0.85:
        return "уже отрабатывает"
    else:
        return "близко к цели, вероятно исчерпан"


def detect_symmetry_patterns(df, lookback, swing_order):
    found = []
    window = df.tail(lookback).reset_index(drop=True)
    if len(window) < 30:
        return found

    highs, lows = window["High"], window["Low"]
    close_last = window["Close"].iloc[-1]
    high_idx, _ = find_swings(highs, swing_order)
    _, low_idx = find_swings(lows, swing_order)
    tol = window["Close"].std() * 0.6 if window["Close"].std() > 0 else close_last * 0.01

    if len(high_idx) >= 3:
        last3 = high_idx[-3:]
        h_vals = highs.iloc[last3].values
        left_sh, head, right_sh = h_vals[0], h_vals[1], h_vals[2]
        if head > left_sh + tol and head > right_sh + tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = lows.iloc[last3[0]:last3[2]].min()
            if close_last < neckline:
                target = neckline - (head - neckline)
                stage = pattern_stage_label(neckline, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Голова и плечи (подтверждено пробоем шеи вниз, медвежий{tag})")
            else:
                found.append("Голова и плечи (формируется, шея не пробита, медвежий)")

    if len(low_idx) >= 3:
        last3 = low_idx[-3:]
        l_vals = lows.iloc[last3].values
        left_sh, head, right_sh = l_vals[0], l_vals[1], l_vals[2]
        if head < left_sh - tol and head < right_sh - tol and abs(left_sh - right_sh) < tol * 1.5:
            neckline = highs.iloc[last3[0]:last3[2]].max()
            if close_last > neckline:
                target = neckline + (neckline - head)
                stage = pattern_stage_label(neckline, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Перевёрнутая голова и плечи (подтверждено пробоем вверх, бычий{tag})")
            else:
                found.append("Перевёрнутая голова и плечи (формируется, бычий)")

    if len(high_idx) >= 2:
        h1, h2 = highs.iloc[high_idx[-2]], highs.iloc[high_idx[-1]]
        if abs(h1 - h2) < tol and (high_idx[-1] - high_idx[-2]) > swing_order * 2:
            trough = lows.iloc[high_idx[-2]:high_idx[-1]].min()
            if close_last < trough:
                top_level = (h1 + h2) / 2
                target = trough - (top_level - trough)
                stage = pattern_stage_label(trough, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Двойная вершина (подтверждена пробоем вниз, медвежий{tag})")
            else:
                found.append("Двойная вершина (формируется, медвежий)")

    if len(low_idx) >= 2:
        l1, l2 = lows.iloc[low_idx[-2]], lows.iloc[low_idx[-1]]
        if abs(l1 - l2) < tol and (low_idx[-1] - low_idx[-2]) > swing_order * 2:
            peak = highs.iloc[low_idx[-2]:low_idx[-1]].max()
            if close_last > peak:
                bottom_level = (l1 + l2) / 2
                target = peak + (peak - bottom_level)
                stage = pattern_stage_label(peak, target, close_last)
                tag = f", {stage}" if stage else ""
                found.append(f"Двойное дно (подтверждено пробоем вверх, бычий{tag})")
            else:
                found.append("Двойное дно (формируется, бычий)")

    return found


# =============================================================================
# 6. РИСК-МЕНЕДЖМЕНТ
# =============================================================================
def compute_risk_levels(last_row, direction, atr_mult, rr_target, bid=None, offer=None):
    close_price, atr = last_row["Close"], last_row["ATR14"]

    entry_price = close_price
    if direction == "long" and offer and offer > 0:
        entry_price = offer
    elif direction == "short" and bid and bid > 0:
        entry_price = bid

    min_risk = entry_price * 0.001

    if direction == "long":
        stop = entry_price - atr_mult * atr
        risk_per_unit = max(entry_price - stop, min_risk)
        target = entry_price + risk_per_unit * rr_target
    else:
        stop = entry_price + atr_mult * atr
        risk_per_unit = max(stop - entry_price, min_risk)
        target = entry_price - risk_per_unit * rr_target

    spread_abs, spread_pct, risk_per_spread = None, None, None
    if bid and offer and bid > 0 and offer > bid:
        spread_abs = offer - bid
        mid = (offer + bid) / 2
        spread_pct = spread_abs / mid * 100
        risk_per_spread = risk_per_unit / spread_abs if spread_abs > 0 else None

    spread_warning = None
    if risk_per_spread is not None:
        if risk_per_spread < 3:
            spread_warning = "Спред широкий относительно стопа"
        elif risk_per_spread < 6:
            spread_warning = "Спред заметный"

    return {
        "entry": smart_round(entry_price),
        "stop_loss": smart_round(stop),
        "take_profit": smart_round(target),
        "risk_reward": rr_target,
        "spread_pct": round(spread_pct, 3) if spread_pct is not None else None,
        "risk_per_spread": round(risk_per_spread, 1) if risk_per_spread is not None else None,
        "spread_warning": spread_warning,
    }


# =============================================================================
# 7. СИГНАЛЬНЫЕ КОМПОНЕНТЫ — 6 отобранных, знаковые (+1/0/-1)
# =============================================================================
DEFAULT_WEIGHTS = {
    # symmetry и rel_strength УБРАНЫ по факту честного train/test — дважды подряд
    # (обычная калибровка и train/test) показали отрицательный/нулевой лифт В
    # ЭТОЙ КОМБИНАЦИИ компонентов. Это не значит, что они "плохие" сами по
    # себе (symmetry была устойчиво положительной в одиночку, в скрипте 2) —
    # значит, что их вклад уже покрывается trend/vwap_dev/fib_level, добавочной
    # информации не даёт. Веса ниже — новые приоры по итогам ДВУХ измерений
    # именно этой четвёрки вместе (не считая уже отрабатывает — она отдельно
    # учитывает то же самое, что symmetry, но с фильтром "уже частично сбылось").
    "vwap_dev": 1.5,             # оба измерения: 0.111, 0.069 — сильнейший из всех
    "trend": 1.3,                # оба измерения: 0.116, 0.039 — стабилен, но послабее в этом составе
    "pattern_playing_out": 1.1,  # оба измерения: 0.070, 0.051 — держится, несмотря на то что symmetry ушла в минус
    "fib_level": 0.8,            # оба измерения: 0.024, 0.060 — растёт, был самым слабым, теперь не факт
}
COMPONENT_NAMES = list(DEFAULT_WEIGHTS.keys())

PATTERN_BULLISH_KW = ["бычий", "вверх", "дно", "Перевёрнутая"]
PATTERN_BEARISH_KW = ["медвежий", "вниз", "вершина"]


def compute_signal_components(last, patterns):
    c = {name: 0 for name in COMPONENT_NAMES}

    if last["trend_up"]:
        c["trend"] = 1
    elif last["trend_down"]:
        c["trend"] = -1

    vwap_dev = last.get("price_vs_vwap_pct", 0)
    if pd.notna(vwap_dev):
        if vwap_dev > 0.3:
            c["vwap_dev"] = 1
        elif vwap_dev < -0.3:
            c["vwap_dev"] = -1

    for p in patterns:
        if "Фибоначчи" in p:
            c["fib_level"] = 1 if "бычий" in p else -1
            continue

        is_bull = any(k in p for k in PATTERN_BULLISH_KW)
        is_bear = any(k in p for k in PATTERN_BEARISH_KW)
        if not (is_bull or is_bear):
            continue
        sign = 1 if is_bull else -1

        # symmetry как отдельный компонент убрана (см. комментарий у
        # DEFAULT_WEIGHTS), но сам детектор Г-и-П/двойного дна всё ещё
        # нужен — именно на его основе считается pattern_playing_out
        if "уже отрабатывает" in p:
            c["pattern_playing_out"] = sign

    return c


def score_from_components(last, components, weights):
    net = sum(weights.get(name, 0.0) * val for name, val in components.items())
    direction = "long" if net >= 0 else "short"
    return round(abs(net), 2), direction


def is_scoring_pattern(p):
    """
    Правда ли, что эта строка паттерна реально влияет на текущий набор
    компонентов score (см. compute_signal_components) — а не просто
    какой-то паттерн, который когда-то был скоринговым (напр. symmetry),
    но сейчас отключён. Общая функция для compute_signal_components и
    format_patterns_for_display — чтобы они не могли разойтись, как уже
    один раз произошло (отображение показывало "ПРОТИВ" от компонента,
    который на деле score больше не трогает).
    """
    if "Фибоначчи" in p:
        return True
    if "уже отрабатывает" in p and (any(k in p for k in PATTERN_BULLISH_KW) or
                                     any(k in p for k in PATTERN_BEARISH_KW)):
        return True
    return False


def format_patterns_for_display(patterns, direction):
    scoring_patterns = [p for p in patterns if is_scoring_pattern(p)]
    if not scoring_patterns:
        return "-"
    is_long = direction == "long"
    supporting, conflicting = [], []
    for p in scoring_patterns:
        is_bull = any(k in p for k in PATTERN_BULLISH_KW)
        is_bear = any(k in p for k in PATTERN_BEARISH_KW)
        if not (is_bull or is_bear):
            continue
        aligned = (is_bull and is_long) or (is_bear and not is_long)
        (supporting if aligned else conflicting).append(p)
    parts = []
    if supporting:
        parts.append("ЗА: " + "; ".join(supporting))
    if conflicting:
        parts.append("ПРОТИВ: " + "; ".join(conflicting))
    return " | ".join(parts) if parts else "-"


# =============================================================================
# 8. БЭКТЕСТ + КАЛИБРОВКА (структура идентична скриптам 1 и 2)
# =============================================================================
def r_multiple(direction, entry, stop, exit_price):
    risk = abs(entry - stop)
    if risk == 0 or exit_price is None:
        return None
    pnl = (exit_price - entry) if direction == "long" else (entry - exit_price)
    return pnl / risk


def simulate_trade_path(future_df, direction, entry, stop, target, max_holding_bars):
    risk = abs(entry - stop)
    if risk == 0:
        return {"outcome": "no_data", "exit_price": None, "bars_held": 0,
                "mae_r": None, "mfe_r": None, "r_multiple": None, "path_label": "нет данных"}

    max_adverse_r, max_favorable_r = 0.0, 0.0
    bars_held = 0
    outcome, exit_price = "timeout", None

    for _, row in future_df.head(max_holding_bars).iterrows():
        bars_held += 1
        high, low = row["High"], row["Low"]
        if direction == "long":
            adverse_r = (entry - low) / risk
            favorable_r = (high - entry) / risk
            hit_stop, hit_target = low <= stop, high >= target
        else:
            adverse_r = (high - entry) / risk
            favorable_r = (entry - low) / risk
            hit_stop, hit_target = high >= stop, low <= target

        max_adverse_r = max(max_adverse_r, adverse_r)
        max_favorable_r = max(max_favorable_r, favorable_r)

        if hit_stop:
            outcome, exit_price = "stop", stop
            break
        if hit_target:
            outcome, exit_price = "target", target
            break
    else:
        tail = future_df.head(max_holding_bars)
        if len(tail) > 0:
            exit_price = tail["Close"].iloc[-1]
        outcome = "timeout"

    final_r = r_multiple(direction, entry, stop, exit_price) if exit_price is not None else None

    if outcome == "stop":
        path_label = "Сразу против" if max_favorable_r < 0.3 else "Сходил в плюс, потом развернулся и выбил"
    elif outcome == "target":
        path_label = "Дошёл до цели с сильной просадкой по пути" if max_adverse_r > 0.5 else "Дошёл до цели уверенно"
    elif outcome == "timeout":
        if final_r is not None and final_r > 0.5:
            path_label = "Шёл по сценарию, не успел дойти за отведённое время"
        elif final_r is not None and final_r < -0.3:
            path_label = "Шёл против, но не успел выбить стоп"
        else:
            path_label = "Болтался без выраженного направления"
    else:
        path_label = "нет данных"

    return {"outcome": outcome, "exit_price": exit_price, "bars_held": bars_held,
            "mae_r": round(max_adverse_r, 3), "mfe_r": round(max_favorable_r, 3),
            "r_multiple": round(final_r, 3) if final_r is not None else None,
            "path_label": path_label}


def _compute_all_patterns(window, config):
    fib_p = compute_fibonacci_signal(window, config["pattern_lookback_bars"],
                                      config["swing_order"], config["fib_tolerance_atr"])
    symmetry_p = detect_symmetry_patterns(window, config["pattern_lookback_bars"], config["swing_order"])
    return fib_p + symmetry_p


def backtest_ticker(df, config, weights=None, start_i=None, end_i=None):
    """
    weights — какими весами определять направление сделки. По умолчанию
    (None) используются DEFAULT_WEIGHTS — так делается при калибровке,
    чтобы не было циклической зависимости (веса же ещё только предстоит
    посчитать). Но для ЧЕСТНОЙ проверки на отложенных данных (см.
    run_train_test_validation) сюда передаются уже ЗАФИКСИРОВАННЫЕ веса,
    посчитанные ТОЛЬКО на обучающей половине истории — так direction
    отражает то, что реально предсказала бы обученная система, а не
    служебный дефолт.
    start_i/end_i — диапазон индексов баров для прохода (по умолчанию —
    вся история). Нужно, чтобы прогонять бэктест отдельно по обучающей и
    по тестовой половине одного и того же тикера.
    """
    weights = weights or DEFAULT_WEIGHTS
    trades = []
    n = len(df)
    start = max(start_i if start_i is not None else 210, 210)  # EMA200 нужен прогрев
    end = min(end_i if end_i is not None else n - 5, n - 5)
    for i in range(start, end, config["backtest_stride"]):
        window = df.iloc[:i + 1]
        last = window.iloc[-1]
        if pd.isna(last["ATR14"]) or last["ATR14"] <= 0:
            continue

        patterns = _compute_all_patterns(window, config)
        components = compute_signal_components(last, patterns)
        _, direction = score_from_components(last, components, weights)
        risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"])

        future = df.iloc[i + 1:]
        if future.empty:
            continue
        path = simulate_trade_path(future, direction, risk["entry"], risk["stop_loss"],
                                    risk["take_profit"], config["backtest_max_holding_bars"])
        if path["r_multiple"] is not None:
            row = dict(components)
            row.update({"direction": direction, "outcome": path["outcome"],
                        "r_multiple": path["r_multiple"], "mae_r": path["mae_r"],
                        "mfe_r": path["mfe_r"], "path_label": path["path_label"],
                        "bars_held": path["bars_held"]})
            trades.append(row)

    return pd.DataFrame(trades)


def stop_multiplier_sensitivity(df, config, multipliers):
    results = {m: [] for m in multipliers}
    n = len(df)
    start = 210
    for i in range(start, n - 5, config["stop_sensitivity_stride"]):
        window = df.iloc[:i + 1]
        last = window.iloc[-1]
        if pd.isna(last["ATR14"]) or last["ATR14"] <= 0:
            continue

        patterns = _compute_all_patterns(window, config)
        components = compute_signal_components(last, patterns)
        _, direction = score_from_components(last, components, DEFAULT_WEIGHTS)

        future = df.iloc[i + 1:]
        if future.empty:
            continue

        for m in multipliers:
            risk = compute_risk_levels(last, direction, m, config["risk_reward_target"])
            path = simulate_trade_path(future, direction, risk["entry"], risk["stop_loss"],
                                        risk["take_profit"], config["backtest_max_holding_bars"])
            if path["r_multiple"] is not None:
                results[m].append({"outcome": path["outcome"], "r_multiple": path["r_multiple"]})

    return results


def run_stop_sensitivity_analysis(data_by_ticker, config):
    tickers = list(data_by_ticker.keys())[:config["stop_sensitivity_tickers"]]
    multipliers = config["stop_sensitivity_multipliers"]
    combined = {m: [] for m in multipliers}

    for t in tqdm(tickers, desc="   Проверка разных множителей стопа по тикерам"):
        try:
            res = stop_multiplier_sensitivity(data_by_ticker[t], config, multipliers)
            for m in multipliers:
                combined[m].extend(res[m])
        except Exception:
            continue

    rows = []
    for m in multipliers:
        trades = combined[m]
        if not trades:
            continue
        n = len(trades)
        win_rate = sum(1 for x in trades if x["outcome"] == "target") / n * 100
        avg_r = sum(x["r_multiple"] for x in trades) / n
        rows.append({"Множитель ATR (стоп)": m, "Сделок": n, "Винрейт %": round(win_rate, 1),
                     "Средний_R": round(avg_r, 3), "Сумма_R": round(sum(x["r_multiple"] for x in trades), 1)})
    return pd.DataFrame(rows)


def compute_component_aggregates(trades):
    aggs = {}
    direction_sign = trades["direction"].map({"long": 1, "short": -1})
    for name in COMPONENT_NAMES:
        if name not in trades.columns:
            continue
        mask = trades[name] == direction_sign
        aggs[name] = {
            "n_true": int(mask.sum()),
            "sum_true": float(trades.loc[mask, "r_multiple"].sum()),
            "n_false": int((~mask).sum()),
            "sum_false": float(trades.loc[~mask, "r_multiple"].sum()),
        }
    return aggs


def merge_aggregates(a, b):
    empty = {"n_true": 0, "sum_true": 0.0, "n_false": 0, "sum_false": 0.0}
    merged = {}
    for name in set(a.keys()) | set(b.keys()):
        av, bv = a.get(name, empty), b.get(name, empty)
        merged[name] = {
            "n_true": av["n_true"] + bv["n_true"],
            "sum_true": av["sum_true"] + bv["sum_true"],
            "n_false": av["n_false"] + bv["n_false"],
            "sum_false": av["sum_false"] + bv["sum_false"],
        }
    return merged


CALIBRATION_CACHE_KEY = "calibration_aggregates_v3"


def load_accumulated_calibration(cache_dir):
    try:
        return _load_from_cache(cache_dir, CALIBRATION_CACHE_KEY, ttl_hours=24 * 30)
    except Exception:
        return None


def save_accumulated_calibration(cache_dir, aggs):
    try:
        _save_to_cache(cache_dir, CALIBRATION_CACHE_KEY, aggs)
    except Exception:
        pass


def fit_weights_from_aggregates(aggs, min_samples=40):
    diag_rows, new_weights = [], {}
    for name in COMPONENT_NAMES:
        a = aggs.get(name)
        if not a or a["n_true"] < min_samples or a["n_false"] < min_samples:
            new_weights[name] = DEFAULT_WEIGHTS[name]
            diag_rows.append({"Компонент": name, "Случаев": a["n_true"] if a else 0,
                               "Средний_R(есть)": None, "Средний_R(нет)": None, "Лифт": None,
                               "Вес": f"{DEFAULT_WEIGHTS[name]} (данных мало, оставлен дефолт)"})
            continue
        mean_true = a["sum_true"] / a["n_true"]
        mean_false = a["sum_false"] / a["n_false"]
        lift = mean_true - mean_false
        diag_rows.append({"Компонент": name, "Случаев": a["n_true"],
                           "Средний_R(есть)": round(mean_true, 3),
                           "Средний_R(нет)": round(mean_false, 3),
                           "Лифт": round(lift, 3), "Вес": None})
        new_weights[name] = lift

    raw_positive = [w for name, w in new_weights.items()
                    if isinstance(w, float) and w > 0 and "оставлен дефолт" not in
                    str(next((r["Вес"] for r in diag_rows if r["Компонент"] == name), ""))]
    scale = (1.5 / max(raw_positive)) if raw_positive else 1.0

    final_weights = {}
    for row in diag_rows:
        name = row["Компонент"]
        if row["Вес"] is not None:
            final_weights[name] = DEFAULT_WEIGHTS[name]
        else:
            w = max(row["Лифт"], 0) * scale
            final_weights[name] = round(w, 2)
            row["Вес"] = round(w, 2)

    return final_weights, pd.DataFrame(diag_rows)


def score_for_fixed_direction(components, direction, weights):
    net = sum(weights.get(name, 0.0) * val for name, val in components.items())
    return round(net if direction == "long" else -net, 2)


def bucket_summary_for(trades, score_col):
    bins = [-100, 1.5, 3, 4.5, 100]
    labels = ["<1.5 (слабый)", "1.5-3", "3-4.5", "4.5+ (сильный)"]
    t = trades.copy()
    t["bucket"] = pd.cut(t[score_col], bins=bins, labels=labels)
    return t.groupby("bucket", observed=True).agg(
        Сделок=("r_multiple", "count"),
        Винрейт=("outcome", lambda x: round((x == "target").mean() * 100, 1)),
        Средний_R=("r_multiple", lambda x: round(x.mean(), 3)),
        Сумма_R=("r_multiple", lambda x: round(x.sum(), 1)),
    ).reset_index()


def run_backtest(data_by_ticker, config):
    tickers = list(data_by_ticker.keys())[:config["backtest_max_tickers"]]
    all_trades = []
    for t in tqdm(tickers, desc="   Бэктест по тикерам"):
        try:
            tr = backtest_ticker(data_by_ticker[t], config)
            if not tr.empty:
                tr["ticker"] = t
                all_trades.append(tr)
        except Exception:
            continue

    if not all_trades:
        return None, None, None, None

    trades = pd.concat(all_trades, ignore_index=True)

    this_run_aggs = compute_component_aggregates(trades)
    if config.get("persist_calibration_across_runs", True):
        cached = load_accumulated_calibration(config["cache_dir"])
        accumulated = merge_aggregates(cached, this_run_aggs) if cached else this_run_aggs
        save_accumulated_calibration(config["cache_dir"], accumulated)
    else:
        accumulated = this_run_aggs

    fitted_weights, diagnostics = fit_weights_from_aggregates(
        accumulated, min_samples=config["backtest_min_samples_per_component"]
    )
    if config.get("persist_calibration_across_runs", True) and diagnostics is not None:
        total_acc = sum(a["n_true"] + a["n_false"] for a in accumulated.values()) // max(len(accumulated), 1)
        print(f"   Накоплено данных калибровки: ~{total_acc} сделок за все прогоны в этой сессии.")

    trades["score_default"] = trades.apply(
        lambda row: score_for_fixed_direction({c: row[c] for c in COMPONENT_NAMES}, row["direction"], DEFAULT_WEIGHTS),
        axis=1
    )
    trades["score_fitted"] = trades.apply(
        lambda row: score_for_fixed_direction({c: row[c] for c in COMPONENT_NAMES}, row["direction"], fitted_weights),
        axis=1
    )

    summary_default = bucket_summary_for(trades, "score_default")
    summary_fitted = bucket_summary_for(trades, "score_fitted")

    return trades, summary_default, summary_fitted, {"weights": fitted_weights, "diagnostics": diagnostics}


# =============================================================================
# 9. ЧЕСТНАЯ TRAIN/TEST ПРОВЕРКА (не то же самое, что калибровка выше!)
# =============================================================================
def run_train_test_validation(config=CONFIG, train_frac=0.6, max_tickers=None):
    """
    В отличие от run_backtest (который калибрует и сразу же показывает
    результат НА ТЕХ ЖЕ данных — это разведка, а не доказательство), здесь
    история каждого тикера жёстко режется на две непересекающиеся части:
      - ОБУЧАЮЩАЯ (первые `train_frac` истории) — по ней и только по ней
        считаются веса компонентов.
      - ТЕСТОВАЯ (остаток) — веса здесь ЗАФИКСИРОВАНЫ (взяты из обучающей
        части) и просто применяются вперёд. Тестовая часть при подборе
        весов не участвовала вообще никак.

    Число сделок в тестовой части обычно заметно больше, чем в
    walk-forward (там всего несколько дат) — потому что здесь берётся ВЕСЬ
    непрерывный тестовый период, а не отдельные точки. Это даёт более
    статистически весомую, при этом честную (без утечки будущего в
    калибровку) оценку.
    """
    print("ЧЕСТНАЯ TRAIN/TEST ПРОВЕРКА: веса считаются на одной половине истории,")
    print("результат — на другой, которую калибровка не видела вообще.\n")

    print("1/4: Тикеры + полная история...")
    snapshot = get_market_snapshot()
    top_n = max_tickers or config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(snapshot, top_n)

    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )
    try:
        index_df = get_index_candles("IMOEX", config["history_days"], config["candle_interval"])
    except Exception:
        index_df = None

    print("2/4: Считаю индикаторы на полной истории (индикаторы каузальны — не заглядывают")
    print("     вперёд сами по себе; честность обеспечивается ниже — тем, что веса калибруются")
    print("     ТОЛЬКО по обучающему диапазону индексов баров)...")
    data_by_ticker, split_index_by_ticker = {}, {}
    for t, raw in raw_by_ticker.items():
        if len(raw) < 210 * 2:  # нужно хотя бы вдвое больше минимума — на обе половины
            continue
        try:
            df_ind = add_core_indicators(raw)
            if df_ind["ATR_pct"].tail(200).median() < config["min_atr_pct"]:
                continue
            df_ind = compute_relative_strength(df_ind, index_df, config["rel_strength_window"])
            data_by_ticker[t] = df_ind
            split_index_by_ticker[t] = int(len(df_ind) * train_frac)
        except Exception:
            continue

    print(f"   Тикеров с достаточной историей на обе половины: {len(data_by_ticker)}")
    if len(data_by_ticker) < 10:
        print("   Недостаточно тикеров для честной проверки.")
        return None

    calib_tickers = list(data_by_ticker.keys())[:config["backtest_max_tickers"]]

    print(f"\n3/4: Калибрую веса ТОЛЬКО на обучающей части (~{train_frac*100:.0f}% истории, "
          f"{len(calib_tickers)} тикеров)...")
    train_trades_all = []
    for t in tqdm(calib_tickers, desc="   Калибровка на train"):
        try:
            tr = backtest_ticker(data_by_ticker[t], config, weights=DEFAULT_WEIGHTS,
                                  start_i=210, end_i=split_index_by_ticker[t])
            if not tr.empty:
                train_trades_all.append(tr)
        except Exception:
            continue

    if not train_trades_all:
        print("   Не набралось сделок на обучающей части.")
        return None

    train_trades = pd.concat(train_trades_all, ignore_index=True)
    train_aggs = compute_component_aggregates(train_trades)
    fitted_weights, diagnostics = fit_weights_from_aggregates(
        train_aggs, min_samples=config["backtest_min_samples_per_component"]
    )
    print(f"   Сделок на обучающей части: {len(train_trades)}")
    print("\n" + "=" * 100)
    print("ВЕСА, ПОДОБРАННЫЕ ТОЛЬКО ПО ОБУЧАЮЩЕЙ ЧАСТИ (тестовая их ещё не видела)")
    print("=" * 100)
    print(diagnostics.to_string(index=False))

    print(f"\n4/4: Проверяю ЭТИ ЖЕ, уже зафиксированные веса на тестовой части "
          f"(~{(1-train_frac)*100:.0f}% истории, все {len(data_by_ticker)} тикеров, "
          f"которую калибровка не видела)...")
    test_trades_all = []
    for t in tqdm(list(data_by_ticker.keys()), desc="   Честная проверка на test"):
        try:
            tr = backtest_ticker(data_by_ticker[t], config, weights=fitted_weights,
                                  start_i=split_index_by_ticker[t], end_i=None)
            if not tr.empty:
                tr["ticker"] = t
                test_trades_all.append(tr)
        except Exception:
            continue

    if not test_trades_all:
        print("   Не набралось сделок на тестовой части.")
        return None

    test_trades = pd.concat(test_trades_all, ignore_index=True)
    test_trades["score"] = test_trades.apply(
        lambda row: score_for_fixed_direction({c: row[c] for c in COMPONENT_NAMES}, row["direction"], fitted_weights),
        axis=1
    )
    summary = bucket_summary_for(test_trades, "score")

    win_rate = (test_trades["outcome"] == "target").mean() * 100
    avg_r, sum_r = test_trades["r_multiple"].mean(), test_trades["r_multiple"].sum()

    print("\n" + "=" * 100)
    print(f"ЧЕСТНЫЙ РЕЗУЛЬТАТ НА ОТЛОЖЕННЫХ ДАННЫХ: {len(test_trades)} сделок, "
          f"{test_trades['ticker'].nunique()} тикеров")
    print("=" * 100)
    print(f"Винрейт: {win_rate:.1f}%   Средний R: {avg_r:.3f}   Суммарный R: {sum_r:.1f}")
    print("\nПо бакетам score (посчитанного зафиксированными весами):")
    print(summary.to_string(index=False))
    print("\nЭто и есть реальный, не подогнанный процент успеха системы — веса ни разу")
    print("не видели эти данные при своём подборе.")

    return {"train_trades": train_trades, "test_trades": test_trades,
            "fitted_weights": fitted_weights, "diagnostics": diagnostics, "summary": summary}


# =============================================================================
# ГЛАВНЫЙ PIPELINE
# =============================================================================
def run_screener(config=CONFIG):
    print("СИНТЕЗ: только 6 компонентов с доказанной устойчивой работой (скрипты 1+2)\n")
    print("1/6: Быстрый префильтр по ликвидности + текущий стакан (bid/offer)...")
    snapshot = get_market_snapshot()
    top_n = config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(snapshot, top_n)
    quotes = snapshot.set_index("SECID")[["BID", "OFFER"]].to_dict("index")
    print(f"   Беру в работу {len(tickers)} тикеров")

    print("2/6: Скачиваю часовые свечи параллельно...")
    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )

    print("2b/6: Скачиваю индекс IMOEX для относительной силы...")
    try:
        index_df = get_index_candles("IMOEX", config["history_days"], config["candle_interval"])
    except Exception:
        index_df = None
        print("   Не удалось получить IMOEX — rel_strength будет пропущен для всех тикеров")

    data_by_ticker = {}
    skipped_low_vol, failed_indicator_tickers = 0, []
    for t, raw in raw_by_ticker.items():
        if len(raw) < 210:
            continue
        try:
            df_ind = add_core_indicators(raw)
            median_atr_pct = df_ind["ATR_pct"].tail(200).median()
            if median_atr_pct < config["min_atr_pct"]:
                skipped_low_vol += 1
                continue
            df_ind = compute_relative_strength(df_ind, index_df, config["rel_strength_window"])
            data_by_ticker[t] = df_ind
        except Exception as e:
            failed_indicator_tickers.append((t, str(e)))

    print(f"   Хватает истории для анализа: {len(data_by_ticker)} тикеров "
          f"(отсеяно как низковолатильные фонды: {skipped_low_vol})")

    print("3/6: Бэктест на истории + калибровка весов score по факту...")
    fitted_weights, weight_diagnostics = dict(DEFAULT_WEIGHTS), None
    backtest_trades, backtest_summary_default, backtest_summary_fitted = None, None, None
    if config["backtest_enabled"]:
        print(f"   ~{config['backtest_max_tickers']} тикеров...")
        backtest_trades, backtest_summary_default, backtest_summary_fitted, fit_result = run_backtest(data_by_ticker, config)
        if fit_result is not None:
            fitted_weights = fit_result["weights"]
            weight_diagnostics = fit_result["diagnostics"]
            print("   Веса пересчитаны по факту истории — используются в отчёте ниже.")

            if backtest_trades is not None and not backtest_trades.empty:
                print("\n   Как вели себя цены в сделках бэктеста (path_label):")
                path_counts = backtest_trades["path_label"].value_counts()
                path_pct = (path_counts / len(backtest_trades) * 100).round(1)
                for label, cnt in path_counts.items():
                    print(f"     {label}: {cnt} ({path_pct[label]}%)")
                print(f"   Средняя MAE: {backtest_trades['mae_r'].mean():.2f}R, "
                      f"средняя MFE: {backtest_trades['mfe_r'].mean():.2f}R")
        else:
            print("   Недостаточно сделок для калибровки — использую веса по умолчанию.")

    stop_sensitivity_df = None
    if config["stop_sensitivity_enabled"] and len(data_by_ticker) > 0:
        print(f"\n3b/6: Чувствительность к множителю стопа "
              f"({config['stop_sensitivity_tickers']} тикеров)...")
        stop_sensitivity_df = run_stop_sensitivity_analysis(data_by_ticker, config)
        if stop_sensitivity_df is not None and not stop_sensitivity_df.empty:
            print("\n" + "=" * 100)
            print("ЧУВСТВИТЕЛЬНОСТЬ К МНОЖИТЕЛЮ СТОПА")
            print("=" * 100)
            print(stop_sensitivity_df.to_string(index=False))

    print("\n4/6: Считаю сигналы (тренд, VWAP, симметрии, Фибо, отн.сила), риск-уровни и score...")
    results = []
    failed_result_tickers = []
    for t, df in data_by_ticker.items():
        try:
            patterns = _compute_all_patterns(df, config)
            last = df.iloc[-1]

            components = compute_signal_components(last, patterns)
            score, direction = score_from_components(last, components, fitted_weights)
            if score == 0:
                continue

            q = quotes.get(t, {})
            bid, offer = q.get("BID"), q.get("OFFER")
            risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"],
                                        bid=bid, offer=offer)
        except Exception as e:
            failed_result_tickers.append((t, str(e)))
            continue

        results.append({
            "Тикер": t,
            "Цена": smart_round(last["Close"]),
            "Направление": "ЛОНГ" if direction == "long" else "ШОРТ",
            "Score": score,
            "Сигналы": format_patterns_for_display(patterns, direction),
            "Вход": risk["entry"],
            "Стоп-лосс": risk["stop_loss"],
            "Тейк-профит": risk["take_profit"],
            "Risk/Reward": risk["risk_reward"],
            "Спред %": risk["spread_pct"],
            "Риск/спред x": risk["risk_per_spread"],
            "⚠ Спред": risk["spread_warning"] or "-",
        })

    report = pd.DataFrame(results)
    if not report.empty:
        report = report.sort_values("Score", ascending=False).reset_index(drop=True)

    if failed_result_tickers:
        print(f"   Пропущено: {len(failed_result_tickers)} тикеров "
              f"(напр. {failed_result_tickers[0][0]}: {failed_result_tickers[0][1][:80]})")

    if weight_diagnostics is not None:
        print("\n" + "=" * 100)
        print("КАКИЕ ИЗ 6 КОМПОНЕНТОВ РЕАЛЬНО ПОКАЗАЛИ ЭДЖ НА ИСТОРИИ ЭТОГО СКРИПТА")
        print("=" * 100)
        print(weight_diagnostics.to_string(index=False))

    if backtest_summary_default is not None and backtest_summary_fitted is not None:
        print("\n" + "=" * 100)
        print("ДО/ПОСЛЕ калибровки")
        print("=" * 100)
        print("-- ДО (стартовые веса по надёжности) --")
        print(backtest_summary_default.to_string(index=False))
        print("-- ПОСЛЕ (веса по факту бэктеста ЭТОГО скрипта) --")
        print(backtest_summary_fitted.to_string(index=False))

    print(f"\n5/6: Готово. Рекомендаций (хотя бы один сигнал сработал): {len(report)}\n")
    print("=" * 100)
    print(f"ТОП-{config['top_n_report']} ИДЕЙ (не финансовая рекомендация!)")
    print("=" * 100)
    if report.empty:
        print("Ни на одном тикере не сработал ни один сигнал на момент запуска.")
    else:
        with pd.option_context("display.max_columns", None, "display.width", 220):
            print(report.head(config["top_n_report"]).to_string(index=False))

    print("\n6/6: Готово.")
    return report


# =============================================================================
# WALK-FORWARD ВАЛИДАЦИЯ
# =============================================================================
def _generate_asof_report(visible_by_ticker, index_visible, weights, config):
    results = []
    for t, raw in visible_by_ticker.items():
        if len(raw) < 210:
            continue
        try:
            df_ind = add_core_indicators(raw)
            if df_ind["ATR_pct"].tail(200).median() < config["min_atr_pct"]:
                continue
            df_ind = compute_relative_strength(df_ind, index_visible, config["rel_strength_window"])

            patterns = _compute_all_patterns(df_ind, config)
            last = df_ind.iloc[-1]

            components = compute_signal_components(last, patterns)
            score, direction = score_from_components(last, components, weights)
            if score == 0:
                continue
            risk = compute_risk_levels(last, direction, config["atr_stop_mult"], config["risk_reward_target"])

            results.append({
                "Тикер": t, "direction": direction, "Score": score,
                "Вход": risk["entry"], "Стоп-лосс": risk["stop_loss"], "Тейк-профит": risk["take_profit"],
            })
        except Exception:
            continue

    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values("Score", ascending=False).reset_index(drop=True)
    return df


def run_walkforward_test(config=CONFIG):
    print("Walk-forward валидация (СИНТЕЗ): 'что бы сказали 6 компонентов тогда — и что было по факту'\n")
    print("1/4: Тикеры + полная история...")
    snapshot = get_market_snapshot()
    top_n = config["max_tickers"] or config["liquidity_top_n"]
    tickers = get_liquid_tickers(snapshot, top_n)

    raw_by_ticker = download_all_candles(
        tickers, config["history_days"], config["candle_interval"],
        config["cache_dir"], config["cache_ttl_hours"], config["max_workers"],
    )
    try:
        index_raw = get_index_candles("IMOEX", config["history_days"], config["candle_interval"])
    except Exception:
        index_raw = None

    all_dates = pd.concat([df["Date"] for df in raw_by_ticker.values()], ignore_index=True)
    min_date, max_date = all_dates.min(), all_dates.max()

    history_buffer = pd.Timedelta(days=45)  # EMA200 нужен приличный прогрев
    eval_buffer = pd.Timedelta(hours=config["walkforward_eval_max_bars"] * 2)
    window_start = min_date + history_buffer
    window_end = max_date - eval_buffer

    if window_start >= window_end:
        print("   Недостаточно истории для walk-forward теста с текущими настройками.")
        return None

    n = config["walkforward_n_dates"]
    test_dates = pd.date_range(window_start, window_end, periods=n) if n > 1 else pd.DatetimeIndex([window_end])
    print(f"   Диапазон истории: {min_date.date()} — {max_date.date()}")
    print(f"   Тестовые даты ({n}): {', '.join(d.strftime('%Y-%m-%d %H:%M') for d in test_dates)}\n")

    all_eval_rows = []
    for di, as_of in enumerate(test_dates, 1):
        print(f"2/4: [{di}/{n}] Дата среза: {as_of.strftime('%Y-%m-%d %H:%M')}...")
        visible_by_ticker = {t: df[df["Date"] <= as_of].reset_index(drop=True) for t, df in raw_by_ticker.items()}
        future_by_ticker = {t: df[df["Date"] > as_of].reset_index(drop=True) for t, df in raw_by_ticker.items()}
        index_visible = index_raw[index_raw["Date"] <= as_of].reset_index(drop=True) if index_raw is not None else None

        data_by_ticker = {}
        for t, raw in visible_by_ticker.items():
            if len(raw) < 210:
                continue
            try:
                df_ind = add_core_indicators(raw)
                if df_ind["ATR_pct"].tail(200).median() < config["min_atr_pct"]:
                    continue
                df_ind = compute_relative_strength(df_ind, index_visible, config["rel_strength_window"])
                data_by_ticker[t] = df_ind
            except Exception:
                continue

        if len(data_by_ticker) < 10:
            print(f"   Мало тикеров на эту дату ({len(data_by_ticker)}), пропускаю.")
            continue

        wf_backtest_config = dict(config)
        wf_backtest_config["backtest_max_tickers"] = config["walkforward_calibration_tickers"]
        wf_backtest_config["persist_calibration_across_runs"] = False
        _, _, _, fit_result = run_backtest(data_by_ticker, wf_backtest_config)
        weights = fit_result["weights"] if fit_result else dict(DEFAULT_WEIGHTS)

        report = _generate_asof_report(visible_by_ticker, index_visible, weights, config)
        if report.empty:
            print("   Ни один сигнал не сработал на эту дату, пропускаю.")
            continue

        min_score = config.get("walkforward_min_score")
        if min_score is not None:
            before = len(report)
            report = report[report["Score"] >= min_score]
            if report.empty:
                print(f"   Ни один сигнал не превысил порог score={min_score} "
                      f"(было {before} до фильтра) — на эту дату сигналов нет, пропускаю.")
                continue

        top = report.head(config["walkforward_top_n"])
        print(f"   Рекомендаций на эту дату: {len(top)}. Проверяю по факту истории...")

        for _, row in top.iterrows():
            t = row["Тикер"]
            future = future_by_ticker.get(t)
            if future is None or future.empty:
                continue
            path = simulate_trade_path(future, row["direction"], row["Вход"], row["Стоп-лосс"],
                                        row["Тейк-профит"], config["walkforward_eval_max_bars"])
            if path["r_multiple"] is None:
                continue
            all_eval_rows.append({
                "Дата среза": as_of.strftime("%Y-%m-%d %H:%M"),
                "Тикер": t,
                "Направление": "ЛОНГ" if row["direction"] == "long" else "ШОРТ",
                "Score": row["Score"],
                "Вход": row["Вход"], "Стоп": row["Стоп-лосс"], "Тейк": row["Тейк-профит"],
                "Исход": {"target": "✅ Тейк", "stop": "❌ Стоп", "timeout": "⏱ Таймаут"}.get(path["outcome"], path["outcome"]),
                "R": path["r_multiple"],
            })

    print("\n3/4: Готово.\n")
    if not all_eval_rows:
        print("Не набралось ни одной проверяемой рекомендации.")
        return None

    eval_df = pd.DataFrame(all_eval_rows)
    n_dates_with_signal = eval_df["Дата среза"].nunique()
    print("=" * 100)
    print(f"WALK-FORWARD РЕЗУЛЬТАТ (СИНТЕЗ): {len(eval_df)} рекомендаций на {n_dates_with_signal} "
          f"из {len(test_dates)} запрошенных дат "
          f"{'(порог score=' + str(config.get('walkforward_min_score')) + ')' if config.get('walkforward_min_score') is not None else ''}")
    print("=" * 100)
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(eval_df.to_string(index=False))

    print("\n4/4: Сводка")
    print("=" * 100)
    win_rate = (eval_df["Исход"] == "✅ Тейк").mean() * 100
    avg_r, sum_r = eval_df["R"].mean(), eval_df["R"].sum()
    print(f"Всего рекомендаций проверено: {len(eval_df)}")
    print(f"Винрейт: {win_rate:.1f}%   Средний R: {avg_r:.3f}   Суммарный R: {sum_r:.1f}")

    by_date = eval_df.groupby("Дата среза").agg(
        Рекомендаций=("R", "count"),
        Винрейт=("Исход", lambda x: round((x == "✅ Тейк").mean() * 100, 1)),
        Средний_R=("R", lambda x: round(x.mean(), 3)),
    ).reset_index()
    print("\nПо каждой тестовой дате отдельно:")
    print(by_date.to_string(index=False))

    print(f"\n⚠ Это {len(test_dates)} исторических даты — доверительный интервал большой,")
    print("это ориентир, а не статистическое доказательство. Как мы уже видели на скрипте 2,")
    print("результат на 4-10 датах может развернуться на другой выборке дат.")

    return eval_df


# =============================================================================
# ЗАПУСК
# =============================================================================
if __name__ == "__main__":
    full_report = run_screener(CONFIG)
    walkforward_results = run_walkforward_test(CONFIG)

СИНТЕЗ: только 6 компонентов с доказанной устойчивой работой (скрипты 1+2)

1/6: Быстрый префильтр по ликвидности + текущий стакан (bid/offer)...
   Беру в работу 120 тикеров
2/6: Скачиваю часовые свечи параллельно...
   Из кэша сессии загружено сразу: 119 тикеров


   Качаю свечи параллельно:   0%|          | 0/1 [00:00<?, ?it/s]

2b/6: Скачиваю индекс IMOEX для относительной силы...
   Хватает истории для анализа: 104 тикеров (отсеяно как низковолатильные фонды: 16)
3/6: Бэктест на истории + калибровка весов score по факту...
   ~20 тикеров...


   Бэктест по тикерам:   0%|          | 0/20 [00:00<?, ?it/s]

   Накоплено данных калибровки: ~32673 сделок за все прогоны в этой сессии.
   Веса пересчитаны по факту истории — используются в отчёте ниже.

   Как вели себя цены в сделках бэктеста (path_label):
     Сходил в плюс, потом развернулся и выбил: 4333 (39.8%)
     Сразу против: 2309 (21.2%)
     Дошёл до цели уверенно: 2267 (20.8%)
     Дошёл до цели с сильной просадкой по пути: 1365 (12.5%)
     Шёл по сценарию, не успел дойти за отведённое время: 318 (2.9%)
     Болтался без выраженного направления: 231 (2.1%)
     Шёл против, но не успел выбить стоп: 68 (0.6%)
   Средняя MAE: 1.06R, средняя MFE: 1.32R

3b/6: Чувствительность к множителю стопа (10 тикеров)...


...

   Рекомендаций на эту дату: 10. Проверяю по факту истории...
2/4: [19/20] Дата среза: 2026-08-02 09:00...


   Бэктест по тикерам:   0%|          | 0/15 [00:00<?, ?it/s]

   Рекомендаций на эту дату: 10. Проверяю по факту истории...
2/4: [20/20] Дата среза: 2026-08-12 09:00...


   Бэктест по тикерам:   0%|          | 0/15 [00:00<?, ?it/s]

   Рекомендаций на эту дату: 10. Проверяю по факту истории...

3/4: Готово.

WALK-FORWARD РЕЗУЛЬТАТ (СИНТЕЗ): 174 рекомендаций на 20 из 20 запрошенных дат (порог score=1.5)
      Дата среза Тикер Направление  Score        Вход         Стоп        Тейк     Исход      R
2026-02-03 09:00  AFKS        ЛОНГ   1.50    13.76000    13.650000    13.97000    ❌ Стоп -1.000
2026-02-03 09:00  POSI        ЛОНГ   1.50  1147.20000  1136.850000  1167.90000    ❌ Стоп -1.000
2026-02-03 09:00  TRMK        ЛОНГ   1.50   104.70000   104.020000   106.05000    ✅ Тейк  1.985
2026-02-03 09:00  LENT        ЛОНГ   1.50  2143.50000  2130.640000  2169.22000    ❌ Стоп -1.000
2026-02-03 09:00  ENPG        ЛОНГ   1.50   503.85000   498.490000   514.57000    ❌ Стоп -1.000
2026-02-13 09:00  UPRO        ШОРТ   1.88     1.57000     1.580000     1.54000    ❌ Стоп -1.000
2026-02-23 09:00  CNRU        ЛОНГ   2.62   604.80000   600.870000   612.67000    ❌ Стоп -1.000
2026-02-23 09:00  AFKS        ЛОНГ   2.62    14.22000    14